<a href="https://colab.research.google.com/github/uvindu827/J26-DS-350/blob/Indhi_Component2_CHL_Forecast/05_XGBoost_Hyperparameter_Optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ======================================================================
# STEP 31.9P-16T-01
# XGBOOST HYPERPARAMETER OPTIMIZATION
# BASELINE VERIFICATION
#
# IMPORTANT:
# - NO MODEL TRAINING
# - NO TEST DATA LOADING
# - NO HYPERPARAMETER CHANGES
# - CURRENT XGBOOST REMAINS UNTOUCHED
# ======================================================================

import os
import json
import numpy as np
import pandas as pd

from pathlib import Path
from google.colab import drive


print("=" * 70)
print("STEP 31.9P-16T-01")
print("XGBOOST HYPERPARAMETER OPTIMIZATION")
print("BASELINE VERIFICATION")
print("=" * 70)


# ======================================================================
# 1. MOUNT GOOGLE DRIVE
# ======================================================================

drive.mount("/content/drive", force_remount=False)


# ======================================================================
# 2. PROJECT PATHS
# ======================================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Indhi_Marine_Research_Data"
)

PREPROCESSING_DIR = PROJECT_DIR / "05_model_preprocessing"

XGB_DIR = PROJECT_DIR / "06_models" / "02_XGBoost"

OPTIMIZATION_DIR = (
    XGB_DIR / "hyperparameter_optimization"
)

OPTIMIZATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FEATURE_FILE = (
    PREPROCESSING_DIR /
    "final_model_feature_columns.csv"
)

TRAIN_FILE = (
    PREPROCESSING_DIR /
    "final_model_ready_train.csv"
)

VALIDATION_FILE = (
    PREPROCESSING_DIR /
    "final_model_ready_validation.csv"
)

TEST_FILE = (
    PREPROCESSING_DIR /
    "final_model_ready_test.csv"
)


print("\n" + "=" * 70)
print("PROJECT PATHS")
print("=" * 70)

print(f"\nProject:")
print(PROJECT_DIR)

print(f"\nOptimization directory:")
print(OPTIMIZATION_DIR)

print(f"\nFeature list:")
print(FEATURE_FILE)

print(f"\nTraining data:")
print(TRAIN_FILE)

print(f"\nValidation data:")
print(VALIDATION_FILE)

print(f"\nTest data:")
print(TEST_FILE)


# ======================================================================
# 3. REQUIRED FILE CHECK
# ======================================================================

print("\n" + "=" * 70)
print("REQUIRED FILE CHECK")
print("=" * 70)

required_files = {
    "Feature list": FEATURE_FILE,
    "Training data": TRAIN_FILE,
    "Validation data": VALIDATION_FILE
}

all_required_found = True

for name, path in required_files.items():

    exists = path.exists()

    print(f"\n{name}:")
    print(path)
    print(f"Exists: {exists}")

    if not exists:
        all_required_found = False

if not all_required_found:

    raise FileNotFoundError(
        "One or more required training/validation files are missing."
    )

print("\n✓ All training/validation source files found.")


# ======================================================================
# 4. TEST DATA SAFETY CHECK
# ======================================================================

print("\n" + "=" * 70)
print("TEST DATA SAFETY CHECK")
print("=" * 70)

print(f"\nTest dataset exists on Drive: {TEST_FILE.exists()}")

print("\nIMPORTANT:")
print("Test data will NOT be loaded.")
print("Test data will NOT be used.")
print("Test predictions will NOT be generated.")

TEST_DATA_LOADED = False


# ======================================================================
# 5. LOAD FINAL 69-FEATURE LIST
# ======================================================================

print("\n" + "=" * 70)
print("LOADING FINAL FEATURE LIST")
print("=" * 70)

feature_df = pd.read_csv(FEATURE_FILE)

print("\nFeature-list shape:")
print(feature_df.shape)

print("\nFeature-list columns:")
print(feature_df.columns.tolist())


if "feature" not in feature_df.columns:

    raise ValueError(
        "Expected 'feature' column not found in feature list."
    )


FEATURES = feature_df["feature"].tolist()

print("\nNumber of selected features:")
print(len(FEATURES))

if len(FEATURES) != 69:

    raise ValueError(
        f"Expected exactly 69 features, found {len(FEATURES)}."
    )

print("\n✓ Exactly 69 final predictors confirmed.")

print("\nFirst 10 features:")
print(FEATURES[:10])

print("\nLast 10 features:")
print(FEATURES[-10:])


# ======================================================================
# 6. LOAD TRAINING DATA
# ======================================================================

print("\n" + "=" * 70)
print("LOADING TRAINING DATA")
print("=" * 70)

train_df = pd.read_csv(TRAIN_FILE)

print("\nTraining shape:")
print(train_df.shape)

print(f"\nTraining rows: {len(train_df):,}")


# ======================================================================
# 7. LOAD VALIDATION DATA
# ======================================================================

print("\n" + "=" * 70)
print("LOADING VALIDATION DATA")
print("=" * 70)

validation_df = pd.read_csv(VALIDATION_FILE)

print("\nValidation shape:")
print(validation_df.shape)

print(f"\nValidation rows: {len(validation_df):,}")


# ======================================================================
# 8. TARGET CHECK
# ======================================================================

TARGET = "chlorophyll_a_log1p"
ORIGINAL_TARGET = "chlorophyll_a"

print("\n" + "=" * 70)
print("TARGET CHECK")
print("=" * 70)

print(f"\nModel target:")
print(TARGET)

print(f"\nOriginal CHL target:")
print(ORIGINAL_TARGET)


for dataset_name, df in [
    ("Training", train_df),
    ("Validation", validation_df)
]:

    print(f"\n{dataset_name} target check:")

    if TARGET not in df.columns:

        raise ValueError(
            f"{TARGET} missing from {dataset_name} dataset."
        )

    if ORIGINAL_TARGET not in df.columns:

        raise ValueError(
            f"{ORIGINAL_TARGET} missing from {dataset_name} dataset."
        )

    print("✓ Required target columns available.")


# ======================================================================
# 9. FEATURE EXISTENCE CHECK
# ======================================================================

print("\n" + "=" * 70)
print("FEATURE EXISTENCE CHECK")
print("=" * 70)


train_missing = [
    f for f in FEATURES
    if f not in train_df.columns
]

validation_missing = [
    f for f in FEATURES
    if f not in validation_df.columns
]


print("\nTraining missing features:")
print(len(train_missing))

if train_missing:
    print(train_missing)


print("\nValidation missing features:")
print(len(validation_missing))

if validation_missing:
    print(validation_missing)


if train_missing or validation_missing:

    raise ValueError(
        "One or more selected model features are missing."
    )

print("\n✓ All 69 model features available in both datasets.")


# ======================================================================
# 10. FEATURE ORDER CHECK
# ======================================================================

print("\n" + "=" * 70)
print("FEATURE ORDER CHECK")
print("=" * 70)

print("\nFeature order will follow:")
print("final_model_feature_columns.csv")

print("\n✓ Fixed 69-feature order established.")


# ======================================================================
# 11. DATA SAFETY CHECK
# ======================================================================

print("\n" + "=" * 70)
print("TRAINING / VALIDATION DATA SAFETY CHECK")
print("=" * 70)


for dataset_name, df in [
    ("Training", train_df),
    ("Validation", validation_df)
]:

    X = df[FEATURES]

    y_log = df[TARGET]

    y_original = df[ORIGINAL_TARGET]


    predictor_nan = int(
        X.isna().sum().sum()
    )

    predictor_inf = int(
        np.isinf(X.to_numpy()).sum()
    )

    log_nan = int(
        y_log.isna().sum()
    )

    original_nan = int(
        y_original.isna().sum()
    )

    print(f"\n{dataset_name}")

    print(
        f"Predictor NaN      : {predictor_nan:,}"
    )

    print(
        f"Predictor infinite : {predictor_inf:,}"
    )

    print(
        f"Log target NaN     : {log_nan:,}"
    )

    print(
        f"Original CHL NaN   : {original_nan:,}"
    )


    if (
        predictor_nan > 0
        or predictor_inf > 0
        or log_nan > 0
        or original_nan > 0
    ):

        raise ValueError(
            f"{dataset_name} failed data safety checks."
        )


print("\n✓ Training and validation data passed safety checks.")


# ======================================================================
# 12. CURRENT XGBOOST BASELINE CONFIGURATION
# ======================================================================

print("\n" + "=" * 70)
print("CURRENT XGBOOST BASELINE CONFIGURATION")
print("=" * 70)


baseline_config = {

    "model": "XGBoost",

    "objective": "reg:squarederror",

    "n_estimators": 2000,

    "max_depth": 8,

    "learning_rate": 0.03,

    "subsample": 0.8,

    "colsample_bytree": 0.8,

    "min_child_weight": 5,

    "reg_alpha": 0.0,

    "reg_lambda": 1.0,

    "random_state": 42,

    "tree_method": "hist",

    "device": "cuda",

    "early_stopping_rounds": 100,

    "target": TARGET,

    "original_target": ORIGINAL_TARGET,

    "number_of_features": 69,

    "training_rows": int(len(train_df)),

    "validation_rows": int(len(validation_df)),

    "test_data_loaded": False,

    "model_retrained": False,

    "model_modified": False

}


for key, value in baseline_config.items():

    print(
        f"{key:<25}: {value}"
    )


# ======================================================================
# 13. EXISTING BASELINE MODEL INFORMATION
# ======================================================================

print("\n" + "=" * 70)
print("EXISTING BASELINE MODEL")
print("=" * 70)


EXISTING_MODEL = (
    XGB_DIR /
    "xgboost_chlorophyll_model_early_stopping.json"
)

print("\nExisting model:")
print(EXISTING_MODEL)

print(
    f"\nExists: {EXISTING_MODEL.exists()}"
)


if not EXISTING_MODEL.exists():

    raise FileNotFoundError(
        "Existing early-stopped XGBoost model not found."
    )


print("\n✓ Existing XGBoost model remains untouched.")


# ======================================================================
# 14. SAVE BASELINE CONFIGURATION
# ======================================================================

print("\n" + "=" * 70)
print("SAVING BASELINE CONFIGURATION")
print("=" * 70)


baseline_json = (
    OPTIMIZATION_DIR /
    "xgboost_baseline_configuration.json"
)

with open(
    baseline_json,
    "w"
) as f:

    json.dump(
        baseline_config,
        f,
        indent=4
    )


print("\nSaved:")
print(baseline_json)


# ======================================================================
# 15. FINAL SAFETY CHECK
# ======================================================================

print("\n" + "=" * 70)
print("FINAL 31.9P-16T-01 SAFETY CHECK")
print("=" * 70)

print(
    "\nTraining data loaded       : TRUE"
)

print(
    "Validation data loaded     : TRUE"
)

print(
    "Test data loaded           : FALSE"
)

print(
    "Test data used             : FALSE"
)

print(
    "Model retraining           : FALSE"
)

print(
    "Hyperparameters changed    : FALSE"
)

print(
    "Existing model modified    : FALSE"
)


print("\n✓ Baseline verification completed.")
print("✓ 69 predictors confirmed.")
print("✓ Training and validation datasets verified.")
print("✓ Current XGBoost configuration recorded.")
print("✓ Test dataset was NOT loaded.")
print("✓ Existing XGBoost model was NOT modified.")


print("\n" + "=" * 70)
print("STEP 31.9P-16T-01 COMPLETE")
print("=" * 70)

print(
    "\nNEXT STEP:"
)

print(
    "31.9P-16T-02 — Baseline Validation Reproduction"
)

print(
    "\nDO NOT TUNE PARAMETERS YET."
)

STEP 31.9P-16T-01
XGBOOST HYPERPARAMETER OPTIMIZATION
BASELINE VERIFICATION
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

PROJECT PATHS

Project:
/content/drive/MyDrive/Indhi_Marine_Research_Data

Optimization directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/02_XGBoost/hyperparameter_optimization

Feature list:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing/final_model_feature_columns.csv

Training data:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing/final_model_ready_train.csv

Validation data:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing/final_model_ready_validation.csv

Test data:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing/final_model_ready_test.csv

REQUIRED FILE CHECK

Feature list:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing/final_model_fe

In [ ]:
# ======================================================================
# STEP 31.9P-16T-02
# CORRECTED XGBOOST BASELINE VALIDATION REPRODUCTION
#
# IMPORTANT:
# - SAME baseline hyperparameters
# - SAME training data
# - SAME validation data
# - SAME 69 features
# - SAME target
# - EARLY STOPPING CONFIGURED IN XGBRegressor
# - TEST DATA NOT USED
# - EXISTING MODEL NOT MODIFIED
# ======================================================================


import time
import json
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


print("=" * 70)
print("STEP 31.9P-16T-02")
print("CORRECTED XGBOOST BASELINE VALIDATION REPRODUCTION")
print("=" * 70)


# ======================================================================
# 1. TEST DATA SAFETY
# ======================================================================

print("\n" + "=" * 70)
print("TEST DATA SAFETY")
print("=" * 70)

print("Test data loaded : FALSE")
print("Test data used   : FALSE")
print("Test predictions : FALSE")


# ======================================================================
# 2. BASELINE PARAMETERS
# ======================================================================

BASELINE_PARAMS = {

    "objective": "reg:squarederror",

    "n_estimators": 2000,

    "max_depth": 8,

    "learning_rate": 0.03,

    "subsample": 0.8,

    "colsample_bytree": 0.8,

    "min_child_weight": 5,

    "reg_alpha": 0.0,

    "reg_lambda": 1.0,

    "random_state": 42,

    "tree_method": "hist",

    "device": "cuda",

    "early_stopping_rounds": 100
}


print("\n" + "=" * 70)
print("BASELINE PARAMETERS")
print("=" * 70)

for key, value in BASELINE_PARAMS.items():

    print(
        f"{key:<25}: {value}"
    )


# ======================================================================
# 3. VERIFY INPUT ARRAYS ALREADY LOADED
# ======================================================================

print("\n" + "=" * 70)
print("INPUT ARRAY VERIFICATION")
print("=" * 70)

required_variables = [
    "X_train",
    "y_train",
    "X_validation",
    "y_validation_log",
    "y_validation_original"
]

for variable in required_variables:

    if variable not in globals():

        raise RuntimeError(
            f"Required variable '{variable}' is not available. "
            "Please rerun the previous preparation cells."
        )

    print(
        f"{variable:<25}: AVAILABLE"
    )


print("\nTraining X shape:")
print(X_train.shape)

print("\nTraining y shape:")
print(y_train.shape)

print("\nValidation X shape:")
print(X_validation.shape)

print("\nValidation y shape:")
print(y_validation_log.shape)


# ======================================================================
# 4. BUILD TEMPORARY BASELINE MODEL
#
# IMPORTANT:
# early_stopping_rounds is placed HERE rather than inside fit()
# because of the XGBoost version used by the current Colab runtime.
# ======================================================================

print("\n" + "=" * 70)
print("BUILDING TEMPORARY BASELINE MODEL")
print("=" * 70)


baseline_model = xgb.XGBRegressor(

    objective=BASELINE_PARAMS[
        "objective"
    ],

    n_estimators=BASELINE_PARAMS[
        "n_estimators"
    ],

    max_depth=BASELINE_PARAMS[
        "max_depth"
    ],

    learning_rate=BASELINE_PARAMS[
        "learning_rate"
    ],

    subsample=BASELINE_PARAMS[
        "subsample"
    ],

    colsample_bytree=BASELINE_PARAMS[
        "colsample_bytree"
    ],

    min_child_weight=BASELINE_PARAMS[
        "min_child_weight"
    ],

    reg_alpha=BASELINE_PARAMS[
        "reg_alpha"
    ],

    reg_lambda=BASELINE_PARAMS[
        "reg_lambda"
    ],

    random_state=BASELINE_PARAMS[
        "random_state"
    ],

    tree_method=BASELINE_PARAMS[
        "tree_method"
    ],

    device=BASELINE_PARAMS[
        "device"
    ],

    early_stopping_rounds=BASELINE_PARAMS[
        "early_stopping_rounds"
    ]
)


print("\n✓ Temporary baseline model created.")

print(
    "✓ early_stopping_rounds configured "
    "through XGBRegressor."
)

print(
    "✓ Existing XGBoost model remains untouched."
)


# ======================================================================
# 5. TRAIN TEMPORARY BASELINE
# ======================================================================

print("\n" + "=" * 70)
print("TRAINING TEMPORARY BASELINE")
print("=" * 70)

print("\nTraining data      : Training split only")
print("Validation data    : Validation split only")
print("Test data          : NOT USED")
print("Maximum trees      : 2000")
print("Early stopping     : 100 rounds")
print("GPU                : CUDA")
print("Random seed        : 42")


start_time = time.time()


baseline_model.fit(

    X_train,

    y_train,

    eval_set=[
        (
            X_train,
            y_train
        ),
        (
            X_validation,
            y_validation_log
        )
    ],

    verbose=100
)


training_time = (
    time.time() - start_time
)


print("\n✓ Temporary baseline training completed.")

print(
    f"Training time: "
    f"{training_time:.2f} seconds"
)


# ======================================================================
# 6. BEST ITERATION
# ======================================================================

print("\n" + "=" * 70)
print("BASELINE BEST ITERATION")
print("=" * 70)


best_iteration = int(
    baseline_model.best_iteration
)

best_score = float(
    baseline_model.best_score
)


print(
    f"\nBest iteration: "
    f"{best_iteration}"
)

print(
    f"Best validation RMSE (log1p): "
    f"{best_score:.12f}"
)

print(
    f"Best number of trees: "
    f"{best_iteration + 1}"
)


# ======================================================================
# 7. GENERATE VALIDATION PREDICTIONS
# ======================================================================

print("\n" + "=" * 70)
print("GENERATING BASELINE VALIDATION PREDICTIONS")
print("=" * 70)


pred_log = baseline_model.predict(
    X_validation
)

pred_original = np.expm1(
    pred_log
)


print(
    f"\nPrediction shape: "
    f"{pred_log.shape}"
)


# ======================================================================
# 8. LOG1P METRICS
# ======================================================================

log_mae = mean_absolute_error(
    y_validation_log,
    pred_log
)

log_rmse = np.sqrt(
    mean_squared_error(
        y_validation_log,
        pred_log
    )
)

log_r2 = r2_score(
    y_validation_log,
    pred_log
)

log_bias = np.mean(
    pred_log -
    y_validation_log
)


# ======================================================================
# 9. ORIGINAL CHL METRICS
# ======================================================================

original_mae = mean_absolute_error(
    y_validation_original,
    pred_original
)

original_rmse = np.sqrt(
    mean_squared_error(
        y_validation_original,
        pred_original
    )
)

original_r2 = r2_score(
    y_validation_original,
    pred_original
)

original_bias = np.mean(
    pred_original -
    y_validation_original
)


# ======================================================================
# 10. DISPLAY RESULTS
# ======================================================================

print("\n" + "=" * 70)
print("REPRODUCED BASELINE VALIDATION RESULTS")
print("=" * 70)


print("\nLOG1P SCALE")

print(
    f"MAE       : {log_mae:.6f}"
)

print(
    f"RMSE      : {log_rmse:.6f}"
)

print(
    f"R²        : {log_r2:.6f}"
)

print(
    f"Bias      : {log_bias:.6f}"
)


print("\nORIGINAL CHL SCALE")

print(
    f"MAE       : {original_mae:.6f}"
)

print(
    f"RMSE      : {original_rmse:.6f}"
)

print(
    f"R²        : {original_r2:.6f}"
)

print(
    f"Bias       : {original_bias:.6f}"
)


# ======================================================================
# 11. ESTABLISHED BASELINE
# ======================================================================

ESTABLISHED = {

    "original_mae":
        0.1894004434498657,

    "original_rmse":
        0.8060904960285136,

    "original_r2":
        0.5371393487829845,

    "log_mae":
        0.06996042122254419,

    "log_rmse":
        0.15257568941934896,

    "log_r2":
        0.8382274310440981
}


# ======================================================================
# 12. COMPARISON
# ======================================================================

print("\n" + "=" * 70)
print("COMPARISON WITH ESTABLISHED BASELINE")
print("=" * 70)


comparison_rows = [

    [
        "Original MAE",
        ESTABLISHED["original_mae"],
        original_mae,
        original_mae -
        ESTABLISHED["original_mae"]
    ],

    [
        "Original RMSE",
        ESTABLISHED["original_rmse"],
        original_rmse,
        original_rmse -
        ESTABLISHED["original_rmse"]
    ],

    [
        "Original R²",
        ESTABLISHED["original_r2"],
        original_r2,
        original_r2 -
        ESTABLISHED["original_r2"]
    ],

    [
        "Log MAE",
        ESTABLISHED["log_mae"],
        log_mae,
        log_mae -
        ESTABLISHED["log_mae"]
    ],

    [
        "Log RMSE",
        ESTABLISHED["log_rmse"],
        log_rmse,
        log_rmse -
        ESTABLISHED["log_rmse"]
    ],

    [
        "Log R²",
        ESTABLISHED["log_r2"],
        log_r2,
        log_r2 -
        ESTABLISHED["log_r2"]
    ]
]


comparison_df = pd.DataFrame(

    comparison_rows,

    columns=[
        "metric",
        "established_baseline",
        "reproduced_baseline",
        "difference"
    ]
)


print(
    comparison_df.to_string(
        index=False
    )
)


# ======================================================================
# 13. REPRODUCTION CHECK
# ======================================================================

print("\n" + "=" * 70)
print("REPRODUCTION CHECK")
print("=" * 70)


# GPU/XGBoost execution can produce extremely small numerical
# differences. We therefore use a practical tolerance.

TOLERANCE = 1e-5


metric_differences = {

    "Original MAE":
        abs(
            original_mae -
            ESTABLISHED["original_mae"]
        ),

    "Original RMSE":
        abs(
            original_rmse -
            ESTABLISHED["original_rmse"]
        ),

    "Original R²":
        abs(
            original_r2 -
            ESTABLISHED["original_r2"]
        ),

    "Log MAE":
        abs(
            log_mae -
            ESTABLISHED["log_mae"]
        ),

    "Log RMSE":
        abs(
            log_rmse -
            ESTABLISHED["log_rmse"]
        ),

    "Log R²":
        abs(
            log_r2 -
            ESTABLISHED["log_r2"]
        )
}


all_within_tolerance = True


for metric, difference in metric_differences.items():

    status = (
        "PASS"
        if difference <= TOLERANCE
        else "CHECK"
    )

    print(
        f"{metric:<20} "
        f"difference = "
        f"{difference:.10f} "
        f"[{status}]"
    )

    if difference > TOLERANCE:

        all_within_tolerance = False


# ======================================================================
# 14. SAVE RESULTS
# ======================================================================

print("\n" + "=" * 70)
print("SAVING BASELINE REPRODUCTION RESULTS")
print("=" * 70)


OPTIMIZATION_DIR = Path(
    "/content/drive/MyDrive/"
    "Indhi_Marine_Research_Data/"
    "06_models/02_XGBoost/"
    "hyperparameter_optimization"
)

OPTIMIZATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)


results = {

    "experiment":
        "31.9P-16T-02",

    "model":
        "XGBoost",

    "experiment_type":
        "baseline_validation_reproduction",

    "training_rows":
        int(len(X_train)),

    "validation_rows":
        int(len(X_validation)),

    "features":
        69,

    "target":
        "chlorophyll_a_log1p",

    "test_data_loaded":
        False,

    "test_data_used":
        False,

    "model_modified":
        False,

    "hyperparameters_changed":
        False,

    "best_iteration":
        best_iteration,

    "best_number_of_trees":
        best_iteration + 1,

    "best_validation_log_rmse":
        best_score,

    "log_mae":
        float(log_mae),

    "log_rmse":
        float(log_rmse),

    "log_r2":
        float(log_r2),

    "log_bias":
        float(log_bias),

    "original_mae":
        float(original_mae),

    "original_rmse":
        float(original_rmse),

    "original_r2":
        float(original_r2),

    "original_bias":
        float(original_bias),

    "training_time_seconds":
        float(training_time),

    "reproduction_within_tolerance":
        bool(all_within_tolerance)
}


results_file = (
    OPTIMIZATION_DIR /
    "xgboost_baseline_reproduction_results.json"
)


with open(
    results_file,
    "w"
) as f:

    json.dump(
        results,
        f,
        indent=4
    )


comparison_file = (
    OPTIMIZATION_DIR /
    "xgboost_baseline_reproduction_comparison.csv"
)


comparison_df.to_csv(
    comparison_file,
    index=False
)


print("\nSaved:")
print(results_file)

print(comparison_file)


# ======================================================================
# 15. FINAL SAFETY CHECK
# ======================================================================

print("\n" + "=" * 70)
print("FINAL 31.9P-16T-02 SAFETY CHECK")
print("=" * 70)


print("\nTraining data used       : TRUE")
print("Validation data used     : TRUE")
print("Test data loaded         : FALSE")
print("Test data used           : FALSE")
print("Temporary model trained  : TRUE")
print("Existing model modified  : FALSE")
print("Hyperparameters changed  : FALSE")


if all_within_tolerance:

    print(
        "\n✓ BASELINE REPRODUCTION PASSED."
    )

    print(
        "✓ Reproduced metrics match "
        "the established baseline "
        "within tolerance."
    )

else:

    print(
        "\n⚠ BASELINE REPRODUCTION "
        "REQUIRES INVESTIGATION."
    )

    print(
        "Do NOT begin hyperparameter "
        "tuning yet."
    )


print("\n" + "=" * 70)
print("STEP 31.9P-16T-02 COMPLETE")
print("=" * 70)


if all_within_tolerance:

    print(
        "\nNEXT STEP:"
    )

    print(
        "31.9P-16T-03 — Controlled "
        "Hyperparameter Tuning Stage A"
    )

else:

    print(
        "\nNEXT STEP:"
    )

    print(
        "Investigate baseline "
        "reproduction difference."
    )

STEP 31.9P-16T-02
CORRECTED XGBOOST BASELINE VALIDATION REPRODUCTION

TEST DATA SAFETY
Test data loaded : FALSE
Test data used   : FALSE
Test predictions : FALSE

BASELINE PARAMETERS
objective                : reg:squarederror
n_estimators             : 2000
max_depth                : 8
learning_rate            : 0.03
subsample                : 0.8
colsample_bytree         : 0.8
min_child_weight         : 5
reg_alpha                : 0.0
reg_lambda               : 1.0
random_state             : 42
tree_method              : hist
device                   : cuda
early_stopping_rounds    : 100

INPUT ARRAY VERIFICATION
X_train                  : AVAILABLE
y_train                  : AVAILABLE
X_validation             : AVAILABLE
y_validation_log         : AVAILABLE
y_validation_original    : AVAILABLE

Training X shape:
(736950, 69)

Training y shape:
(736950,)

Validation X shape:
(157896, 69)

Validation y shape:
(157896,)

BUILDING TEMPORARY BASELINE MODEL

✓ Temporary baseline model cr

/usr/local/lib/python3.13/dist-packages/xgboost/callback.py:385: UserWarning: [15:14:07] WARNING: /__w/xgboost/xgboost/src/context.cc:55: No visible GPU is found, setting device to CPU.
  self.starting_round = model.num_boosted_rounds()
/usr/local/lib/python3.13/dist-packages/xgboost/callback.py:385: UserWarning: [15:14:07] WARNING: /__w/xgboost/xgboost/src/context.cc:218: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  self.starting_round = model.num_boosted_rounds()


[0]	validation_0-rmse:0.26263	validation_1-rmse:0.37472
[100]	validation_0-rmse:0.07866	validation_1-rmse:0.15298
[200]	validation_0-rmse:0.06809	validation_1-rmse:0.15536
[214]	validation_0-rmse:0.06710	validation_1-rmse:0.15551

✓ Temporary baseline training completed.
Training time: 83.23 seconds

BASELINE BEST ITERATION

Best iteration: 114
Best validation RMSE (log1p): 0.152169028041
Best number of trees: 115

GENERATING BASELINE VALIDATION PREDICTIONS

Prediction shape: (157896,)

REPRODUCED BASELINE VALIDATION RESULTS

LOG1P SCALE
MAE       : 0.069965
RMSE      : 0.152169
R²        : 0.839089
Bias      : -0.010765

ORIGINAL CHL SCALE
MAE       : 0.189216
RMSE      : 0.803659
R²        : 0.539928
Bias       : -0.078427

COMPARISON WITH ESTABLISHED BASELINE
       metric  established_baseline  reproduced_baseline  difference
 Original MAE              0.189400             0.189216   -0.000185
Original RMSE              0.806090             0.803659   -0.002432
  Original R²       

In [ ]:
# ======================================================================
# STEP 31.9P-16T-02-FIX
# GPU AVAILABILITY VERIFICATION
# ======================================================================

import subprocess
import xgboost as xgb

print("=" * 70)
print("STEP 31.9P-16T-02-FIX")
print("GPU AVAILABILITY VERIFICATION")
print("=" * 70)


# ======================================================================
# 1. XGBOOST VERSION
# ======================================================================

print("\n" + "=" * 70)
print("XGBOOST VERSION")
print("=" * 70)

print(
    "XGBoost version:",
    xgb.__version__
)


# ======================================================================
# 2. NVIDIA GPU CHECK
# ======================================================================

print("\n" + "=" * 70)
print("NVIDIA GPU CHECK")
print("=" * 70)

try:

    result = subprocess.run(
        [
            "nvidia-smi",
            "--query-gpu=name",
            "--format=csv,noheader"
        ],
        capture_output=True,
        text=True
    )

    if result.returncode == 0:

        gpu_names = result.stdout.strip()

        if gpu_names:

            print("GPU detected:")
            print(gpu_names)

            GPU_AVAILABLE = True

        else:

            print("No NVIDIA GPU detected.")
            GPU_AVAILABLE = False

    else:

        print("nvidia-smi could not detect a GPU.")
        GPU_AVAILABLE = False

except Exception as e:

    print("GPU check failed:")
    print(e)

    GPU_AVAILABLE = False


# ======================================================================
# 3. XGBOOST CUDA BUILD INFORMATION
# ======================================================================

print("\n" + "=" * 70)
print("XGBOOST BUILD INFORMATION")
print("=" * 70)

try:

    print(
        xgb.build_info()
    )

except Exception as e:

    print(
        "Could not retrieve XGBoost build information:"
    )

    print(e)


# ======================================================================
# 4. FINAL DECISION
# ======================================================================

print("\n" + "=" * 70)
print("GPU VERIFICATION RESULT")
print("=" * 70)

if GPU_AVAILABLE:

    print("✓ NVIDIA GPU is available.")
    print("✓ We can attempt GPU-controlled baseline reproduction.")

else:

    print("⚠ NVIDIA GPU is NOT available.")
    print()
    print(
        "DO NOT START HYPERPARAMETER TUNING YET."
    )

    print(
        "Reconnect/restart the Colab GPU runtime first."
    )

STEP 31.9P-16T-02-FIX
GPU AVAILABILITY VERIFICATION

XGBOOST VERSION
XGBoost version: 3.4.1

NVIDIA GPU CHECK
GPU detected:
Tesla T4

XGBOOST BUILD INFORMATION
{'BUILTIN_PREFETCH_PRESENT': True, 'CUDA_VERSION': [13, 3], 'DEBUG': False, 'GCC_VERSION': [10, 3, 1], 'GLIBC_VERSION': [2, 28], 'MM_PREFETCH_PRESENT': True, 'NCCL_VERSION': [2, 30, 7], 'THRUST_VERSION': [3, 3, 3], 'USE_CUDA': True, 'USE_DLOPEN_NCCL': True, 'USE_FEDERATED': False, 'USE_NCCL': True, 'USE_NVCOMP': False, 'USE_OPENMP': True, 'USE_RMM': False, 'libxgboost': '/usr/local/lib/python3.13/dist-packages/xgboost/lib/libxgboost.so'}

GPU VERIFICATION RESULT
✓ NVIDIA GPU is available.
✓ We can attempt GPU-controlled baseline reproduction.


In [ ]:
# ======================================================================
# STEP 31.9P-16T-02-GPU
# GPU-CONTROLLED XGBOOST BASELINE VALIDATION REPRODUCTION
# ======================================================================

import os
import gc
import json
import time
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from google.colab import drive


print("=" * 70)
print("STEP 31.9P-16T-02-GPU")
print("GPU-CONTROLLED XGBOOST BASELINE VALIDATION REPRODUCTION")
print("=" * 70)


# ======================================================================
# 1. MOUNT DRIVE
# ======================================================================

drive.mount(
    "/content/drive",
    force_remount=False
)


# ======================================================================
# 2. PATHS
# ======================================================================

PROJECT_DIR = (
    "/content/drive/MyDrive/"
    "Indhi_Marine_Research_Data"
)

OPT_DIR = os.path.join(
    PROJECT_DIR,
    "06_models",
    "02_XGBoost",
    "hyperparameter_optimization"
)

FEATURE_FILE = os.path.join(
    PROJECT_DIR,
    "05_model_preprocessing",
    "final_model_feature_columns.csv"
)

TRAIN_FILE = os.path.join(
    PROJECT_DIR,
    "05_model_preprocessing",
    "final_model_ready_train.csv"
)

VALIDATION_FILE = os.path.join(
    PROJECT_DIR,
    "05_model_preprocessing",
    "final_model_ready_validation.csv"
)

TEST_FILE = os.path.join(
    PROJECT_DIR,
    "05_model_preprocessing",
    "final_model_ready_test.csv"
)

os.makedirs(
    OPT_DIR,
    exist_ok=True
)


print("\n" + "=" * 70)
print("PROJECT PATHS")
print("=" * 70)

print("Project:")
print(PROJECT_DIR)

print("\nOptimization directory:")
print(OPT_DIR)

print("\nFeature list:")
print(FEATURE_FILE)

print("\nTraining data:")
print(TRAIN_FILE)

print("\nValidation data:")
print(VALIDATION_FILE)

print("\nTest data:")
print(TEST_FILE)


# ======================================================================
# 3. TEST DATA SAFETY
# ======================================================================

print("\n" + "=" * 70)
print("TEST DATA SAFETY")
print("=" * 70)

print(
    "Test dataset exists on Drive:",
    os.path.exists(TEST_FILE)
)

print()
print("Test data will NOT be loaded.")
print("Test data will NOT be used.")
print("Test predictions will NOT be generated.")


# ======================================================================
# 4. REQUIRED FILE CHECK
# ======================================================================

print("\n" + "=" * 70)
print("REQUIRED FILE CHECK")
print("=" * 70)

required_files = {
    "Feature list": FEATURE_FILE,
    "Training data": TRAIN_FILE,
    "Validation data": VALIDATION_FILE
}

for name, path in required_files.items():

    exists = os.path.exists(path)

    print(f"\n{name}:")
    print(path)
    print("Exists:", exists)

    if not exists:
        raise FileNotFoundError(
            f"Required file not found: {path}"
        )

print("\n✓ All training/validation files found.")


# ======================================================================
# 5. LOAD FEATURE LIST
# ======================================================================

print("\n" + "=" * 70)
print("LOADING FINAL FEATURE LIST")
print("=" * 70)

feature_df = pd.read_csv(
    FEATURE_FILE
)

features = feature_df["feature"].tolist()

print(
    "Number of selected features:",
    len(features)
)

if len(features) != 69:

    raise ValueError(
        f"Expected 69 features, found {len(features)}"
    )

print("✓ Exactly 69 predictors confirmed.")


# ======================================================================
# 6. LOAD TRAINING DATA
# ======================================================================

print("\n" + "=" * 70)
print("LOADING TRAINING DATA")
print("=" * 70)

train_df = pd.read_csv(
    TRAIN_FILE
)

print(
    "Training shape:",
    train_df.shape
)


# ======================================================================
# 7. LOAD VALIDATION DATA
# ======================================================================

print("\n" + "=" * 70)
print("LOADING VALIDATION DATA")
print("=" * 70)

validation_df = pd.read_csv(
    VALIDATION_FILE
)

print(
    "Validation shape:",
    validation_df.shape
)


# ======================================================================
# 8. TARGET CONFIGURATION
# ======================================================================

TARGET_LOG = "chlorophyll_a_log1p"
TARGET_ORIGINAL = "chlorophyll_a"

print("\n" + "=" * 70)
print("TARGET CONFIGURATION")
print("=" * 70)

print("Model target:", TARGET_LOG)
print("Original target:", TARGET_ORIGINAL)


# ======================================================================
# 9. COLUMN CHECK
# ======================================================================

required_columns = (
    features
    + [
        TARGET_LOG,
        TARGET_ORIGINAL
    ]
)

train_missing = [
    c for c in required_columns
    if c not in train_df.columns
]

validation_missing = [
    c for c in required_columns
    if c not in validation_df.columns
]

print("\n" + "=" * 70)
print("COLUMN CHECK")
print("=" * 70)

print(
    "Training missing columns:",
    len(train_missing)
)

print(
    "Validation missing columns:",
    len(validation_missing)
)

if train_missing:
    raise ValueError(
        f"Training missing columns: {train_missing}"
    )

if validation_missing:
    raise ValueError(
        f"Validation missing columns: {validation_missing}"
    )

print("✓ All required columns available.")


# ======================================================================
# 10. PREPARE INPUT ARRAYS
# ======================================================================

print("\n" + "=" * 70)
print("PREPARING MODEL INPUTS")
print("=" * 70)

X_train = train_df[
    features
].to_numpy(
    dtype=np.float32
)

y_train = train_df[
    TARGET_LOG
].to_numpy(
    dtype=np.float32
)

X_validation = validation_df[
    features
].to_numpy(
    dtype=np.float32
)

y_validation_log = validation_df[
    TARGET_LOG
].to_numpy(
    dtype=np.float32
)

y_validation_original = validation_df[
    TARGET_ORIGINAL
].to_numpy(
    dtype=np.float32
)

print(
    "Training X:",
    X_train.shape
)

print(
    "Training y:",
    y_train.shape
)

print(
    "Validation X:",
    X_validation.shape
)

print(
    "Validation y:",
    y_validation_log.shape
)


# ======================================================================
# 11. DATA SAFETY CHECK
# ======================================================================

print("\n" + "=" * 70)
print("DATA SAFETY CHECK")
print("=" * 70)

checks = {
    "Training predictor NaN":
        np.isnan(X_train).sum(),

    "Training predictor infinite":
        np.isinf(X_train).sum(),

    "Training target NaN":
        np.isnan(y_train).sum(),

    "Validation predictor NaN":
        np.isnan(X_validation).sum(),

    "Validation predictor infinite":
        np.isinf(X_validation).sum(),

    "Validation log target NaN":
        np.isnan(y_validation_log).sum(),

    "Validation original target NaN":
        np.isnan(y_validation_original).sum()
}

for name, value in checks.items():

    print(
        f"{name:<40}: {value}"
    )

if any(value != 0 for value in checks.values()):

    raise ValueError(
        "Data safety check failed."
    )

print("\n✓ Training and validation data are clean.")


# ======================================================================
# 12. EXACT BASELINE PARAMETERS
# ======================================================================

params = {

    "objective":
        "reg:squarederror",

    "n_estimators":
        2000,

    "max_depth":
        8,

    "learning_rate":
        0.03,

    "subsample":
        0.8,

    "colsample_bytree":
        0.8,

    "min_child_weight":
        5,

    "reg_alpha":
        0.0,

    "reg_lambda":
        1.0,

    "random_state":
        42,

    "tree_method":
        "hist",

    "device":
        "cuda",

    "early_stopping_rounds":
        100
}


print("\n" + "=" * 70)
print("EXACT BASELINE PARAMETERS")
print("=" * 70)

for key, value in params.items():

    print(
        f"{key:<25}: {value}"
    )


# ======================================================================
# 13. GPU VERIFICATION
# ======================================================================

print("\n" + "=" * 70)
print("GPU VERIFICATION")
print("=" * 70)

print(
    "XGBoost version:",
    xgb.__version__
)

try:

    build_info = xgb.build_info()

    print(
        "CUDA version:",
        build_info.get("CUDA_VERSION")
    )

    print(
        "CUDA enabled:",
        build_info.get("USE_CUDA")
    )

except Exception as e:

    print(
        "Build information unavailable:",
        e
    )

try:

    import subprocess

    gpu_result = subprocess.run(
        [
            "nvidia-smi",
            "--query-gpu=name",
            "--format=csv,noheader"
        ],
        capture_output=True,
        text=True
    )

    gpu_name = gpu_result.stdout.strip()

    print(
        "GPU:",
        gpu_name
    )

    if "Tesla T4" not in gpu_name:

        print(
            "⚠ GPU is not Tesla T4."
        )

    else:

        print(
            "✓ Tesla T4 confirmed."
        )

except Exception as e:

    raise RuntimeError(
        f"Unable to verify GPU: {e}"
    )


# ======================================================================
# 14. BUILD TEMPORARY BASELINE MODEL
# ======================================================================

print("\n" + "=" * 70)
print("BUILDING TEMPORARY BASELINE MODEL")
print("=" * 70)

baseline_model = xgb.XGBRegressor(
    objective=params["objective"],
    n_estimators=params["n_estimators"],
    max_depth=params["max_depth"],
    learning_rate=params["learning_rate"],
    subsample=params["subsample"],
    colsample_bytree=params["colsample_bytree"],
    min_child_weight=params["min_child_weight"],
    reg_alpha=params["reg_alpha"],
    reg_lambda=params["reg_lambda"],
    random_state=params["random_state"],
    tree_method=params["tree_method"],
    device=params["device"],
    early_stopping_rounds=params[
        "early_stopping_rounds"
    ]
)

print(
    "✓ Temporary GPU baseline model created."
)

print(
    "✓ Existing XGBoost model remains untouched."
)


# ======================================================================
# 15. TRAIN
# ======================================================================

print("\n" + "=" * 70)
print("GPU-CONTROLLED BASELINE TRAINING")
print("=" * 70)

print("Training data      : Training split only")
print("Validation data    : Validation split only")
print("Test data          : NOT USED")
print("GPU                : Tesla T4")
print("Maximum trees      : 2000")
print("Early stopping     : 100 rounds")
print("Random seed        : 42")

start_time = time.time()

baseline_model.fit(
    X_train,
    y_train,

    eval_set=[
        (
            X_train,
            y_train
        ),
        (
            X_validation,
            y_validation_log
        )
    ],

    verbose=100
)

training_time = (
    time.time()
    - start_time
)

print(
    "\n✓ GPU baseline training completed."
)

print(
    "Training time:",
    round(training_time, 2),
    "seconds"
)


# ======================================================================
# 16. BEST ITERATION
# ======================================================================

print("\n" + "=" * 70)
print("BASELINE BEST ITERATION")
print("=" * 70)

best_iteration = (
    baseline_model.best_iteration
)

best_score = (
    baseline_model.best_score
)

best_trees = (
    best_iteration + 1
)

print(
    "Best iteration:",
    best_iteration
)

print(
    "Best validation RMSE:",
    best_score
)

print(
    "Best number of trees:",
    best_trees
)


# ======================================================================
# 17. VALIDATION PREDICTIONS
# ======================================================================

print("\n" + "=" * 70)
print("GENERATING BASELINE VALIDATION PREDICTIONS")
print("=" * 70)

pred_log = baseline_model.predict(
    X_validation
)

pred_original = np.expm1(
    pred_log
)

print(
    "Prediction shape:",
    pred_log.shape
)


# ======================================================================
# 18. LOG1P METRICS
# ======================================================================

log_mae = mean_absolute_error(
    y_validation_log,
    pred_log
)

log_rmse = np.sqrt(
    mean_squared_error(
        y_validation_log,
        pred_log
    )
)

log_r2 = r2_score(
    y_validation_log,
    pred_log
)

log_bias = np.mean(
    pred_log - y_validation_log
)


# ======================================================================
# 19. ORIGINAL CHL METRICS
# ======================================================================

original_mae = mean_absolute_error(
    y_validation_original,
    pred_original
)

original_rmse = np.sqrt(
    mean_squared_error(
        y_validation_original,
        pred_original
    )
)

original_r2 = r2_score(
    y_validation_original,
    pred_original
)

original_bias = np.mean(
    pred_original
    - y_validation_original
)


# ======================================================================
# 20. DISPLAY RESULTS
# ======================================================================

print("\n" + "=" * 70)
print("GPU-REPRODUCED BASELINE VALIDATION RESULTS")
print("=" * 70)

print("\nLOG1P SCALE")

print(
    f"MAE       : {log_mae:.6f}"
)

print(
    f"RMSE      : {log_rmse:.6f}"
)

print(
    f"R²        : {log_r2:.6f}"
)

print(
    f"Bias      : {log_bias:.6f}"
)


print("\nORIGINAL CHL SCALE")

print(
    f"MAE       : {original_mae:.6f}"
)

print(
    f"RMSE      : {original_rmse:.6f}"
)

print(
    f"R²        : {original_r2:.6f}"
)

print(
    f"Bias      : {original_bias:.6f}"
)


# ======================================================================
# 21. ESTABLISHED BASELINE
# ======================================================================

established = {

    "Original MAE":
        0.1894004434498657,

    "Original RMSE":
        0.8060904960285136,

    "Original R2":
        0.5371393487829845,

    "Log MAE":
        0.06996042122254419,

    "Log RMSE":
        0.15257568941934896,

    "Log R2":
        0.8382274310440981
}

reproduced = {

    "Original MAE":
        original_mae,

    "Original RMSE":
        original_rmse,

    "Original R2":
        original_r2,

    "Log MAE":
        log_mae,

    "Log RMSE":
        log_rmse,

    "Log R2":
        log_r2
}


comparison_rows = []

for metric in established:

    difference = (
        reproduced[metric]
        - established[metric]
    )

    comparison_rows.append({

        "metric":
            metric,

        "established_baseline":
            established[metric],

        "gpu_reproduced_baseline":
            reproduced[metric],

        "difference":
            difference,

        "absolute_difference":
            abs(difference)
    })


comparison_df = pd.DataFrame(
    comparison_rows
)


print("\n" + "=" * 70)
print("COMPARISON WITH ESTABLISHED BASELINE")
print("=" * 70)

print(
    comparison_df.to_string(
        index=False
    )
)


# ======================================================================
# 22. REPRODUCTION INTERPRETATION
# ======================================================================

print("\n" + "=" * 70)
print("REPRODUCTION INTERPRETATION")
print("=" * 70)

print(
    "Established model:"
)

print(
    "Best iteration = 128"
)

print(
    "Best trees = 129"
)

print(
    "Log RMSE = 0.152576"
)

print(
    "\nGPU reproduction:"
)

print(
    f"Best iteration = {best_iteration}"
)

print(
    f"Best trees = {best_trees}"
)

print(
    f"Log RMSE = {log_rmse:.6f}"
)


# ======================================================================
# 23. SAVE RESULTS
# ======================================================================

reproduction_results = {

    "experiment":
        "31.9P-16T-02-GPU",

    "model":
        "XGBoost",

    "device":
        "cuda",

    "gpu":
        "Tesla T4",

    "xgboost_version":
        xgb.__version__,

    "training_rows":
        int(len(X_train)),

    "validation_rows":
        int(len(X_validation)),

    "features":
        int(len(features)),

    "best_iteration":
        int(best_iteration),

    "best_number_of_trees":
        int(best_trees),

    "best_validation_rmse_log1p":
        float(best_score),

    "training_time_seconds":
        float(training_time),

    "validation_metrics": {

        "original_mae":
            float(original_mae),

        "original_rmse":
            float(original_rmse),

        "original_r2":
            float(original_r2),

        "original_bias":
            float(original_bias),

        "log_mae":
            float(log_mae),

        "log_rmse":
            float(log_rmse),

        "log_r2":
            float(log_r2),

        "log_bias":
            float(log_bias)
    },

    "test_data_loaded":
        False,

    "test_data_used":
        False,

    "existing_model_modified":
        False,

    "hyperparameters_changed":
        False
}


results_path = os.path.join(
    OPT_DIR,
    "xgboost_gpu_baseline_reproduction_results.json"
)

comparison_path = os.path.join(
    OPT_DIR,
    "xgboost_gpu_baseline_reproduction_comparison.csv"
)

with open(
    results_path,
    "w"
) as f:

    json.dump(
        reproduction_results,
        f,
        indent=4
    )


comparison_df.to_csv(
    comparison_path,
    index=False
)


print("\n" + "=" * 70)
print("RESULTS SAVED")
print("=" * 70)

print(
    results_path
)

print(
    comparison_path
)


# ======================================================================
# 24. FINAL SAFETY CHECK
# ======================================================================

print("\n" + "=" * 70)
print("FINAL 31.9P-16T-02-GPU SAFETY CHECK")
print("=" * 70)

print(
    "Training data used       : TRUE"
)

print(
    "Validation data used     : TRUE"
)

print(
    "Test data loaded         : FALSE"
)

print(
    "Test data used           : FALSE"
)

print(
    "Temporary model trained  : TRUE"
)

print(
    "Existing model modified  : FALSE"
)

print(
    "Hyperparameters changed  : FALSE"
)

print(
    "GPU used                 : TRUE"
)

print(
    "GPU                     : Tesla T4"
)

print(
    "\n✓ GPU-controlled baseline reproduction completed."
)

print(
    "✓ Existing XGBoost model remains untouched."
)

print(
    "✓ No test data was used."
)

print(
    "✓ No hyperparameters were changed."
)

print(
    "✓ Ready for baseline comparison."
)

print("\n" + "=" * 70)
print("STEP 31.9P-16T-02-GPU COMPLETE")
print("=" * 70)

STEP 31.9P-16T-02-GPU
GPU-CONTROLLED XGBOOST BASELINE VALIDATION REPRODUCTION
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

PROJECT PATHS
Project:
/content/drive/MyDrive/Indhi_Marine_Research_Data

Optimization directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/02_XGBoost/hyperparameter_optimization

Feature list:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing/final_model_feature_columns.csv

Training data:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing/final_model_ready_train.csv

Validation data:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing/final_model_ready_validation.csv

Test data:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing/final_model_ready_test.csv

TEST DATA SAFETY
Test dataset exists on Drive: True

Test data will NOT be loaded.
Test data will NOT be used.
Test predi

/usr/local/lib/python3.13/dist-packages/xgboost/core.py:569: UserWarning: [15:19:15] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Prediction shape: (157896,)

GPU-REPRODUCED BASELINE VALIDATION RESULTS

LOG1P SCALE
MAE       : 0.069960
RMSE      : 0.152576
R²        : 0.838227
Bias      : -0.010555

ORIGINAL CHL SCALE
MAE       : 0.189400
RMSE      : 0.806090
R²        : 0.537139
Bias      : -0.076182

COMPARISON WITH ESTABLISHED BASELINE
       metric  established_baseline  gpu_reproduced_baseline    difference  absolute_difference
 Original MAE              0.189400                 0.189400  5.945314e-09         5.945314e-09
Original RMSE              0.806090                 0.806090 -4.008807e-08         4.008807e-08
  Original R2              0.537139                 0.537139  6.695798e-08         6.695798e-08
      Log MAE              0.069960                 0.069960  1.591348e-09         1.591348e-09
     Log RMSE              0.152576                 0.152576 -2.099154e-10         2.099154e-10
       Log R2              0.838227                 0.838227  1.980353e-08         1.980353e-08

REPRODUCTION I

In [ ]:
# ======================================================================
# STEP 31.9P-16T-03
# XGBOOST TREE COMPLEXITY OPTIMIZATION
# CONTROLLED MAX_DEPTH EXPERIMENT
# ======================================================================

import os
import json
import time
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("=" * 70)
print("STEP 31.9P-16T-03")
print("XGBOOST TREE COMPLEXITY OPTIMIZATION")
print("=" * 70)

# ======================================================================
# PROJECT PATHS
# ======================================================================

PROJECT_DIR = "/content/drive/MyDrive/Indhi_Marine_Research_Data"

PREPROCESS_DIR = os.path.join(
    PROJECT_DIR,
    "05_model_preprocessing"
)

OPT_DIR = os.path.join(
    PROJECT_DIR,
    "06_models",
    "02_XGBoost",
    "hyperparameter_optimization"
)

os.makedirs(OPT_DIR, exist_ok=True)

FEATURE_FILE = os.path.join(
    PREPROCESS_DIR,
    "final_model_feature_columns.csv"
)

TRAIN_FILE = os.path.join(
    PREPROCESS_DIR,
    "final_model_ready_train.csv"
)

VALIDATION_FILE = os.path.join(
    PREPROCESS_DIR,
    "final_model_ready_validation.csv"
)

TEST_FILE = os.path.join(
    PREPROCESS_DIR,
    "final_model_ready_test.csv"
)

print("\n" + "=" * 70)
print("PROJECT PATHS")
print("=" * 70)

print("Project:")
print(PROJECT_DIR)

print("\nOptimization directory:")
print(OPT_DIR)

print("\nFeature list:")
print(FEATURE_FILE)

print("\nTraining data:")
print(TRAIN_FILE)

print("\nValidation data:")
print(VALIDATION_FILE)

print("\nTest data:")
print(TEST_FILE)

# ======================================================================
# TEST DATA SAFETY
# ======================================================================

print("\n" + "=" * 70)
print("TEST DATA SAFETY")
print("=" * 70)

print("Test dataset exists on Drive:", os.path.exists(TEST_FILE))
print()
print("IMPORTANT:")
print("Test data will NOT be loaded.")
print("Test data will NOT be used.")
print("Test predictions will NOT be generated.")

# ======================================================================
# REQUIRED FILE CHECK
# ======================================================================

print("\n" + "=" * 70)
print("REQUIRED FILE CHECK")
print("=" * 70)

required_files = {
    "Feature list": FEATURE_FILE,
    "Training data": TRAIN_FILE,
    "Validation data": VALIDATION_FILE
}

for name, path in required_files.items():
    exists = os.path.exists(path)
    print(f"{name}: {exists}")
    if not exists:
        raise FileNotFoundError(path)

print("\n✓ All required training/validation files found.")

# ======================================================================
# LOAD FEATURE LIST
# ======================================================================

print("\n" + "=" * 70)
print("LOADING FINAL FEATURE LIST")
print("=" * 70)

feature_df = pd.read_csv(FEATURE_FILE)

features = feature_df["feature"].tolist()

print("Number of features:", len(features))

if len(features) != 69:
    raise ValueError(
        f"Expected 69 features, found {len(features)}"
    )

print("✓ Exactly 69 predictors confirmed.")

# ======================================================================
# LOAD TRAINING DATA
# ======================================================================

print("\n" + "=" * 70)
print("LOADING TRAINING DATA")
print("=" * 70)

train_df = pd.read_csv(TRAIN_FILE)

print("Training shape:", train_df.shape)

# ======================================================================
# LOAD VALIDATION DATA
# ======================================================================

print("\n" + "=" * 70)
print("LOADING VALIDATION DATA")
print("=" * 70)

validation_df = pd.read_csv(VALIDATION_FILE)

print("Validation shape:", validation_df.shape)

# ======================================================================
# TARGET CONFIGURATION
# ======================================================================

TARGET_LOG = "chlorophyll_a_log1p"
TARGET_ORIGINAL = "chlorophyll_a"

print("\n" + "=" * 70)
print("TARGET CONFIGURATION")
print("=" * 70)

print("Model target:", TARGET_LOG)
print("Original target:", TARGET_ORIGINAL)

# ======================================================================
# FEATURE CHECK
# ======================================================================

missing_train = [
    f for f in features
    if f not in train_df.columns
]

missing_validation = [
    f for f in features
    if f not in validation_df.columns
]

if missing_train:
    raise ValueError(
        f"Training missing features: {missing_train}"
    )

if missing_validation:
    raise ValueError(
        f"Validation missing features: {missing_validation}"
    )

print("\n✓ All 69 features exist in both datasets.")

# ======================================================================
# PREPARE INPUTS
# ======================================================================

X_train = train_df[features]
y_train = train_df[TARGET_LOG]

X_val = validation_df[features]
y_val_log = validation_df[TARGET_LOG]
y_val_original = validation_df[TARGET_ORIGINAL]

print("\nTraining X:", X_train.shape)
print("Training y:", y_train.shape)

print("\nValidation X:", X_val.shape)
print("Validation y:", y_val_log.shape)

# ======================================================================
# DATA SAFETY CHECK
# ======================================================================

print("\n" + "=" * 70)
print("DATA SAFETY CHECK")
print("=" * 70)

train_nan = X_train.isna().sum().sum()
val_nan = X_val.isna().sum().sum()

train_inf = np.isinf(X_train.to_numpy()).sum()
val_inf = np.isinf(X_val.to_numpy()).sum()

train_target_nan = y_train.isna().sum()
val_target_nan = y_val_log.isna().sum()

print("Training predictor NaN:", train_nan)
print("Training predictor infinite:", train_inf)

print("Validation predictor NaN:", val_nan)
print("Validation predictor infinite:", val_inf)

print("Training target NaN:", train_target_nan)
print("Validation target NaN:", val_target_nan)

if any([
    train_nan,
    val_nan,
    train_inf,
    val_inf,
    train_target_nan,
    val_target_nan
]):
    raise ValueError(
        "Data safety check failed."
    )

print("\n✓ Training and validation data are clean.")

# ======================================================================
# BASELINE CONFIGURATION
# ======================================================================

BASELINE_PARAMS = {
    "objective": "reg:squarederror",
    "n_estimators": 2000,
    "max_depth": 8,
    "learning_rate": 0.03,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 5,
    "reg_alpha": 0.0,
    "reg_lambda": 1.0,
    "random_state": 42,
    "tree_method": "hist",
    "device": "cuda",
    "early_stopping_rounds": 100
}

# ======================================================================
# EXPERIMENT CONFIGURATION
# ======================================================================

EXPERIMENTS = {
    "Baseline_depth_8": 8,
    "A1_depth_6": 6,
    "A2_depth_10": 10
}

print("\n" + "=" * 70)
print("TREE COMPLEXITY EXPERIMENT")
print("=" * 70)

print()
print("Baseline max_depth : 8")
print("A1 max_depth       : 6")
print("A2 max_depth       : 10")

print()
print("ONLY max_depth will change.")
print("All other parameters remain identical.")

# ======================================================================
# METRIC FUNCTION
# ======================================================================

def calculate_metrics(y_true_log, y_pred_log, y_true_original):

    y_pred_original = np.expm1(y_pred_log)

    log_mae = mean_absolute_error(
        y_true_log,
        y_pred_log
    )

    log_rmse = np.sqrt(
        mean_squared_error(
            y_true_log,
            y_pred_log
        )
    )

    log_r2 = r2_score(
        y_true_log,
        y_pred_log
    )

    original_mae = mean_absolute_error(
        y_true_original,
        y_pred_original
    )

    original_rmse = np.sqrt(
        mean_squared_error(
            y_true_original,
            y_pred_original
        )
    )

    original_r2 = r2_score(
        y_true_original,
        y_pred_original
    )

    log_bias = np.mean(
        y_pred_log - y_true_log
    )

    original_bias = np.mean(
        y_pred_original - y_true_original
    )

    return {
        "MAE": original_mae,
        "RMSE": original_rmse,
        "R2": original_r2,
        "Bias": original_bias,
        "Log_MAE": log_mae,
        "Log_RMSE": log_rmse,
        "Log_R2": log_r2,
        "Log_Bias": log_bias
    }

# ======================================================================
# RUN EXPERIMENTS
# ======================================================================

results = []

for experiment_name, depth in EXPERIMENTS.items():

    print("\n")
    print("=" * 70)
    print(f"RUNNING: {experiment_name}")
    print("=" * 70)

    params = BASELINE_PARAMS.copy()

    # ONLY parameter being changed
    params["max_depth"] = depth

    print("\nParameters:")
    print("max_depth         :", depth)
    print("learning_rate     :", params["learning_rate"])
    print("subsample         :", params["subsample"])
    print("colsample_bytree  :", params["colsample_bytree"])
    print("min_child_weight  :", params["min_child_weight"])
    print("reg_alpha         :", params["reg_alpha"])
    print("reg_lambda        :", params["reg_lambda"])
    print("n_estimators      :", params["n_estimators"])
    print("early stopping    :", params["early_stopping_rounds"])
    print("device             :", params["device"])

    # --------------------------------------------------------------
    # TEMPORARY MODEL
    # --------------------------------------------------------------

    model = xgb.XGBRegressor(
        objective=params["objective"],
        n_estimators=params["n_estimators"],
        max_depth=params["max_depth"],
        learning_rate=params["learning_rate"],
        subsample=params["subsample"],
        colsample_bytree=params["colsample_bytree"],
        min_child_weight=params["min_child_weight"],
        reg_alpha=params["reg_alpha"],
        reg_lambda=params["reg_lambda"],
        random_state=params["random_state"],
        tree_method=params["tree_method"],
        device=params["device"],
        early_stopping_rounds=params["early_stopping_rounds"]
    )

    print("\n✓ Temporary model created.")
    print("✓ Existing XGBoost model remains untouched.")

    # --------------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------------

    start_time = time.time()

    model.fit(
        X_train,
        y_train,
        eval_set=[
            (X_train, y_train),
            (X_val, y_val_log)
        ],
        verbose=False
    )

    training_time = time.time() - start_time

    # --------------------------------------------------------------
    # BEST ITERATION
    # --------------------------------------------------------------

    best_iteration = model.best_iteration
    best_trees = best_iteration + 1
    best_score = model.best_score

    print("\nTraining completed.")
    print("Training time:", round(training_time, 2), "seconds")
    print("Best iteration:", best_iteration)
    print("Best number of trees:", best_trees)
    print("Best validation RMSE:", best_score)

    # --------------------------------------------------------------
    # VALIDATION PREDICTIONS
    # --------------------------------------------------------------

    pred_log = model.predict(X_val)

    metrics = calculate_metrics(
        y_val_log,
        pred_log,
        y_val_original
    )

    print("\nValidation results:")

    print("\nORIGINAL CHL SCALE")
    print("MAE :", f"{metrics['MAE']:.6f}")
    print("RMSE:", f"{metrics['RMSE']:.6f}")
    print("R²  :", f"{metrics['R2']:.6f}")
    print("Bias:", f"{metrics['Bias']:.6f}")

    print("\nLOG1P SCALE")
    print("MAE :", f"{metrics['Log_MAE']:.6f}")
    print("RMSE:", f"{metrics['Log_RMSE']:.6f}")
    print("R²  :", f"{metrics['Log_R2']:.6f}")
    print("Bias:", f"{metrics['Log_Bias']:.6f}")

    # --------------------------------------------------------------
    # STORE RESULTS
    # --------------------------------------------------------------

    row = {
        "experiment": experiment_name,
        "max_depth": depth,
        "learning_rate": params["learning_rate"],
        "subsample": params["subsample"],
        "colsample_bytree": params["colsample_bytree"],
        "min_child_weight": params["min_child_weight"],
        "reg_alpha": params["reg_alpha"],
        "reg_lambda": params["reg_lambda"],
        "n_estimators": params["n_estimators"],
        "early_stopping_rounds": params["early_stopping_rounds"],
        "best_iteration": best_iteration,
        "best_trees": best_trees,
        "best_validation_log_rmse": best_score,
        "training_time_seconds": training_time,
        **metrics
    }

    results.append(row)

    # --------------------------------------------------------------
    # RELEASE TEMPORARY MODEL
    # --------------------------------------------------------------

    del model

# ======================================================================
# RESULTS TABLE
# ======================================================================

results_df = pd.DataFrame(results)

print("\n" + "=" * 70)
print("TREE COMPLEXITY RESULTS")
print("=" * 70)

display(
    results_df[
        [
            "experiment",
            "max_depth",
            "MAE",
            "RMSE",
            "R2",
            "Log_MAE",
            "Log_RMSE",
            "Log_R2",
            "Bias",
            "best_iteration",
            "best_trees",
            "training_time_seconds"
        ]
    ].round(6)
)

# ======================================================================
# SAVE RESULTS
# ======================================================================

results_file = os.path.join(
    OPT_DIR,
    "xgboost_tree_complexity_results.csv"
)

results_df.to_csv(
    results_file,
    index=False
)

print("\nSaved:")
print(results_file)

# ======================================================================
# IDENTIFY BEST MODELS
# ======================================================================

best_original_mae = results_df.loc[
    results_df["MAE"].idxmin()
]

best_original_rmse = results_df.loc[
    results_df["RMSE"].idxmin()
]

best_original_r2 = results_df.loc[
    results_df["R2"].idxmax()
]

best_log_mae = results_df.loc[
    results_df["Log_MAE"].idxmin()
]

best_log_rmse = results_df.loc[
    results_df["Log_RMSE"].idxmin()
]

best_log_r2 = results_df.loc[
    results_df["Log_R2"].idxmax()
]

print("\n" + "=" * 70)
print("BEST MODEL BY METRIC")
print("=" * 70)

print(
    "Original CHL MAE :",
    best_original_mae["experiment"]
)

print(
    "Original CHL RMSE:",
    best_original_rmse["experiment"]
)

print(
    "Original CHL R²  :",
    best_original_r2["experiment"]
)

print(
    "Log1p MAE        :",
    best_log_mae["experiment"]
)

print(
    "Log1p RMSE       :",
    best_log_rmse["experiment"]
)

print(
    "Log1p R²         :",
    best_log_r2["experiment"]
)

# ======================================================================
# COMPARE AGAINST ESTABLISHED BASELINE
# ======================================================================

ESTABLISHED_BASELINE = {
    "MAE": 0.1894004434498657,
    "RMSE": 0.8060904960285136,
    "R2": 0.5371393487829845,
    "Log_MAE": 0.06996042122254419,
    "Log_RMSE": 0.15257568941934896,
    "Log_R2": 0.8382274310440981
}

print("\n" + "=" * 70)
print("COMPARISON WITH ESTABLISHED XGBOOST BASELINE")
print("=" * 70)

baseline_comparison = []

for _, row in results_df.iterrows():

    comparison = {
        "experiment": row["experiment"],
        "max_depth": row["max_depth"],

        "MAE_difference":
            row["MAE"] - ESTABLISHED_BASELINE["MAE"],

        "RMSE_difference":
            row["RMSE"] - ESTABLISHED_BASELINE["RMSE"],

        "R2_difference":
            row["R2"] - ESTABLISHED_BASELINE["R2"],

        "Log_MAE_difference":
            row["Log_MAE"] - ESTABLISHED_BASELINE["Log_MAE"],

        "Log_RMSE_difference":
            row["Log_RMSE"] - ESTABLISHED_BASELINE["Log_RMSE"],

        "Log_R2_difference":
            row["Log_R2"] - ESTABLISHED_BASELINE["Log_R2"]
    }

    baseline_comparison.append(comparison)

baseline_comparison_df = pd.DataFrame(
    baseline_comparison
)

display(
    baseline_comparison_df.round(8)
)

comparison_file = os.path.join(
    OPT_DIR,
    "xgboost_tree_complexity_vs_established_baseline.csv"
)

baseline_comparison_df.to_csv(
    comparison_file,
    index=False
)

print("\nSaved:")
print(comparison_file)

# ======================================================================
# SAFETY CHECK
# ======================================================================

print("\n" + "=" * 70)
print("FINAL 31.9P-16T-03 SAFETY CHECK")
print("=" * 70)

print("Training data used       : TRUE")
print("Validation data used     : TRUE")
print("Test data loaded         : FALSE")
print("Test data used           : FALSE")
print("Test predictions         : FALSE")
print("Existing model modified  : FALSE")
print("Existing model overwritten: FALSE")
print("Only max_depth changed   : TRUE")

print("\n✓ Tree complexity experiment completed.")
print("✓ Baseline model remains untouched.")
print("✓ Test data was not loaded.")
print("✓ Test data was not used.")
print("✓ Only max_depth was varied.")
print("✓ Results saved.")

print("\n" + "=" * 70)
print("STEP 31.9P-16T-03 COMPLETE")
print("=" * 70)

print("\nNEXT:")
print("Send me the complete output.")
print("We will determine whether max_depth = 6, 8, or 10 is best.")
print("DO NOT start the next tuning stage yet.")

STEP 31.9P-16T-03
XGBOOST TREE COMPLEXITY OPTIMIZATION

PROJECT PATHS
Project:
/content/drive/MyDrive/Indhi_Marine_Research_Data

Optimization directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/02_XGBoost/hyperparameter_optimization

Feature list:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing/final_model_feature_columns.csv

Training data:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing/final_model_ready_train.csv

Validation data:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing/final_model_ready_validation.csv

Test data:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing/final_model_ready_test.csv

TEST DATA SAFETY
Test dataset exists on Drive: True

IMPORTANT:
Test data will NOT be loaded.
Test data will NOT be used.
Test predictions will NOT be generated.

REQUIRED FILE CHECK
Feature list: True
Training data: True
Validation data: True

✓ All required training/vali

,experiment,max_depth,MAE,RMSE,R2,Log_MAE,Log_RMSE,Log_R2,Bias,best_iteration,best_trees,training_time_seconds
0,Baseline_depth_8,8,0.189400,0.806091,0.537139,0.069960,0.152576,0.838227,-0.076182,128,129,11.484395
1,A1_depth_6,6,0.183790,0.790759,0.554579,0.067813,0.148974,0.845774,-0.068226,118,119,9.984729
2,A2_depth_10,10,0.193719,0.809281,0.533468,0.071794,0.154811,0.833453,-0.080562,139,140,14.601561



Saved:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/02_XGBoost/hyperparameter_optimization/xgboost_tree_complexity_results.csv

BEST MODEL BY METRIC
Original CHL MAE : A1_depth_6
Original CHL RMSE: A1_depth_6
Original CHL R²  : A1_depth_6
Log1p MAE        : A1_depth_6
Log1p RMSE       : A1_depth_6
Log1p R²         : A1_depth_6

COMPARISON WITH ESTABLISHED XGBOOST BASELINE


,experiment,max_depth,MAE_difference,RMSE_difference,R2_difference,Log_MAE_difference,Log_RMSE_difference,Log_R2_difference
0,Baseline_depth_8,8,-0.000000,1.000000e-08,-1.000000e-08,-0.000000,-0.000000,0.000000
1,A1_depth_6,6,-0.005611,-1.533134e-02,1.743921e-02,-0.002147,-0.003601,0.007547
2,A2_depth_10,10,0.004319,3.190680e-03,-3.671460e-03,0.001834,0.002235,-0.004775



Saved:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/02_XGBoost/hyperparameter_optimization/xgboost_tree_complexity_vs_established_baseline.csv

FINAL 31.9P-16T-03 SAFETY CHECK
Training data used       : TRUE
Validation data used     : TRUE
Test data loaded         : FALSE
Test data used           : FALSE
Test predictions         : FALSE
Existing model modified  : FALSE
Existing model overwritten: FALSE
Only max_depth changed   : TRUE

✓ Tree complexity experiment completed.
✓ Baseline model remains untouched.
✓ Test data was not loaded.
✓ Test data was not used.
✓ Only max_depth was varied.
✓ Results saved.

STEP 31.9P-16T-03 COMPLETE

NEXT:
Send me the complete output.
We will determine whether max_depth = 6, 8, or 10 is best.
DO NOT start the next tuning stage yet.


In [1]:
# ======================================================================
# STEP 31.9P-16T-04
# XGBOOST MIN_CHILD_WEIGHT OPTIMIZATION
# CONTROLLED SPLIT REGULARIZATION EXPERIMENT
# ======================================================================

import os
import json
import time
import gc

import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("=" * 70)
print("STEP 31.9P-16T-04")
print("XGBOOST MIN_CHILD_WEIGHT OPTIMIZATION")
print("=" * 70)


# ======================================================================
# PROJECT PATHS
# ======================================================================

PROJECT_DIR = "/content/drive/MyDrive/Indhi_Marine_Research_Data"

PREPROCESS_DIR = os.path.join(
    PROJECT_DIR,
    "05_model_preprocessing"
)

OPT_DIR = os.path.join(
    PROJECT_DIR,
    "06_models",
    "02_XGBoost",
    "hyperparameter_optimization"
)

os.makedirs(OPT_DIR, exist_ok=True)

FEATURE_FILE = os.path.join(
    PREPROCESS_DIR,
    "final_model_feature_columns.csv"
)

TRAIN_FILE = os.path.join(
    PREPROCESS_DIR,
    "final_model_ready_train.csv"
)

VALIDATION_FILE = os.path.join(
    PREPROCESS_DIR,
    "final_model_ready_validation.csv"
)

TEST_FILE = os.path.join(
    PREPROCESS_DIR,
    "final_model_ready_test.csv"
)


print("\n" + "=" * 70)
print("PROJECT PATHS")
print("=" * 70)

print("Project:")
print(PROJECT_DIR)

print("\nOptimization directory:")
print(OPT_DIR)

print("\nFeature list:")
print(FEATURE_FILE)

print("\nTraining data:")
print(TRAIN_FILE)

print("\nValidation data:")
print(VALIDATION_FILE)

print("\nTest data:")
print(TEST_FILE)


# ======================================================================
# TEST DATA SAFETY
# ======================================================================

print("\n" + "=" * 70)
print("TEST DATA SAFETY")
print("=" * 70)

print("Test dataset exists on Drive:", os.path.exists(TEST_FILE))

print()
print("IMPORTANT:")
print("Test data will NOT be loaded.")
print("Test data will NOT be used.")
print("Test predictions will NOT be generated.")


# ======================================================================
# REQUIRED FILE CHECK
# ======================================================================

print("\n" + "=" * 70)
print("REQUIRED FILE CHECK")
print("=" * 70)

required_files = {
    "Feature list": FEATURE_FILE,
    "Training data": TRAIN_FILE,
    "Validation data": VALIDATION_FILE
}

for name, path in required_files.items():

    exists = os.path.exists(path)

    print(f"{name}: {exists}")

    if not exists:
        raise FileNotFoundError(path)

print("\n✓ All required training/validation files found.")


# ======================================================================
# LOAD FINAL FEATURE LIST
# ======================================================================

print("\n" + "=" * 70)
print("LOADING FINAL FEATURE LIST")
print("=" * 70)

feature_df = pd.read_csv(FEATURE_FILE)

features = feature_df["feature"].tolist()

print("Number of features:", len(features))

if len(features) != 69:
    raise ValueError(
        f"Expected 69 features, found {len(features)}"
    )

print("✓ Exactly 69 predictors confirmed.")


# ======================================================================
# LOAD TRAINING DATA
# ======================================================================

print("\n" + "=" * 70)
print("LOADING TRAINING DATA")
print("=" * 70)

train_df = pd.read_csv(TRAIN_FILE)

print("Training shape:", train_df.shape)


# ======================================================================
# LOAD VALIDATION DATA
# ======================================================================

print("\n" + "=" * 70)
print("LOADING VALIDATION DATA")
print("=" * 70)

validation_df = pd.read_csv(VALIDATION_FILE)

print("Validation shape:", validation_df.shape)


# ======================================================================
# TARGET CONFIGURATION
# ======================================================================

TARGET_LOG = "chlorophyll_a_log1p"
TARGET_ORIGINAL = "chlorophyll_a"

print("\n" + "=" * 70)
print("TARGET CONFIGURATION")
print("=" * 70)

print("Model target:", TARGET_LOG)
print("Original target:", TARGET_ORIGINAL)


# ======================================================================
# FEATURE EXISTENCE CHECK
# ======================================================================

missing_train = [
    feature for feature in features
    if feature not in train_df.columns
]

missing_validation = [
    feature for feature in features
    if feature not in validation_df.columns
]

if missing_train:
    raise ValueError(
        f"Training missing features: {missing_train}"
    )

if missing_validation:
    raise ValueError(
        f"Validation missing features: {missing_validation}"
    )

print("\n✓ All 69 features exist in both datasets.")


# ======================================================================
# PREPARE MODEL INPUTS
# ======================================================================

X_train = train_df[features]
y_train = train_df[TARGET_LOG]

X_val = validation_df[features]
y_val_log = validation_df[TARGET_LOG]
y_val_original = validation_df[TARGET_ORIGINAL]

print("\nTraining X:", X_train.shape)
print("Training y:", y_train.shape)

print("\nValidation X:", X_val.shape)
print("Validation y:", y_val_log.shape)


# ======================================================================
# DATA SAFETY CHECK
# ======================================================================

print("\n" + "=" * 70)
print("DATA SAFETY CHECK")
print("=" * 70)

train_nan = X_train.isna().sum().sum()
val_nan = X_val.isna().sum().sum()

train_inf = np.isinf(
    X_train.to_numpy()
).sum()

val_inf = np.isinf(
    X_val.to_numpy()
).sum()

train_target_nan = y_train.isna().sum()
val_target_nan = y_val_log.isna().sum()

print("Training predictor NaN:", train_nan)
print("Training predictor infinite:", train_inf)

print("Validation predictor NaN:", val_nan)
print("Validation predictor infinite:", val_inf)

print("Training target NaN:", train_target_nan)
print("Validation target NaN:", val_target_nan)

if any([
    train_nan,
    val_nan,
    train_inf,
    val_inf,
    train_target_nan,
    val_target_nan
]):
    raise ValueError(
        "Data safety check failed."
    )

print("\n✓ Training and validation data are clean.")


# ======================================================================
# EXPERIMENTAL BASELINE
# ======================================================================

BASELINE_MAX_DEPTH = 6

BASELINE_PARAMS = {
    "objective": "reg:squarederror",
    "n_estimators": 2000,
    "max_depth": BASELINE_MAX_DEPTH,
    "learning_rate": 0.03,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 5,
    "reg_alpha": 0.0,
    "reg_lambda": 1.0,
    "random_state": 42,
    "tree_method": "hist",
    "device": "cuda",
    "early_stopping_rounds": 100
}


# ======================================================================
# EXPERIMENT CONFIGURATION
# ======================================================================

EXPERIMENTS = {
    "Baseline_depth6_child5": 5,
    "B1_depth6_child1": 1,
    "B2_depth6_child10": 10
}


print("\n" + "=" * 70)
print("MIN_CHILD_WEIGHT EXPERIMENT")
print("=" * 70)

print()
print("Experimental max_depth: 6")

print()
print("Baseline:")
print("min_child_weight = 5")

print()
print("B1:")
print("min_child_weight = 1")

print()
print("B2:")
print("min_child_weight = 10")

print()
print("ONLY min_child_weight will change.")
print("max_depth remains fixed at 6.")
print("All other parameters remain identical.")


# ======================================================================
# METRIC FUNCTION
# ======================================================================

def calculate_metrics(
    y_true_log,
    y_pred_log,
    y_true_original
):

    y_pred_original = np.expm1(
        y_pred_log
    )

    # --------------------------------------------------------------
    # LOG SCALE
    # --------------------------------------------------------------

    log_mae = mean_absolute_error(
        y_true_log,
        y_pred_log
    )

    log_rmse = np.sqrt(
        mean_squared_error(
            y_true_log,
            y_pred_log
        )
    )

    log_r2 = r2_score(
        y_true_log,
        y_pred_log
    )

    log_bias = np.mean(
        y_pred_log - y_true_log
    )

    # --------------------------------------------------------------
    # ORIGINAL CHL SCALE
    # --------------------------------------------------------------

    original_mae = mean_absolute_error(
        y_true_original,
        y_pred_original
    )

    original_rmse = np.sqrt(
        mean_squared_error(
            y_true_original,
            y_pred_original
        )
    )

    original_r2 = r2_score(
        y_true_original,
        y_pred_original
    )

    original_bias = np.mean(
        y_pred_original - y_true_original
    )

    return {

        "MAE": original_mae,

        "RMSE": original_rmse,

        "R2": original_r2,

        "Bias": original_bias,

        "Log_MAE": log_mae,

        "Log_RMSE": log_rmse,

        "Log_R2": log_r2,

        "Log_Bias": log_bias
    }


# ======================================================================
# RUN EXPERIMENTS
# ======================================================================

results = []


for experiment_name, child_weight in EXPERIMENTS.items():

    print("\n")
    print("=" * 70)
    print(f"RUNNING: {experiment_name}")
    print("=" * 70)

    params = BASELINE_PARAMS.copy()

    # --------------------------------------------------------------
    # ONLY PARAMETER CHANGED
    # --------------------------------------------------------------

    params["min_child_weight"] = child_weight

    print("\nParameters:")

    print(
        "max_depth         :",
        params["max_depth"]
    )

    print(
        "min_child_weight  :",
        params["min_child_weight"]
    )

    print(
        "learning_rate     :",
        params["learning_rate"]
    )

    print(
        "subsample         :",
        params["subsample"]
    )

    print(
        "colsample_bytree  :",
        params["colsample_bytree"]
    )

    print(
        "reg_alpha         :",
        params["reg_alpha"]
    )

    print(
        "reg_lambda        :",
        params["reg_lambda"]
    )

    print(
        "n_estimators      :",
        params["n_estimators"]
    )

    print(
        "early stopping    :",
        params["early_stopping_rounds"]
    )

    print(
        "device             :",
        params["device"]
    )


    # --------------------------------------------------------------
    # CREATE TEMPORARY MODEL
    # --------------------------------------------------------------

    model = xgb.XGBRegressor(

        objective=params["objective"],

        n_estimators=params["n_estimators"],

        max_depth=params["max_depth"],

        learning_rate=params["learning_rate"],

        subsample=params["subsample"],

        colsample_bytree=params["colsample_bytree"],

        min_child_weight=params["min_child_weight"],

        reg_alpha=params["reg_alpha"],

        reg_lambda=params["reg_lambda"],

        random_state=params["random_state"],

        tree_method=params["tree_method"],

        device=params["device"],

        early_stopping_rounds=params[
            "early_stopping_rounds"
        ]
    )


    print("\n✓ Temporary model created.")

    print(
        "✓ Existing XGBoost model remains untouched."
    )


    # --------------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------------

    start_time = time.time()

    model.fit(

        X_train,

        y_train,

        eval_set=[
            (X_train, y_train),
            (X_val, y_val_log)
        ],

        verbose=False
    )

    training_time = time.time() - start_time


    # --------------------------------------------------------------
    # BEST ITERATION
    # --------------------------------------------------------------

    best_iteration = model.best_iteration

    best_trees = best_iteration + 1

    best_score = model.best_score


    print("\nTraining completed.")

    print(
        "Training time:",
        round(training_time, 2),
        "seconds"
    )

    print(
        "Best iteration:",
        best_iteration
    )

    print(
        "Best number of trees:",
        best_trees
    )

    print(
        "Best validation RMSE:",
        best_score
    )


    # --------------------------------------------------------------
    # VALIDATION PREDICTIONS
    # --------------------------------------------------------------

    pred_log = model.predict(X_val)

    metrics = calculate_metrics(

        y_val_log,

        pred_log,

        y_val_original
    )


    # --------------------------------------------------------------
    # DISPLAY METRICS
    # --------------------------------------------------------------

    print("\nValidation results:")

    print("\nORIGINAL CHL SCALE")

    print(
        "MAE :",
        f"{metrics['MAE']:.6f}"
    )

    print(
        "RMSE:",
        f"{metrics['RMSE']:.6f}"
    )

    print(
        "R²  :",
        f"{metrics['R2']:.6f}"
    )

    print(
        "Bias:",
        f"{metrics['Bias']:.6f}"
    )


    print("\nLOG1P SCALE")

    print(
        "MAE :",
        f"{metrics['Log_MAE']:.6f}"
    )

    print(
        "RMSE:",
        f"{metrics['Log_RMSE']:.6f}"
    )

    print(
        "R²  :",
        f"{metrics['Log_R2']:.6f}"
    )

    print(
        "Bias:",
        f"{metrics['Log_Bias']:.6f}"
    )


    # --------------------------------------------------------------
    # STORE RESULTS
    # --------------------------------------------------------------

    row = {

        "experiment": experiment_name,

        "max_depth": params[
            "max_depth"
        ],

        "min_child_weight": params[
            "min_child_weight"
        ],

        "learning_rate": params[
            "learning_rate"
        ],

        "subsample": params[
            "subsample"
        ],

        "colsample_bytree": params[
            "colsample_bytree"
        ],

        "reg_alpha": params[
            "reg_alpha"
        ],

        "reg_lambda": params[
            "reg_lambda"
        ],

        "n_estimators": params[
            "n_estimators"
        ],

        "early_stopping_rounds": params[
            "early_stopping_rounds"
        ],

        "best_iteration": best_iteration,

        "best_trees": best_trees,

        "best_validation_log_rmse": best_score,

        "training_time_seconds": training_time,

        **metrics
    }

    results.append(row)


    # --------------------------------------------------------------
    # CLEAN TEMPORARY MODEL
    # --------------------------------------------------------------

    del model
    gc.collect()


# ======================================================================
# RESULTS TABLE
# ======================================================================

results_df = pd.DataFrame(results)


print("\n" + "=" * 70)
print("MIN_CHILD_WEIGHT RESULTS")
print("=" * 70)


display(
    results_df[
        [
            "experiment",
            "max_depth",
            "min_child_weight",

            "MAE",
            "RMSE",
            "R2",

            "Log_MAE",
            "Log_RMSE",
            "Log_R2",

            "Bias",

            "best_iteration",
            "best_trees",

            "training_time_seconds"
        ]
    ].round(6)
)


# ======================================================================
# SAVE RESULTS
# ======================================================================

results_file = os.path.join(

    OPT_DIR,

    "xgboost_min_child_weight_results.csv"
)


results_df.to_csv(

    results_file,

    index=False
)


print("\nSaved:")
print(results_file)


# ======================================================================
# BEST MODEL BY METRIC
# ======================================================================

best_original_mae = results_df.loc[
    results_df["MAE"].idxmin()
]

best_original_rmse = results_df.loc[
    results_df["RMSE"].idxmin()
]

best_original_r2 = results_df.loc[
    results_df["R2"].idxmax()
]

best_log_mae = results_df.loc[
    results_df["Log_MAE"].idxmin()
]

best_log_rmse = results_df.loc[
    results_df["Log_RMSE"].idxmin()
]

best_log_r2 = results_df.loc[
    results_df["Log_R2"].idxmax()
]


print("\n" + "=" * 70)
print("BEST MODEL BY METRIC")
print("=" * 70)


print(
    "Original CHL MAE :",
    best_original_mae["experiment"]
)

print(
    "Original CHL RMSE:",
    best_original_rmse["experiment"]
)

print(
    "Original CHL R²  :",
    best_original_r2["experiment"]
)

print(
    "Log1p MAE        :",
    best_log_mae["experiment"]
)

print(
    "Log1p RMSE       :",
    best_log_rmse["experiment"]
)

print(
    "Log1p R²         :",
    best_log_r2["experiment"]
)


# ======================================================================
# COMPARE WITH DEPTH-6 BASELINE
# ======================================================================

DEPTH6_REFERENCE = {

    "MAE": 0.183790,

    "RMSE": 0.790759,

    "R2": 0.554579,

    "Log_MAE": 0.067813,

    "Log_RMSE": 0.148974,

    "Log_R2": 0.845774
}


print("\n" + "=" * 70)
print("COMPARISON WITH DEPTH-6 REFERENCE")
print("=" * 70)


reference_comparison = []


for _, row in results_df.iterrows():

    comparison = {

        "experiment":
            row["experiment"],

        "min_child_weight":
            row["min_child_weight"],

        "MAE_difference":
            row["MAE"] -
            DEPTH6_REFERENCE["MAE"],

        "RMSE_difference":
            row["RMSE"] -
            DEPTH6_REFERENCE["RMSE"],

        "R2_difference":
            row["R2"] -
            DEPTH6_REFERENCE["R2"],

        "Log_MAE_difference":
            row["Log_MAE"] -
            DEPTH6_REFERENCE["Log_MAE"],

        "Log_RMSE_difference":
            row["Log_RMSE"] -
            DEPTH6_REFERENCE["Log_RMSE"],

        "Log_R2_difference":
            row["Log_R2"] -
            DEPTH6_REFERENCE["Log_R2"]
    }

    reference_comparison.append(
        comparison
    )


reference_comparison_df = pd.DataFrame(
    reference_comparison
)


display(
    reference_comparison_df.round(8)
)


comparison_file = os.path.join(

    OPT_DIR,

    "xgboost_min_child_weight_vs_depth6_reference.csv"
)


reference_comparison_df.to_csv(

    comparison_file,

    index=False
)


print("\nSaved:")
print(comparison_file)


# ======================================================================
# EXPERIMENT SAFETY CHECK
# ======================================================================

print("\n" + "=" * 70)
print("FINAL 31.9P-16T-04 SAFETY CHECK")
print("=" * 70)

print(
    "Training data used       : TRUE"
)

print(
    "Validation data used     : TRUE"
)

print(
    "Test data loaded         : FALSE"
)

print(
    "Test data used           : FALSE"
)

print(
    "Test predictions         : FALSE"
)

print(
    "Existing model modified  : FALSE"
)

print(
    "Existing model overwritten: FALSE"
)

print(
    "max_depth fixed at       : 6"
)

print(
    "Only min_child_weight changed: TRUE"
)


print("\n✓ min_child_weight experiment completed.")

print(
    "✓ max_depth = 6 retained from previous experiment."
)

print(
    "✓ Existing XGBoost model remains untouched."
)

print(
    "✓ Test data was not loaded."
)

print(
    "✓ Test data was not used."
)

print(
    "✓ Only min_child_weight was varied."
)

print(
    "✓ Results saved."
)


print("\n" + "=" * 70)
print("STEP 31.9P-16T-04 COMPLETE")
print("=" * 70)

print("\nNEXT:")
print(
    "Send me the complete output before running another tuning stage."
)

print(
    "We will determine whether min_child_weight = 1, 5, or 10 is best."
)

print(
    "DO NOT start the next tuning stage yet."
)

STEP 31.9P-16T-04
XGBOOST MIN_CHILD_WEIGHT OPTIMIZATION

PROJECT PATHS
Project:
/content/drive/MyDrive/Indhi_Marine_Research_Data

Optimization directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/02_XGBoost/hyperparameter_optimization

Feature list:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing/final_model_feature_columns.csv

Training data:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing/final_model_ready_train.csv

Validation data:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing/final_model_ready_validation.csv

Test data:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing/final_model_ready_test.csv

TEST DATA SAFETY
Test dataset exists on Drive: True

IMPORTANT:
Test data will NOT be loaded.
Test data will NOT be used.
Test predictions will NOT be generated.

REQUIRED FILE CHECK
Feature list: True
Training data: True
Validation data: True

✓ All required training/val

/usr/local/lib/python3.13/dist-packages/xgboost/core.py:569: UserWarning: [16:27:48] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)



Validation results:

ORIGINAL CHL SCALE
MAE : 0.183790
RMSE: 0.790759
R²  : 0.554579
Bias: -0.068226

LOG1P SCALE
MAE : 0.067813
RMSE: 0.148974
R²  : 0.845774
Bias: -0.008486


RUNNING: B1_depth6_child1

Parameters:
max_depth         : 6
min_child_weight  : 1
learning_rate     : 0.03
subsample         : 0.8
colsample_bytree  : 0.8
reg_alpha         : 0.0
reg_lambda        : 1.0
n_estimators      : 2000
early stopping    : 100
device             : cuda

✓ Temporary model created.
✓ Existing XGBoost model remains untouched.

Training completed.
Training time: 10.15 seconds
Best iteration: 120
Best number of trees: 121
Best validation RMSE: 0.14907544575429882

Validation results:

ORIGINAL CHL SCALE
MAE : 0.183907
RMSE: 0.793657
R²  : 0.551308
Bias: -0.068903

LOG1P SCALE
MAE : 0.067769
RMSE: 0.149075
R²  : 0.845565
Bias: -0.008409


RUNNING: B2_depth6_child10

Parameters:
max_depth         : 6
min_child_weight  : 10
learning_rate     : 0.03
subsample         : 0.8
colsample_bytree  : 0

,experiment,max_depth,min_child_weight,MAE,RMSE,R2,Log_MAE,Log_RMSE,Log_R2,Bias,best_iteration,best_trees,training_time_seconds
0,Baseline_depth6_child5,6,5,0.183790,0.790759,0.554579,0.067813,0.148974,0.845774,-0.068226,118,119,11.038709
1,B1_depth6_child1,6,1,0.183907,0.793657,0.551308,0.067769,0.149075,0.845565,-0.068903,120,121,10.153825
2,B2_depth6_child10,6,10,0.183352,0.789430,0.556075,0.067509,0.148762,0.846214,-0.068570,122,123,9.984091



Saved:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/02_XGBoost/hyperparameter_optimization/xgboost_min_child_weight_results.csv

BEST MODEL BY METRIC
Original CHL MAE : B2_depth6_child10
Original CHL RMSE: B2_depth6_child10
Original CHL R²  : B2_depth6_child10
Log1p MAE        : B2_depth6_child10
Log1p RMSE       : B2_depth6_child10
Log1p R²         : B2_depth6_child10

COMPARISON WITH DEPTH-6 REFERENCE


,experiment,min_child_weight,MAE_difference,RMSE_difference,R2_difference,Log_MAE_difference,Log_RMSE_difference,Log_R2_difference
0,Baseline_depth6_child5,5,-1.600000e-07,1.600000e-07,-4.400000e-07,2.200000e-07,3.200000e-07,2.100000e-07
1,B1_depth6_child1,1,1.174800e-04,2.898000e-03,-3.271040e-03,-4.448000e-05,1.014500e-04,-2.092500e-04
2,B2_depth6_child10,10,-4.384600e-04,-1.329450e-03,1.496190e-03,-3.039100e-04,-2.122200e-04,4.399600e-04



Saved:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/02_XGBoost/hyperparameter_optimization/xgboost_min_child_weight_vs_depth6_reference.csv

FINAL 31.9P-16T-04 SAFETY CHECK
Training data used       : TRUE
Validation data used     : TRUE
Test data loaded         : FALSE
Test data used           : FALSE
Test predictions         : FALSE
Existing model modified  : FALSE
Existing model overwritten: FALSE
max_depth fixed at       : 6
Only min_child_weight changed: TRUE

✓ min_child_weight experiment completed.
✓ max_depth = 6 retained from previous experiment.
✓ Existing XGBoost model remains untouched.
✓ Test data was not loaded.
✓ Test data was not used.
✓ Only min_child_weight was varied.
✓ Results saved.

STEP 31.9P-16T-04 COMPLETE

NEXT:
Send me the complete output before running another tuning stage.
We will determine whether min_child_weight = 1, 5, or 10 is best.
DO NOT start the next tuning stage yet.


In [3]:
# ======================================================================
# STEP 31.9P-16T-05
# XGBOOST LEARNING RATE OPTIMIZATION
# ======================================================================

import os
import json
import time
import numpy as np
import pandas as pd
import xgboost as xgb
from google.colab import drive

drive.mount('/content/drive')

print("=" * 70)
print("STEP 31.9P-16T-05")
print("XGBOOST LEARNING RATE OPTIMIZATION")
print("=" * 70)

# ======================================================================
# PROJECT PATHS
# ======================================================================

PROJECT_DIR = "/content/drive/MyDrive/Indhi_Marine_Research_Data"

OPT_DIR = os.path.join(
    PROJECT_DIR,
    "06_models",
    "02_XGBoost",
    "hyperparameter_optimization"
)

FEATURE_FILE = os.path.join(
    PROJECT_DIR,
    "05_model_preprocessing",
    "final_model_feature_columns.csv"
)

TRAIN_FILE = os.path.join(
    PROJECT_DIR,
    "05_model_preprocessing",
    "final_model_ready_train.csv"
)

VALIDATION_FILE = os.path.join(
    PROJECT_DIR,
    "05_model_preprocessing",
    "final_model_ready_validation.csv"
)

TEST_FILE = os.path.join(
    PROJECT_DIR,
    "05_model_preprocessing",
    "final_model_ready_test.csv"
)

os.makedirs(OPT_DIR, exist_ok=True)

print("\n" + "=" * 70)
print("PROJECT PATHS")
print("=" * 70)

print("Project:")
print(PROJECT_DIR)

print("\nOptimization directory:")
print(OPT_DIR)

print("\nFeature list:")
print(FEATURE_FILE)

print("\nTraining data:")
print(TRAIN_FILE)

print("\nValidation data:")
print(VALIDATION_FILE)

print("\nTest data:")
print(TEST_FILE)

# ======================================================================
# TEST DATA SAFETY
# ======================================================================

print("\n" + "=" * 70)
print("TEST DATA SAFETY")
print("=" * 70)

print(f"Test dataset exists on Drive: {os.path.exists(TEST_FILE)}")

print("""
IMPORTANT:
Test data will NOT be loaded.
Test data will NOT be used.
Test predictions will NOT be generated.
""")

# ======================================================================
# REQUIRED FILE CHECK
# ======================================================================

print("=" * 70)
print("REQUIRED FILE CHECK")
print("=" * 70)

required_files = {
    "Feature list": FEATURE_FILE,
    "Training data": TRAIN_FILE,
    "Validation data": VALIDATION_FILE
}

for name, path in required_files.items():
    print(f"{name}: {os.path.exists(path)}")

    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing required file: {path}")

print("\n✓ All required training/validation files found.")

# ======================================================================
# LOAD FEATURE LIST
# ======================================================================

print("\n" + "=" * 70)
print("LOADING FINAL FEATURE LIST")
print("=" * 70)

feature_df = pd.read_csv(FEATURE_FILE)

if "feature" not in feature_df.columns:
    raise ValueError("Feature list must contain 'feature' column.")

FEATURES = feature_df["feature"].tolist()

print("Number of features:", len(FEATURES))

if len(FEATURES) != 69:
    raise ValueError(
        f"Expected exactly 69 features, found {len(FEATURES)}"
    )

print("✓ Exactly 69 predictors confirmed.")

# ======================================================================
# LOAD TRAINING DATA
# ======================================================================

print("\n" + "=" * 70)
print("LOADING TRAINING DATA")
print("=" * 70)

train_df = pd.read_csv(TRAIN_FILE)

print("Training shape:", train_df.shape)

# ======================================================================
# LOAD VALIDATION DATA
# ======================================================================

print("\n" + "=" * 70)
print("LOADING VALIDATION DATA")
print("=" * 70)

validation_df = pd.read_csv(VALIDATION_FILE)

print("Validation shape:", validation_df.shape)

# ======================================================================
# TARGET CONFIGURATION
# ======================================================================

TARGET_LOG = "chlorophyll_a_log1p"
TARGET_ORIGINAL = "chlorophyll_a"

print("\n" + "=" * 70)
print("TARGET CONFIGURATION")
print("=" * 70)

print("Model target:", TARGET_LOG)
print("Original target:", TARGET_ORIGINAL)

# ======================================================================
# COLUMN CHECK
# ======================================================================

required_columns = FEATURES + [
    TARGET_LOG,
    TARGET_ORIGINAL
]

missing_train = [
    c for c in required_columns
    if c not in train_df.columns
]

missing_validation = [
    c for c in required_columns
    if c not in validation_df.columns
]

print("\nTraining missing columns:", len(missing_train))
print("Validation missing columns:", len(missing_validation))

if missing_train:
    print(missing_train)
    raise ValueError("Missing training columns.")

if missing_validation:
    print(missing_validation)
    raise ValueError("Missing validation columns.")

print("✓ All required columns available.")

# ======================================================================
# PREPARE INPUTS
# ======================================================================

X_train = train_df[FEATURES]
y_train = train_df[TARGET_LOG]

X_validation = validation_df[FEATURES]
y_validation_log = validation_df[TARGET_LOG]
y_validation_original = validation_df[TARGET_ORIGINAL]

print("\nTraining X:", X_train.shape)
print("Training y:", y_train.shape)

print("\nValidation X:", X_validation.shape)
print("Validation y:", y_validation_log.shape)

# ======================================================================
# DATA SAFETY CHECK
# ======================================================================

print("\n" + "=" * 70)
print("DATA SAFETY CHECK")
print("=" * 70)

checks = {
    "Training predictor NaN": X_train.isna().sum().sum(),
    "Training predictor infinite":
        np.isinf(X_train.to_numpy()).sum(),

    "Validation predictor NaN":
        X_validation.isna().sum().sum(),

    "Validation predictor infinite":
        np.isinf(X_validation.to_numpy()).sum(),

    "Training target NaN":
        y_train.isna().sum(),

    "Validation target NaN":
        y_validation_log.isna().sum()
}

for name, value in checks.items():
    print(f"{name:<40}: {value}")

if any(value != 0 for value in checks.values()):
    raise ValueError(
        "Data safety check failed. "
        "Do not continue with tuning."
    )

print("\n✓ Training and validation data are clean.")

# ======================================================================
# GPU CHECK
# ======================================================================

print("\n" + "=" * 70)
print("GPU VERIFICATION")
print("=" * 70)

print("XGBoost version:", xgb.__version__)

build_info = xgb.build_info()

print("CUDA enabled:", build_info.get("USE_CUDA"))

if not build_info.get("USE_CUDA"):
    raise RuntimeError(
        "XGBoost CUDA support is unavailable."
    )

print("✓ CUDA-enabled XGBoost confirmed.")

# ======================================================================
# EXPERIMENT DESIGN
# ======================================================================

print("\n" + "=" * 70)
print("LEARNING RATE EXPERIMENT")
print("=" * 70)

print("""
Current best configuration from previous stage:

max_depth        = 6
min_child_weight = 10
learning_rate    = 0.03
subsample        = 0.8
colsample_bytree = 0.8
reg_alpha        = 0.0
reg_lambda       = 1.0
n_estimators     = 2000
early stopping   = 100

Only learning_rate will change.
All other parameters remain fixed.
""")

learning_rates = [
    0.01,
    0.03,
    0.05
]

print("Learning rates to test:", learning_rates)

# ======================================================================
# MODEL EVALUATION FUNCTION
# ======================================================================

def evaluate_predictions(
    y_log_actual,
    y_log_pred,
    y_original_actual
):

    y_original_pred = np.expm1(y_log_pred)

    # Prevent tiny numerical negative values
    y_original_pred = np.maximum(y_original_pred, 0)

    # -----------------------------
    # LOG SCALE
    # -----------------------------

    log_error = y_log_pred - y_log_actual

    log_mae = np.mean(np.abs(log_error))
    log_rmse = np.sqrt(np.mean(log_error ** 2))

    ss_res_log = np.sum(log_error ** 2)
    ss_tot_log = np.sum(
        (y_log_actual - np.mean(y_log_actual)) ** 2
    )

    log_r2 = (
        1 - ss_res_log / ss_tot_log
        if ss_tot_log != 0
        else np.nan
    )

    log_bias = np.mean(log_error)

    # -----------------------------
    # ORIGINAL CHL SCALE
    # -----------------------------

    original_error = (
        y_original_pred - y_original_actual
    )

    original_mae = np.mean(
        np.abs(original_error)
    )

    original_rmse = np.sqrt(
        np.mean(original_error ** 2)
    )

    ss_res_original = np.sum(
        original_error ** 2
    )

    ss_tot_original = np.sum(
        (y_original_actual -
         np.mean(y_original_actual)) ** 2
    )

    original_r2 = (
        1 - ss_res_original / ss_tot_original
        if ss_tot_original != 0
        else np.nan
    )

    original_bias = np.mean(original_error)

    return {
        "MAE": original_mae,
        "RMSE": original_rmse,
        "R2": original_r2,
        "Bias": original_bias,
        "Log_MAE": log_mae,
        "Log_RMSE": log_rmse,
        "Log_R2": log_r2,
        "Log_Bias": log_bias
    }

# ======================================================================
# RUN EXPERIMENTS
# ======================================================================

results = []

for lr in learning_rates:

    if lr == 0.03:
        experiment_name = "LR2_reference_0.03"
    elif lr == 0.01:
        experiment_name = "LR1_0.01"
    elif lr == 0.05:
        experiment_name = "LR3_0.05"
    else:
        experiment_name = f"LR_{lr}"

    print("\n" + "=" * 70)
    print(f"RUNNING: {experiment_name}")
    print("=" * 70)

    print("\nParameters:")
    print("max_depth         : 6")
    print("min_child_weight  : 10")
    print("learning_rate     :", lr)
    print("subsample         : 0.8")
    print("colsample_bytree  : 0.8")
    print("reg_alpha         : 0.0")
    print("reg_lambda        : 1.0")
    print("n_estimators      : 2000")
    print("early stopping    : 100")
    print("device             : cuda")

    model = xgb.XGBRegressor(
        objective="reg:squarederror",
        n_estimators=2000,
        max_depth=6,
        learning_rate=lr,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=10,
        reg_alpha=0.0,
        reg_lambda=1.0,
        random_state=42,
        tree_method="hist",
        device="cuda",
        early_stopping_rounds=100
    )

    print("\n✓ Temporary model created.")
    print("✓ Existing XGBoost model remains untouched.")

    start_time = time.time()

    model.fit(
        X_train,
        y_train,
        eval_set=[
            (X_train, y_train),
            (X_validation, y_validation_log)
        ],
        verbose=False
    )

    elapsed = time.time() - start_time

    print("\nTraining completed.")
    print(f"Training time: {elapsed:.2f} seconds")

    best_iteration = model.best_iteration
    best_trees = best_iteration + 1
    best_score = model.best_score

    print("Best iteration:", best_iteration)
    print("Best number of trees:", best_trees)
    print("Best validation RMSE:", best_score)

    # --------------------------------------------------------------
    # VALIDATION PREDICTIONS
    # --------------------------------------------------------------

    pred_log = model.predict(
        X_validation
    )

    metrics = evaluate_predictions(
        y_validation_log.to_numpy(),
        pred_log,
        y_validation_original.to_numpy()
    )

    print("\nValidation results:")

    print("\nORIGINAL CHL SCALE")
    print(f"MAE : {metrics['MAE']:.6f}")
    print(f"RMSE: {metrics['RMSE']:.6f}")
    print(f"R²  : {metrics['R2']:.6f}")
    print(f"Bias: {metrics['Bias']:.6f}")

    print("\nLOG1P SCALE")
    print(f"MAE : {metrics['Log_MAE']:.6f}")
    print(f"RMSE: {metrics['Log_RMSE']:.6f}")
    print(f"R²  : {metrics['Log_R2']:.6f}")
    print(f"Bias: {metrics['Log_Bias']:.6f}")

    results.append({
        "experiment": experiment_name,
        "max_depth": 6,
        "min_child_weight": 10,
        "learning_rate": lr,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 0.0,
        "reg_lambda": 1.0,
        "n_estimators": 2000,
        "early_stopping_rounds": 100,
        "MAE": metrics["MAE"],
        "RMSE": metrics["RMSE"],
        "R2": metrics["R2"],
        "Bias": metrics["Bias"],
        "Log_MAE": metrics["Log_MAE"],
        "Log_RMSE": metrics["Log_RMSE"],
        "Log_R2": metrics["Log_R2"],
        "Log_Bias": metrics["Log_Bias"],
        "best_iteration": best_iteration,
        "best_trees": best_trees,
        "best_validation_log_rmse": best_score,
        "training_time_seconds": elapsed
    })

    del model

# ======================================================================
# RESULTS TABLE
# ======================================================================

results_df = pd.DataFrame(results)

print("\n" + "=" * 70)
print("LEARNING RATE RESULTS")
print("=" * 70)

display(
    results_df[
        [
            "experiment",
            "learning_rate",
            "MAE",
            "RMSE",
            "R2",
            "Log_MAE",
            "Log_RMSE",
            "Log_R2",
            "Bias",
            "best_iteration",
            "best_trees"
        ]
    ]
)

# ======================================================================
# DETERMINE BEST MODEL BY EACH METRIC
# ======================================================================

print("\n" + "=" * 70)
print("BEST MODEL BY METRIC")
print("=" * 70)

best_mae = results_df.loc[
    results_df["MAE"].idxmin(),
    "experiment"
]

best_rmse = results_df.loc[
    results_df["RMSE"].idxmin(),
    "experiment"
]

best_r2 = results_df.loc[
    results_df["R2"].idxmax(),
    "experiment"
]

best_log_mae = results_df.loc[
    results_df["Log_MAE"].idxmin(),
    "experiment"
]

best_log_rmse = results_df.loc[
    results_df["Log_RMSE"].idxmin(),
    "experiment"
]

best_log_r2 = results_df.loc[
    results_df["Log_R2"].idxmax(),
    "experiment"
]

print("Original CHL MAE :", best_mae)
print("Original CHL RMSE:", best_rmse)
print("Original CHL R²  :", best_r2)

print("Log1p MAE        :", best_log_mae)
print("Log1p RMSE       :", best_log_rmse)
print("Log1p R²         :", best_log_r2)

# ======================================================================
# SELECT BEST CANDIDATE
# ======================================================================

# Rank candidates using the six primary performance metrics.
#
# Lower is better:
# MAE
# RMSE
# Log_MAE
# Log_RMSE
#
# Higher is better:
# R2
# Log_R2

ranking_df = results_df.copy()

ranking_df["MAE_rank"] = ranking_df["MAE"].rank(
    ascending=True,
    method="min"
)

ranking_df["RMSE_rank"] = ranking_df["RMSE"].rank(
    ascending=True,
    method="min"
)

ranking_df["R2_rank"] = ranking_df["R2"].rank(
    ascending=False,
    method="min"
)

ranking_df["Log_MAE_rank"] = ranking_df["Log_MAE"].rank(
    ascending=True,
    method="min"
)

ranking_df["Log_RMSE_rank"] = ranking_df["Log_RMSE"].rank(
    ascending=True,
    method="min"
)

ranking_df["Log_R2_rank"] = ranking_df["Log_R2"].rank(
    ascending=False,
    method="min"
)

ranking_df["average_rank"] = ranking_df[
    [
        "MAE_rank",
        "RMSE_rank",
        "R2_rank",
        "Log_MAE_rank",
        "Log_RMSE_rank",
        "Log_R2_rank"
    ]
].mean(axis=1)

ranking_df = ranking_df.sort_values(
    "average_rank"
).reset_index(drop=True)

ranking_df["overall_rank"] = (
    np.arange(len(ranking_df)) + 1
)

print("\n" + "=" * 70)
print("LEARNING RATE OVERALL RANKING")
print("=" * 70)

display(
    ranking_df[
        [
            "experiment",
            "learning_rate",
            "MAE",
            "RMSE",
            "R2",
            "Log_MAE",
            "Log_RMSE",
            "Log_R2",
            "average_rank",
            "overall_rank"
        ]
    ]
)

best_candidate = ranking_df.iloc[0]

print("\n" + "=" * 70)
print("CURRENT BEST LEARNING-RATE CANDIDATE")
print("=" * 70)

print(
    f"Winner: {best_candidate['experiment']}"
)

print(
    f"Learning rate: "
    f"{best_candidate['learning_rate']}"
)

print(
    f"Average rank: "
    f"{best_candidate['average_rank']:.4f}"
)

# ======================================================================
# SAVE RESULTS
# ======================================================================

results_file = os.path.join(
    OPT_DIR,
    "xgboost_learning_rate_results.csv"
)

ranking_file = os.path.join(
    OPT_DIR,
    "xgboost_learning_rate_ranking.csv"
)

summary_file = os.path.join(
    OPT_DIR,
    "xgboost_learning_rate_summary.json"
)

results_df.to_csv(
    results_file,
    index=False
)

ranking_df.to_csv(
    ranking_file,
    index=False
)

summary = {
    "step": "31.9P-16T-05",
    "experiment": "Learning Rate Optimization",

    "training_data_used": True,
    "validation_data_used": True,
    "test_data_loaded": False,
    "test_data_used": False,
    "test_predictions_generated": False,

    "features": 69,
    "training_rows": int(len(train_df)),
    "validation_rows": int(len(validation_df)),

    "fixed_parameters": {
        "max_depth": 6,
        "min_child_weight": 10,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 0.0,
        "reg_lambda": 1.0,
        "n_estimators": 2000,
        "early_stopping_rounds": 100,
        "random_state": 42
    },

    "tested_learning_rates": learning_rates,

    "best_candidate": {
        "experiment": best_candidate["experiment"],
        "learning_rate": float(
            best_candidate["learning_rate"]
        ),
        "average_rank": float(
            best_candidate["average_rank"]
        )
    }
}

with open(summary_file, "w") as f:
    json.dump(summary, f, indent=2)

print("\n" + "=" * 70)
print("RESULTS SAVED")
print("=" * 70)

print(results_file)
print(ranking_file)
print(summary_file)

# ======================================================================
# FINAL SAFETY CHECK
# ======================================================================

print("\n" + "=" * 70)
print("FINAL 31.9P-16T-05 SAFETY CHECK")
print("=" * 70)

print("Training data used       : TRUE")
print("Validation data used     : TRUE")
print("Test data loaded         : FALSE")
print("Test data used           : FALSE")
print("Test predictions         : FALSE")
print("Existing model modified  : FALSE")
print("Existing model overwritten: FALSE")

print("\n✓ Only learning_rate was varied.")
print("✓ max_depth = 6 remained fixed.")
print("✓ min_child_weight = 10 remained fixed.")
print("✓ Training/validation data only.")
print("✓ Test data was not loaded.")
print("✓ Existing XGBoost model remains untouched.")
print("✓ Learning-rate experiment completed.")

print("\n" + "=" * 70)
print("STEP 31.9P-16T-05 COMPLETE")
print("=" * 70)

print("""
NEXT:
Send me the complete output before running the next tuning stage.

DO NOT start another tuning experiment yet.
We will inspect which learning rate is best first.
""")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
STEP 31.9P-16T-05
XGBOOST LEARNING RATE OPTIMIZATION

PROJECT PATHS
Project:
/content/drive/MyDrive/Indhi_Marine_Research_Data

Optimization directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/02_XGBoost/hyperparameter_optimization

Feature list:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing/final_model_feature_columns.csv

Training data:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing/final_model_ready_train.csv

Validation data:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing/final_model_ready_validation.csv

Test data:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing/final_model_ready_test.csv

TEST DATA SAFETY
Test dataset exists on Drive: True

IMPORTANT:
Test data will NOT be loaded.
Test data will NOT be used.
Test predictions will NO

,experiment,learning_rate,MAE,RMSE,R2,Log_MAE,Log_RMSE,Log_R2,Bias,best_iteration,best_trees
0,LR1_0.01,0.01,0.183331,0.788530,0.557086,0.067400,0.148697,0.846349,-0.066810,447,448
1,LR2_reference_0.03,0.03,0.183352,0.789430,0.556075,0.067509,0.148762,0.846214,-0.068570,122,123
2,LR3_0.05,0.05,0.183749,0.786333,0.559551,0.067715,0.148389,0.846983,-0.059129,114,115



BEST MODEL BY METRIC
Original CHL MAE : LR1_0.01
Original CHL RMSE: LR3_0.05
Original CHL R²  : LR3_0.05
Log1p MAE        : LR1_0.01
Log1p RMSE       : LR3_0.05
Log1p R²         : LR3_0.05

LEARNING RATE OVERALL RANKING


,experiment,learning_rate,MAE,RMSE,R2,Log_MAE,Log_RMSE,Log_R2,average_rank,overall_rank
0,LR1_0.01,0.01,0.183331,0.788530,0.557086,0.067400,0.148697,0.846349,1.666667,1
1,LR3_0.05,0.05,0.183749,0.786333,0.559551,0.067715,0.148389,0.846983,1.666667,2
2,LR2_reference_0.03,0.03,0.183352,0.789430,0.556075,0.067509,0.148762,0.846214,2.666667,3



CURRENT BEST LEARNING-RATE CANDIDATE
Winner: LR1_0.01
Learning rate: 0.01
Average rank: 1.6667

RESULTS SAVED
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/02_XGBoost/hyperparameter_optimization/xgboost_learning_rate_results.csv
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/02_XGBoost/hyperparameter_optimization/xgboost_learning_rate_ranking.csv
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/02_XGBoost/hyperparameter_optimization/xgboost_learning_rate_summary.json

FINAL 31.9P-16T-05 SAFETY CHECK
Training data used       : TRUE
Validation data used     : TRUE
Test data loaded         : FALSE
Test data used           : FALSE
Test predictions         : FALSE
Existing model modified  : FALSE
Existing model overwritten: FALSE

✓ Only learning_rate was varied.
✓ max_depth = 6 remained fixed.
✓ min_child_weight = 10 remained fixed.
✓ Training/validation data only.
✓ Test data was not loaded.
✓ Existing XGBoost model remains untouched.
✓ Learning-rate

In [4]:
# ======================================================================
# STEP 31.9P-16T-06
# XGBOOST SUBSAMPLING OPTIMIZATION
# ======================================================================

import os
import json
import time
import numpy as np
import pandas as pd
import xgboost as xgb

from google.colab import drive
drive.mount('/content/drive')

print("=" * 70)
print("STEP 31.9P-16T-06")
print("XGBOOST SUBSAMPLING OPTIMIZATION")
print("=" * 70)


# ======================================================================
# PROJECT PATHS
# ======================================================================

PROJECT_DIR = "/content/drive/MyDrive/Indhi_Marine_Research_Data"

OPT_DIR = os.path.join(
    PROJECT_DIR,
    "06_models",
    "02_XGBoost",
    "hyperparameter_optimization"
)

FEATURE_FILE = os.path.join(
    PROJECT_DIR,
    "05_model_preprocessing",
    "final_model_feature_columns.csv"
)

TRAIN_FILE = os.path.join(
    PROJECT_DIR,
    "05_model_preprocessing",
    "final_model_ready_train.csv"
)

VALIDATION_FILE = os.path.join(
    PROJECT_DIR,
    "05_model_preprocessing",
    "final_model_ready_validation.csv"
)

TEST_FILE = os.path.join(
    PROJECT_DIR,
    "05_model_preprocessing",
    "final_model_ready_test.csv"
)

os.makedirs(OPT_DIR, exist_ok=True)


print("\n" + "=" * 70)
print("PROJECT PATHS")
print("=" * 70)

print("Project:")
print(PROJECT_DIR)

print("\nOptimization directory:")
print(OPT_DIR)

print("\nFeature list:")
print(FEATURE_FILE)

print("\nTraining data:")
print(TRAIN_FILE)

print("\nValidation data:")
print(VALIDATION_FILE)

print("\nTest data:")
print(TEST_FILE)


# ======================================================================
# TEST DATA SAFETY
# ======================================================================

print("\n" + "=" * 70)
print("TEST DATA SAFETY")
print("=" * 70)

print(f"Test dataset exists on Drive: {os.path.exists(TEST_FILE)}")

print("""
IMPORTANT:
Test data will NOT be loaded.
Test data will NOT be used.
Test predictions will NOT be generated.
""")


# ======================================================================
# REQUIRED FILE CHECK
# ======================================================================

print("=" * 70)
print("REQUIRED FILE CHECK")
print("=" * 70)

required_files = {
    "Feature list": FEATURE_FILE,
    "Training data": TRAIN_FILE,
    "Validation data": VALIDATION_FILE
}

for name, path in required_files.items():
    print(f"{name}: {os.path.exists(path)}")

if not all(os.path.exists(path) for path in required_files.values()):
    raise FileNotFoundError(
        "One or more required training/validation files are missing."
    )

print("\n✓ All required training/validation files found.")


# ======================================================================
# LOAD FEATURE LIST
# ======================================================================

print("\n" + "=" * 70)
print("LOADING FINAL FEATURE LIST")
print("=" * 70)

feature_df = pd.read_csv(FEATURE_FILE)

FEATURES = feature_df["feature"].tolist()

print("Number of features:", len(FEATURES))

if len(FEATURES) != 69:
    raise ValueError(
        f"Expected exactly 69 features, found {len(FEATURES)}"
    )

print("✓ Exactly 69 predictors confirmed.")


# ======================================================================
# LOAD TRAINING DATA
# ======================================================================

print("\n" + "=" * 70)
print("LOADING TRAINING DATA")
print("=" * 70)

train_df = pd.read_csv(TRAIN_FILE)

print("Training shape:", train_df.shape)


# ======================================================================
# LOAD VALIDATION DATA
# ======================================================================

print("\n" + "=" * 70)
print("LOADING VALIDATION DATA")
print("=" * 70)

val_df = pd.read_csv(VALIDATION_FILE)

print("Validation shape:", val_df.shape)


# ======================================================================
# TARGET CONFIGURATION
# ======================================================================

TARGET_LOG = "chlorophyll_a_log1p"
TARGET_ORIGINAL = "chlorophyll_a"

print("\n" + "=" * 70)
print("TARGET CONFIGURATION")
print("=" * 70)

print("Model target:", TARGET_LOG)
print("Original target:", TARGET_ORIGINAL)


# ======================================================================
# COLUMN CHECK
# ======================================================================

required_columns = FEATURES + [
    TARGET_LOG,
    TARGET_ORIGINAL
]

train_missing = [
    c for c in required_columns
    if c not in train_df.columns
]

val_missing = [
    c for c in required_columns
    if c not in val_df.columns
]

print("\nTraining missing columns:", len(train_missing))
print("Validation missing columns:", len(val_missing))

if train_missing:
    print("Training missing:", train_missing)

if val_missing:
    print("Validation missing:", val_missing)

if train_missing or val_missing:
    raise ValueError("Required columns are missing.")

print("✓ All required columns available.")


# ======================================================================
# PREPARE INPUT ARRAYS
# ======================================================================

X_train = train_df[FEATURES]
y_train = train_df[TARGET_LOG]

X_val = val_df[FEATURES]
y_val_log = val_df[TARGET_LOG]
y_val_original = val_df[TARGET_ORIGINAL]

print("\n" + "=" * 70)
print("PREPARING MODEL INPUTS")
print("=" * 70)

print("Training X:", X_train.shape)
print("Training y:", y_train.shape)

print("\nValidation X:", X_val.shape)
print("Validation y:", y_val_log.shape)


# ======================================================================
# DATA SAFETY CHECK
# ======================================================================

print("\n" + "=" * 70)
print("DATA SAFETY CHECK")
print("=" * 70)

train_nan = X_train.isna().sum().sum()
train_inf = np.isinf(X_train.to_numpy()).sum()
train_target_nan = y_train.isna().sum()

val_nan = X_val.isna().sum().sum()
val_inf = np.isinf(X_val.to_numpy()).sum()
val_target_nan = y_val_log.isna().sum()
val_original_nan = y_val_original.isna().sum()

print("Training predictor NaN      :", train_nan)
print("Training predictor infinite :", train_inf)
print("Training target NaN         :", train_target_nan)

print("\nValidation predictor NaN      :", val_nan)
print("Validation predictor infinite :", val_inf)
print("Validation target NaN         :", val_target_nan)
print("Validation original CHL NaN   :", val_original_nan)

if any([
    train_nan,
    train_inf,
    train_target_nan,
    val_nan,
    val_inf,
    val_target_nan,
    val_original_nan
]):
    raise ValueError(
        "Training/validation data contain invalid values."
    )

print("\n✓ Training and validation data are clean.")


# ======================================================================
# GPU VERIFICATION
# ======================================================================

print("\n" + "=" * 70)
print("GPU VERIFICATION")
print("=" * 70)

print("XGBoost version:", xgb.__version__)

build_info = xgb.build_info()

print("CUDA enabled:", build_info.get("USE_CUDA", False))

if not build_info.get("USE_CUDA", False):
    raise RuntimeError("CUDA-enabled XGBoost is required.")

print("✓ CUDA-enabled XGBoost confirmed.")


# ======================================================================
# CONTROLLED EXPERIMENT CONFIGURATION
# ======================================================================

print("\n" + "=" * 70)
print("SUBSAMPLING EXPERIMENT")
print("=" * 70)

print("""
Previous best configuration:

max_depth        = 6
min_child_weight = 10
learning_rate    = 0.05
subsample        = 0.8
colsample_bytree = 0.8
reg_alpha        = 0.0
reg_lambda       = 1.0
n_estimators     = 2000
early stopping   = 100

Only subsample will change.
All other parameters remain fixed.
""")

SUBSAMPLE_VALUES = [0.7, 0.8, 0.9]

print("Subsample values to test:", SUBSAMPLE_VALUES)


# ======================================================================
# FIXED PARAMETERS
# ======================================================================

FIXED_PARAMS = {
    "objective": "reg:squarederror",
    "n_estimators": 2000,
    "max_depth": 6,
    "learning_rate": 0.05,
    "colsample_bytree": 0.8,
    "min_child_weight": 10,
    "reg_alpha": 0.0,
    "reg_lambda": 1.0,
    "random_state": 42,
    "tree_method": "hist",
    "device": "cuda",
    "early_stopping_rounds": 100
}


# ======================================================================
# RUN EXPERIMENTS
# ======================================================================

results = []

for i, subsample_value in enumerate(SUBSAMPLE_VALUES, start=1):

    experiment_name = f"SUB{i}_{subsample_value}"

    print("\n" + "=" * 70)
    print(f"RUNNING: {experiment_name}")
    print("=" * 70)

    print("\nParameters:")
    print("max_depth         :", FIXED_PARAMS["max_depth"])
    print("min_child_weight  :", FIXED_PARAMS["min_child_weight"])
    print("learning_rate     :", FIXED_PARAMS["learning_rate"])
    print("subsample         :", subsample_value)
    print("colsample_bytree  :", FIXED_PARAMS["colsample_bytree"])
    print("reg_alpha         :", FIXED_PARAMS["reg_alpha"])
    print("reg_lambda        :", FIXED_PARAMS["reg_lambda"])
    print("n_estimators      :", FIXED_PARAMS["n_estimators"])
    print("early stopping    :", FIXED_PARAMS["early_stopping_rounds"])
    print("device             :", FIXED_PARAMS["device"])

    params = FIXED_PARAMS.copy()
    params["subsample"] = subsample_value

    model = xgb.XGBRegressor(**params)

    print("\n✓ Temporary model created.")
    print("✓ Existing XGBoost model remains untouched.")

    start_time = time.time()

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_val, y_val_log)],
        verbose=False
    )

    training_time = time.time() - start_time

    best_iteration = model.best_iteration
    best_trees = best_iteration + 1

    print("\nTraining completed.")
    print(f"Training time: {training_time:.2f} seconds")
    print(f"Best iteration: {best_iteration}")
    print(f"Best number of trees: {best_trees}")
    print(f"Best validation RMSE: {model.best_score:.12f}")


    # --------------------------------------------------------------
    # VALIDATION PREDICTIONS
    # --------------------------------------------------------------

    pred_log = model.predict(X_val)

    pred_original = np.expm1(pred_log)

    actual_log = y_val_log.to_numpy()
    actual_original = y_val_original.to_numpy()


    # --------------------------------------------------------------
    # LOG METRICS
    # --------------------------------------------------------------

    log_error = pred_log - actual_log

    log_mae = np.mean(np.abs(log_error))
    log_rmse = np.sqrt(np.mean(log_error ** 2))

    ss_res_log = np.sum(log_error ** 2)
    ss_tot_log = np.sum(
        (actual_log - np.mean(actual_log)) ** 2
    )

    log_r2 = 1 - (ss_res_log / ss_tot_log)

    log_bias = np.mean(log_error)


    # --------------------------------------------------------------
    # ORIGINAL CHL METRICS
    # --------------------------------------------------------------

    original_error = pred_original - actual_original

    mae = np.mean(np.abs(original_error))
    rmse = np.sqrt(np.mean(original_error ** 2))

    ss_res = np.sum(original_error ** 2)
    ss_tot = np.sum(
        (actual_original - np.mean(actual_original)) ** 2
    )

    r2 = 1 - (ss_res / ss_tot)

    bias = np.mean(original_error)


    print("\nValidation results:")

    print("\nORIGINAL CHL SCALE")
    print(f"MAE : {mae:.6f}")
    print(f"RMSE: {rmse:.6f}")
    print(f"R²  : {r2:.6f}")
    print(f"Bias: {bias:.6f}")

    print("\nLOG1P SCALE")
    print(f"MAE : {log_mae:.6f}")
    print(f"RMSE: {log_rmse:.6f}")
    print(f"R²  : {log_r2:.6f}")
    print(f"Bias: {log_bias:.6f}")


    results.append({
        "experiment": experiment_name,
        "subsample": subsample_value,

        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,

        "Log_MAE": log_mae,
        "Log_RMSE": log_rmse,
        "Log_R2": log_r2,

        "Bias": bias,
        "Log_Bias": log_bias,

        "best_iteration": best_iteration,
        "best_trees": best_trees,

        "training_time_seconds": training_time
    })


# ======================================================================
# RESULTS TABLE
# ======================================================================

results_df = pd.DataFrame(results)

print("\n" + "=" * 70)
print("SUBSAMPLING RESULTS")
print("=" * 70)

print(results_df.to_string(index=False))


# ======================================================================
# BEST MODEL BY INDIVIDUAL METRIC
# ======================================================================

print("\n" + "=" * 70)
print("BEST MODEL BY METRIC")
print("=" * 70)

best_mae = results_df.loc[
    results_df["MAE"].idxmin(),
    "experiment"
]

best_rmse = results_df.loc[
    results_df["RMSE"].idxmin(),
    "experiment"
]

best_r2 = results_df.loc[
    results_df["R2"].idxmax(),
    "experiment"
]

best_log_mae = results_df.loc[
    results_df["Log_MAE"].idxmin(),
    "experiment"
]

best_log_rmse = results_df.loc[
    results_df["Log_RMSE"].idxmin(),
    "experiment"
]

best_log_r2 = results_df.loc[
    results_df["Log_R2"].idxmax(),
    "experiment"
]

print("Original CHL MAE :", best_mae)
print("Original CHL RMSE:", best_rmse)
print("Original CHL R²  :", best_r2)

print("Log1p MAE        :", best_log_mae)
print("Log1p RMSE       :", best_log_rmse)
print("Log1p R²         :", best_log_r2)


# ======================================================================
# OVERALL RANKING
# ======================================================================

rank_df = results_df.copy()

rank_df["MAE_rank"] = rank_df["MAE"].rank(
    ascending=True,
    method="min"
)

rank_df["RMSE_rank"] = rank_df["RMSE"].rank(
    ascending=True,
    method="min"
)

rank_df["R2_rank"] = rank_df["R2"].rank(
    ascending=False,
    method="min"
)

rank_df["Log_MAE_rank"] = rank_df["Log_MAE"].rank(
    ascending=True,
    method="min"
)

rank_df["Log_RMSE_rank"] = rank_df["Log_RMSE"].rank(
    ascending=True,
    method="min"
)

rank_df["Log_R2_rank"] = rank_df["Log_R2"].rank(
    ascending=False,
    method="min"
)

rank_df["average_rank"] = rank_df[
    [
        "MAE_rank",
        "RMSE_rank",
        "R2_rank",
        "Log_MAE_rank",
        "Log_RMSE_rank",
        "Log_R2_rank"
    ]
].mean(axis=1)

rank_df = rank_df.sort_values(
    "average_rank"
).reset_index(drop=True)

rank_df["overall_rank"] = np.arange(
    1,
    len(rank_df) + 1
)


print("\n" + "=" * 70)
print("SUBSAMPLING OVERALL RANKING")
print("=" * 70)

print(
    rank_df[
        [
            "experiment",
            "subsample",
            "MAE",
            "RMSE",
            "R2",
            "Log_MAE",
            "Log_RMSE",
            "Log_R2",
            "average_rank",
            "overall_rank"
        ]
    ].to_string(index=False)
)


# ======================================================================
# CURRENT BEST SUBSAMPLE
# ======================================================================

best_row = rank_df.iloc[0]

best_subsample = float(
    best_row["subsample"]
)

print("\n" + "=" * 70)
print("CURRENT BEST SUBSAMPLING CANDIDATE")
print("=" * 70)

print("Winner:", best_row["experiment"])
print("Subsample:", best_subsample)
print(
    "Average rank:",
    f"{best_row['average_rank']:.4f}"
)


# ======================================================================
# SAVE RESULTS
# ======================================================================

results_path = os.path.join(
    OPT_DIR,
    "xgboost_subsampling_results.csv"
)

ranking_path = os.path.join(
    OPT_DIR,
    "xgboost_subsampling_ranking.csv"
)

summary_path = os.path.join(
    OPT_DIR,
    "xgboost_subsampling_summary.json"
)

results_df.to_csv(
    results_path,
    index=False
)

rank_df.to_csv(
    ranking_path,
    index=False
)

summary = {
    "step": "31.9P-16T-06",
    "experiment": "XGBoost subsampling optimization",

    "test_data_loaded": False,
    "test_data_used": False,
    "test_predictions_generated": False,

    "training_rows": len(train_df),
    "validation_rows": len(val_df),
    "number_of_features": len(FEATURES),

    "fixed_parameters": {
        "max_depth": 6,
        "min_child_weight": 10,
        "learning_rate": 0.05,
        "colsample_bytree": 0.8,
        "reg_alpha": 0.0,
        "reg_lambda": 1.0,
        "n_estimators": 2000,
        "early_stopping_rounds": 100,
        "random_state": 42,
        "tree_method": "hist",
        "device": "cuda"
    },

    "tested_subsample_values": SUBSAMPLE_VALUES,

    "best_subsample_candidate": best_subsample,

    "selection_basis": (
        "Validation-only average rank across "
        "original-scale and log1p MAE, RMSE and R2."
    )
}

with open(summary_path, "w") as f:
    json.dump(summary, f, indent=4)


print("\n" + "=" * 70)
print("RESULTS SAVED")
print("=" * 70)

print(results_path)
print(ranking_path)
print(summary_path)


# ======================================================================
# FINAL SAFETY CHECK
# ======================================================================

print("\n" + "=" * 70)
print("FINAL 31.9P-16T-06 SAFETY CHECK")
print("=" * 70)

print("Training data used       : TRUE")
print("Validation data used     : TRUE")
print("Test data loaded         : FALSE")
print("Test data used           : FALSE")
print("Test predictions         : FALSE")
print("Existing model modified  : FALSE")
print("Existing model overwritten: FALSE")

print("\n✓ Only subsample was varied.")
print("✓ max_depth = 6 remained fixed.")
print("✓ min_child_weight = 10 remained fixed.")
print("✓ learning_rate = 0.05 remained fixed.")
print("✓ Training/validation data only.")
print("✓ Test data was not loaded.")
print("✓ Existing XGBoost model remains untouched.")
print("✓ Subsampling experiment completed.")


print("\n" + "=" * 70)
print("STEP 31.9P-16T-06 COMPLETE")
print("=" * 70)

print("\nNEXT:")
print("Send me the complete output before running the next tuning stage.")
print("DO NOT start another tuning experiment yet.")
print("We will inspect which subsample value is best first.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
STEP 31.9P-16T-06
XGBOOST SUBSAMPLING OPTIMIZATION

PROJECT PATHS
Project:
/content/drive/MyDrive/Indhi_Marine_Research_Data

Optimization directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/02_XGBoost/hyperparameter_optimization

Feature list:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing/final_model_feature_columns.csv

Training data:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing/final_model_ready_train.csv

Validation data:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing/final_model_ready_validation.csv

Test data:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing/final_model_ready_test.csv

TEST DATA SAFETY
Test dataset exists on Drive: True

IMPORTANT:
Test data will NOT be loaded.
Test data will NOT be used.
Test predictions will NOT 

In [5]:
# ======================================================================
# STEP 31.9P-16T-07
# XGBOOST COLUMNSAMPLING OPTIMIZATION
# ======================================================================

import os
import json
import time
import warnings
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

warnings.filterwarnings("ignore")

print("=" * 70)
print("STEP 31.9P-16T-07")
print("XGBOOST COLUMNSAMPLING OPTIMIZATION")
print("=" * 70)

# ======================================================================
# PROJECT PATHS
# ======================================================================

PROJECT_DIR = "/content/drive/MyDrive/Indhi_Marine_Research_Data"

OPT_DIR = os.path.join(
    PROJECT_DIR,
    "06_models",
    "02_XGBoost",
    "hyperparameter_optimization"
)

FEATURE_FILE = os.path.join(
    PROJECT_DIR,
    "05_model_preprocessing",
    "final_model_feature_columns.csv"
)

TRAIN_FILE = os.path.join(
    PROJECT_DIR,
    "05_model_preprocessing",
    "final_model_ready_train.csv"
)

VALIDATION_FILE = os.path.join(
    PROJECT_DIR,
    "05_model_preprocessing",
    "final_model_ready_validation.csv"
)

TEST_FILE = os.path.join(
    PROJECT_DIR,
    "05_model_preprocessing",
    "final_model_ready_test.csv"
)

os.makedirs(OPT_DIR, exist_ok=True)

print("\n" + "=" * 70)
print("PROJECT PATHS")
print("=" * 70)

print(f"\nProject:\n{PROJECT_DIR}")
print(f"\nOptimization directory:\n{OPT_DIR}")
print(f"\nFeature list:\n{FEATURE_FILE}")
print(f"\nTraining data:\n{TRAIN_FILE}")
print(f"\nValidation data:\n{VALIDATION_FILE}")
print(f"\nTest data:\n{TEST_FILE}")

# ======================================================================
# TEST DATA SAFETY
# ======================================================================

print("\n" + "=" * 70)
print("TEST DATA SAFETY")
print("=" * 70)

print(f"\nTest dataset exists on Drive: {os.path.exists(TEST_FILE)}")

print("""
IMPORTANT:
Test data will NOT be loaded.
Test data will NOT be used.
Test predictions will NOT be generated.
""")

# ======================================================================
# REQUIRED FILE CHECK
# ======================================================================

print("=" * 70)
print("REQUIRED FILE CHECK")
print("=" * 70)

required_files = {
    "Feature list": FEATURE_FILE,
    "Training data": TRAIN_FILE,
    "Validation data": VALIDATION_FILE
}

for name, path in required_files.items():
    exists = os.path.exists(path)
    print(f"{name}: {exists}")

    if not exists:
        raise FileNotFoundError(
            f"Required file not found: {path}"
        )

print("\n✓ All required training/validation files found.")

# ======================================================================
# LOAD FEATURE LIST
# ======================================================================

print("\n" + "=" * 70)
print("LOADING FINAL FEATURE LIST")
print("=" * 70)

feature_df = pd.read_csv(FEATURE_FILE)

if "feature" not in feature_df.columns:
    raise ValueError(
        "Feature list does not contain the 'feature' column."
    )

FEATURES = feature_df["feature"].tolist()

print(f"Number of features: {len(FEATURES)}")

if len(FEATURES) != 69:
    raise ValueError(
        f"Expected exactly 69 features, found {len(FEATURES)}."
    )

print("✓ Exactly 69 predictors confirmed.")

# ======================================================================
# LOAD TRAINING DATA
# ======================================================================

print("\n" + "=" * 70)
print("LOADING TRAINING DATA")
print("=" * 70)

train_df = pd.read_csv(TRAIN_FILE)

print(f"Training shape: {train_df.shape}")

# ======================================================================
# LOAD VALIDATION DATA
# ======================================================================

print("\n" + "=" * 70)
print("LOADING VALIDATION DATA")
print("=" * 70)

validation_df = pd.read_csv(VALIDATION_FILE)

print(f"Validation shape: {validation_df.shape}")

# ======================================================================
# TARGET CONFIGURATION
# ======================================================================

TARGET_LOG = "chlorophyll_a_log1p"
TARGET_ORIGINAL = "chlorophyll_a"

print("\n" + "=" * 70)
print("TARGET CONFIGURATION")
print("=" * 70)

print(f"Model target: {TARGET_LOG}")
print(f"Original target: {TARGET_ORIGINAL}")

# ======================================================================
# COLUMN CHECK
# ======================================================================

required_columns = FEATURES + [
    TARGET_LOG,
    TARGET_ORIGINAL
]

train_missing = [
    c for c in required_columns
    if c not in train_df.columns
]

validation_missing = [
    c for c in required_columns
    if c not in validation_df.columns
]

print(f"\nTraining missing columns: {len(train_missing)}")
print(f"Validation missing columns: {len(validation_missing)}")

if train_missing:
    print(train_missing)
    raise ValueError("Training data is missing required columns.")

if validation_missing:
    print(validation_missing)
    raise ValueError("Validation data is missing required columns.")

print("✓ All required columns available.")

# ======================================================================
# PREPARE INPUT ARRAYS
# ======================================================================

print("\n" + "=" * 70)
print("PREPARING MODEL INPUTS")
print("=" * 70)

X_train = train_df[FEATURES].values
y_train = train_df[TARGET_LOG].values

X_validation = validation_df[FEATURES].values
y_validation_log = validation_df[TARGET_LOG].values
y_validation_original = validation_df[TARGET_ORIGINAL].values

print(f"Training X: {X_train.shape}")
print(f"Training y: {y_train.shape}")

print(f"Validation X: {X_validation.shape}")
print(f"Validation y: {y_validation_log.shape}")

# ======================================================================
# DATA SAFETY CHECK
# ======================================================================

print("\n" + "=" * 70)
print("DATA SAFETY CHECK")
print("=" * 70)

checks = {
    "Training predictor NaN": np.isnan(X_train).sum(),
    "Training predictor infinite": np.isinf(X_train).sum(),
    "Training target NaN": np.isnan(y_train).sum(),
    "Validation predictor NaN": np.isnan(X_validation).sum(),
    "Validation predictor infinite": np.isinf(X_validation).sum(),
    "Validation target NaN": np.isnan(y_validation_log).sum(),
    "Validation original CHL NaN": np.isnan(y_validation_original).sum()
}

for name, value in checks.items():
    print(f"{name:40s}: {value}")

if any(value != 0 for value in checks.values()):
    raise ValueError(
        "NaN or infinite values detected."
    )

print("\n✓ Training and validation data are clean.")

# ======================================================================
# GPU VERIFICATION
# ======================================================================

print("\n" + "=" * 70)
print("GPU VERIFICATION")
print("=" * 70)

print(f"XGBoost version: {xgb.__version__}")

try:
    build_info = xgb.build_info()

    cuda_enabled = build_info.get(
        "USE_CUDA",
        False
    )

    print(f"CUDA enabled: {cuda_enabled}")

except Exception:
    cuda_enabled = False
    print("Unable to read XGBoost build information.")

# Check NVIDIA GPU using system command
gpu_available = False

try:
    import subprocess

    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
        capture_output=True,
        text=True
    )

    if result.returncode == 0 and result.stdout.strip():
        gpu_available = True
        print(f"GPU detected: {result.stdout.strip()}")

except Exception:
    pass

if not gpu_available:
    raise RuntimeError(
        "No NVIDIA GPU detected. "
        "Do not run this experiment because the controlled "
        "experiments are intended to use the GPU."
    )

print("✓ CUDA-enabled GPU confirmed.")

# ======================================================================
# CURRENT BEST CONFIGURATION
# ======================================================================

print("\n" + "=" * 70)
print("CURRENT BEST CONFIGURATION")
print("=" * 70)

BASE_PARAMS = {
    "objective": "reg:squarederror",
    "n_estimators": 2000,
    "max_depth": 6,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 10,
    "reg_alpha": 0.0,
    "reg_lambda": 1.0,
    "random_state": 42,
    "tree_method": "hist",
    "device": "cuda"
}

print(f"max_depth        : {BASE_PARAMS['max_depth']}")
print(f"min_child_weight : {BASE_PARAMS['min_child_weight']}")
print(f"learning_rate    : {BASE_PARAMS['learning_rate']}")
print(f"subsample        : {BASE_PARAMS['subsample']}")
print(f"colsample_bytree : {BASE_PARAMS['colsample_bytree']}")
print(f"reg_alpha        : {BASE_PARAMS['reg_alpha']}")
print(f"reg_lambda       : {BASE_PARAMS['reg_lambda']}")
print(f"n_estimators     : {BASE_PARAMS['n_estimators']}")
print("early stopping   : 100")

# ======================================================================
# COLUMNSAMPLING EXPERIMENT
# ======================================================================

print("\n" + "=" * 70)
print("COLUMNSAMPLING EXPERIMENT")
print("=" * 70)

print("""
Only colsample_bytree will change.

All other parameters remain fixed.

Values to test:
    0.7
    0.8
    0.9
""")

COLSAMPLE_VALUES = [0.7, 0.8, 0.9]

results = []

# ======================================================================
# RUN EXPERIMENTS
# ======================================================================

for i, colsample_value in enumerate(
    COLSAMPLE_VALUES,
    start=1
):

    experiment_name = f"COL{i}_{str(colsample_value).replace('.', '_')}"

    print("\n" + "=" * 70)
    print(f"RUNNING: {experiment_name}")
    print("=" * 70)

    params = BASE_PARAMS.copy()
    params["colsample_bytree"] = colsample_value

    print("\nParameters:")
    for key, value in params.items():
        print(f"{key:20s}: {value}")

    print("early_stopping_rounds : 100")

    # --------------------------------------------------------------
    # CREATE TEMPORARY MODEL
    # --------------------------------------------------------------

    model = xgb.XGBRegressor(
        **params,
        early_stopping_rounds=100
    )

    print("\n✓ Temporary model created.")
    print("✓ Existing XGBoost model remains untouched.")

    # --------------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------------

    start_time = time.time()

    model.fit(
        X_train,
        y_train,
        eval_set=[
            (X_train, y_train),
            (X_validation, y_validation_log)
        ],
        verbose=False
    )

    elapsed = time.time() - start_time

    # --------------------------------------------------------------
    # BEST ITERATION
    # --------------------------------------------------------------

    best_iteration = model.best_iteration
    best_trees = best_iteration + 1
    best_score = model.best_score

    print("\nTraining completed.")
    print(f"Training time: {elapsed:.2f} seconds")
    print(f"Best iteration: {best_iteration}")
    print(f"Best number of trees: {best_trees}")
    print(f"Best validation RMSE: {best_score:.12f}")

    # --------------------------------------------------------------
    # VALIDATION PREDICTIONS
    # --------------------------------------------------------------

    pred_log = model.predict(
        X_validation
    )

    pred_original = np.expm1(pred_log)

    # --------------------------------------------------------------
    # ORIGINAL CHL METRICS
    # --------------------------------------------------------------

    original_mae = mean_absolute_error(
        y_validation_original,
        pred_original
    )

    original_rmse = np.sqrt(
        mean_squared_error(
            y_validation_original,
            pred_original
        )
    )

    original_r2 = r2_score(
        y_validation_original,
        pred_original
    )

    original_bias = np.mean(
        pred_original - y_validation_original
    )

    # --------------------------------------------------------------
    # LOG1P METRICS
    # --------------------------------------------------------------

    log_mae = mean_absolute_error(
        y_validation_log,
        pred_log
    )

    log_rmse = np.sqrt(
        mean_squared_error(
            y_validation_log,
            pred_log
        )
    )

    log_r2 = r2_score(
        y_validation_log,
        pred_log
    )

    log_bias = np.mean(
        pred_log - y_validation_log
    )

    # --------------------------------------------------------------
    # STORE RESULTS
    # --------------------------------------------------------------

    results.append({
        "experiment": experiment_name,
        "colsample_bytree": colsample_value,

        "MAE": original_mae,
        "RMSE": original_rmse,
        "R2": original_r2,
        "Bias": original_bias,

        "Log_MAE": log_mae,
        "Log_RMSE": log_rmse,
        "Log_R2": log_r2,
        "Log_Bias": log_bias,

        "best_iteration": best_iteration,
        "best_trees": best_trees,
        "best_validation_log_rmse": best_score,

        "training_time_seconds": elapsed
    })

    # --------------------------------------------------------------
    # DISPLAY RESULTS
    # --------------------------------------------------------------

    print("\nValidation results:")

    print("\nORIGINAL CHL SCALE")
    print(f"MAE : {original_mae:.6f}")
    print(f"RMSE: {original_rmse:.6f}")
    print(f"R²  : {original_r2:.6f}")
    print(f"Bias: {original_bias:.6f}")

    print("\nLOG1P SCALE")
    print(f"MAE : {log_mae:.6f}")
    print(f"RMSE: {log_rmse:.6f}")
    print(f"R²  : {log_r2:.6f}")
    print(f"Bias: {log_bias:.6f}")

    # Explicitly delete temporary model
    del model

# ======================================================================
# RESULTS DATAFRAME
# ======================================================================

results_df = pd.DataFrame(results)

print("\n" + "=" * 70)
print("COLUMNSAMPLING RESULTS")
print("=" * 70)

print(
    results_df[
        [
            "experiment",
            "colsample_bytree",
            "MAE",
            "RMSE",
            "R2",
            "Log_MAE",
            "Log_RMSE",
            "Log_R2",
            "Bias",
            "Log_Bias",
            "best_iteration",
            "best_trees",
            "training_time_seconds"
        ]
    ].to_string(index=False)
)

# ======================================================================
# MULTI-METRIC RANKING
# ======================================================================

print("\n" + "=" * 70)
print("COLUMNSAMPLING OVERALL RANKING")
print("=" * 70)

ranking_df = results_df.copy()

# Lower is better
ranking_df["MAE_rank"] = ranking_df["MAE"].rank(
    ascending=True,
    method="min"
)

ranking_df["RMSE_rank"] = ranking_df["RMSE"].rank(
    ascending=True,
    method="min"
)

# Higher is better
ranking_df["R2_rank"] = ranking_df["R2"].rank(
    ascending=False,
    method="min"
)

ranking_df["Log_MAE_rank"] = ranking_df["Log_MAE"].rank(
    ascending=True,
    method="min"
)

ranking_df["Log_RMSE_rank"] = ranking_df["Log_RMSE"].rank(
    ascending=True,
    method="min"
)

ranking_df["Log_R2_rank"] = ranking_df["Log_R2"].rank(
    ascending=False,
    method="min"
)

ranking_columns = [
    "MAE_rank",
    "RMSE_rank",
    "R2_rank",
    "Log_MAE_rank",
    "Log_RMSE_rank",
    "Log_R2_rank"
]

ranking_df["average_rank"] = ranking_df[
    ranking_columns
].mean(axis=1)

ranking_df = ranking_df.sort_values(
    "average_rank"
).reset_index(drop=True)

ranking_df["overall_rank"] = (
    ranking_df.index + 1
)

print(
    ranking_df[
        [
            "experiment",
            "colsample_bytree",
            "MAE",
            "RMSE",
            "R2",
            "Log_MAE",
            "Log_RMSE",
            "Log_R2",
            "average_rank",
            "overall_rank"
        ]
    ].to_string(index=False)
)

# ======================================================================
# BEST MODEL BY INDIVIDUAL METRIC
# ======================================================================

print("\n" + "=" * 70)
print("BEST MODEL BY METRIC")
print("=" * 70)

best_by_metric = {}

best_by_metric["Original CHL MAE"] = results_df.loc[
    results_df["MAE"].idxmin(),
    "experiment"
]

best_by_metric["Original CHL RMSE"] = results_df.loc[
    results_df["RMSE"].idxmin(),
    "experiment"
]

best_by_metric["Original CHL R²"] = results_df.loc[
    results_df["R2"].idxmax(),
    "experiment"
]

best_by_metric["Log1p MAE"] = results_df.loc[
    results_df["Log_MAE"].idxmin(),
    "experiment"
]

best_by_metric["Log1p RMSE"] = results_df.loc[
    results_df["Log_RMSE"].idxmin(),
    "experiment"
]

best_by_metric["Log1p R²"] = results_df.loc[
    results_df["Log_R2"].idxmax(),
    "experiment"
]

for metric, winner in best_by_metric.items():
    print(f"{metric:25s}: {winner}")

# ======================================================================
# CURRENT WINNER
# ======================================================================

winner_row = ranking_df.iloc[0]

winner_experiment = winner_row["experiment"]
winner_colsample = winner_row["colsample_bytree"]
winner_average_rank = winner_row["average_rank"]

print("\n" + "=" * 70)
print("CURRENT BEST COLUMNSAMPLING CANDIDATE")
print("=" * 70)

print(f"Winner: {winner_experiment}")
print(f"colsample_bytree: {winner_colsample}")
print(f"Average rank: {winner_average_rank:.4f}")

# ======================================================================
# SAVE RESULTS
# ======================================================================

results_file = os.path.join(
    OPT_DIR,
    "xgboost_colsample_results.csv"
)

ranking_file = os.path.join(
    OPT_DIR,
    "xgboost_colsample_ranking.csv"
)

summary_file = os.path.join(
    OPT_DIR,
    "xgboost_colsample_summary.json"
)

results_df.to_csv(
    results_file,
    index=False
)

ranking_df.to_csv(
    ranking_file,
    index=False
)

summary = {
    "step": "31.9P-16T-07",
    "experiment": "XGBoost Column Subsampling Optimization",

    "test_data_loaded": False,
    "test_data_used": False,
    "test_predictions_generated": False,

    "training_rows": int(len(train_df)),
    "validation_rows": int(len(validation_df)),
    "number_of_features": int(len(FEATURES)),

    "parameter_tested": "colsample_bytree",
    "values_tested": COLSAMPLE_VALUES,

    "fixed_parameters": {
        "max_depth": BASE_PARAMS["max_depth"],
        "min_child_weight": BASE_PARAMS["min_child_weight"],
        "learning_rate": BASE_PARAMS["learning_rate"],
        "subsample": BASE_PARAMS["subsample"],
        "reg_alpha": BASE_PARAMS["reg_alpha"],
        "reg_lambda": BASE_PARAMS["reg_lambda"],
        "n_estimators": BASE_PARAMS["n_estimators"],
        "early_stopping_rounds": 100,
        "random_state": BASE_PARAMS["random_state"]
    },

    "winner": {
        "experiment": winner_experiment,
        "colsample_bytree": float(winner_colsample),
        "average_rank": float(winner_average_rank)
    },

    "results": results_df.to_dict(
        orient="records"
    )
}

with open(
    summary_file,
    "w"
) as f:
    json.dump(
        summary,
        f,
        indent=4
    )

print("\n" + "=" * 70)
print("RESULTS SAVED")
print("=" * 70)

print(f"\n{results_file}")
print(f"{ranking_file}")
print(f"{summary_file}")

# ======================================================================
# FINAL SAFETY CHECK
# ======================================================================

print("\n" + "=" * 70)
print("FINAL 31.9P-16T-07 SAFETY CHECK")
print("=" * 70)

print("Training data used       : TRUE")
print("Validation data used     : TRUE")
print("Test data loaded         : FALSE")
print("Test data used           : FALSE")
print("Test predictions         : FALSE")
print("Existing model modified  : FALSE")
print("Existing model overwritten: FALSE")

print("\n✓ Only colsample_bytree was varied.")
print("✓ max_depth = 6 remained fixed.")
print("✓ min_child_weight = 10 remained fixed.")
print("✓ learning_rate = 0.05 remained fixed.")
print("✓ subsample = 0.8 remained fixed.")
print("✓ Training/validation data only.")
print("✓ Test data was not loaded.")
print("✓ Existing XGBoost model remains untouched.")
print("✓ Column subsampling experiment completed.")

print("\n" + "=" * 70)
print("STEP 31.9P-16T-07 COMPLETE")
print("=" * 70)

print("\nNEXT:")
print("Send me the complete output before running the next tuning stage.")
print("DO NOT start another tuning experiment yet.")
print("We will inspect which colsample_bytree value is best first.")

STEP 31.9P-16T-07
XGBOOST COLUMNSAMPLING OPTIMIZATION

PROJECT PATHS

Project:
/content/drive/MyDrive/Indhi_Marine_Research_Data

Optimization directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/02_XGBoost/hyperparameter_optimization

Feature list:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing/final_model_feature_columns.csv

Training data:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing/final_model_ready_train.csv

Validation data:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing/final_model_ready_validation.csv

Test data:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing/final_model_ready_test.csv

TEST DATA SAFETY

Test dataset exists on Drive: True

IMPORTANT:
Test data will NOT be loaded.
Test data will NOT be used.
Test predictions will NOT be generated.

REQUIRED FILE CHECK
Feature list: True
Training data: True
Validation data: True

✓ All required training/val

In [6]:
# ======================================================================
# STEP 31.9P-16T-08A
# XGBOOST L1 REGULARIZATION OPTIMIZATION
# ======================================================================

import os
import json
import time
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("=" * 70)
print("STEP 31.9P-16T-08A")
print("XGBOOST L1 REGULARIZATION OPTIMIZATION")
print("=" * 70)

# ======================================================================
# PROJECT PATHS
# ======================================================================

PROJECT_DIR = "/content/drive/MyDrive/Indhi_Marine_Research_Data"

OPT_DIR = os.path.join(
    PROJECT_DIR,
    "06_models",
    "02_XGBoost",
    "hyperparameter_optimization"
)

FEATURE_FILE = os.path.join(
    PROJECT_DIR,
    "05_model_preprocessing",
    "final_model_feature_columns.csv"
)

TRAIN_FILE = os.path.join(
    PROJECT_DIR,
    "05_model_preprocessing",
    "final_model_ready_train.csv"
)

VAL_FILE = os.path.join(
    PROJECT_DIR,
    "05_model_preprocessing",
    "final_model_ready_validation.csv"
)

TEST_FILE = os.path.join(
    PROJECT_DIR,
    "05_model_preprocessing",
    "final_model_ready_test.csv"
)

os.makedirs(OPT_DIR, exist_ok=True)

print("\n" + "=" * 70)
print("PROJECT PATHS")
print("=" * 70)

print("Optimization directory:")
print(OPT_DIR)

print("Feature list:")
print(FEATURE_FILE)

print("Training data:")
print(TRAIN_FILE)

print("Validation data:")
print(VAL_FILE)

print("Test data:")
print(TEST_FILE)

# ======================================================================
# TEST DATA SAFETY
# ======================================================================

print("\n" + "=" * 70)
print("TEST DATA SAFETY")
print("=" * 70)

print("Test dataset exists on Drive:", os.path.exists(TEST_FILE))

print()
print("IMPORTANT:")
print("Test data will NOT be loaded.")
print("Test data will NOT be used.")
print("Test predictions will NOT be generated.")

# ======================================================================
# REQUIRED FILE CHECK
# ======================================================================

required_files = {
    "Feature list": FEATURE_FILE,
    "Training data": TRAIN_FILE,
    "Validation data": VAL_FILE
}

print("\n" + "=" * 70)
print("REQUIRED FILE CHECK")
print("=" * 70)

for name, path in required_files.items():
    exists = os.path.exists(path)
    print(f"{name}: {exists}")
    if not exists:
        raise FileNotFoundError(f"Missing required file: {path}")

print("\n✓ All required training/validation files found.")

# ======================================================================
# LOAD FEATURE LIST
# ======================================================================

print("\n" + "=" * 70)
print("LOADING FINAL FEATURE LIST")
print("=" * 70)

feature_df = pd.read_csv(FEATURE_FILE)

features = feature_df["feature"].tolist()

print("Number of features:", len(features))

if len(features) != 69:
    raise ValueError(
        f"Expected 69 features, but found {len(features)}"
    )

print("✓ Exactly 69 predictors confirmed.")

# ======================================================================
# LOAD TRAINING DATA
# ======================================================================

print("\n" + "=" * 70)
print("LOADING TRAINING DATA")
print("=" * 70)

train_df = pd.read_csv(TRAIN_FILE)

print("Training shape:", train_df.shape)

# ======================================================================
# LOAD VALIDATION DATA
# ======================================================================

print("\n" + "=" * 70)
print("LOADING VALIDATION DATA")
print("=" * 70)

val_df = pd.read_csv(VAL_FILE)

print("Validation shape:", val_df.shape)

# ======================================================================
# TARGET CONFIGURATION
# ======================================================================

TARGET = "chlorophyll_a_log1p"
ORIGINAL_TARGET = "chlorophyll_a"

print("\n" + "=" * 70)
print("TARGET CONFIGURATION")
print("=" * 70)

print("Model target:", TARGET)
print("Original target:", ORIGINAL_TARGET)

# ======================================================================
# COLUMN CHECK
# ======================================================================

missing_train = [f for f in features if f not in train_df.columns]
missing_val = [f for f in features if f not in val_df.columns]

print("\nTraining missing columns:", len(missing_train))
print("Validation missing columns:", len(missing_val))

if missing_train:
    raise ValueError(f"Missing training features: {missing_train}")

if missing_val:
    raise ValueError(f"Missing validation features: {missing_val}")

print("✓ All required columns available.")

# ======================================================================
# PREPARE MODEL INPUTS
# ======================================================================

print("\n" + "=" * 70)
print("PREPARING MODEL INPUTS")
print("=" * 70)

X_train = train_df[features]
y_train = train_df[TARGET]

X_val = val_df[features]
y_val_log = val_df[TARGET]
y_val_original = val_df[ORIGINAL_TARGET]

print("Training X:", X_train.shape)
print("Training y:", y_train.shape)

print("Validation X:", X_val.shape)
print("Validation y:", y_val_log.shape)

# ======================================================================
# DATA SAFETY CHECK
# ======================================================================

print("\n" + "=" * 70)
print("DATA SAFETY CHECK")
print("=" * 70)

checks = {
    "Training predictor NaN": X_train.isna().sum().sum(),
    "Training predictor infinite": np.isinf(X_train.to_numpy()).sum(),
    "Training target NaN": y_train.isna().sum(),
    "Validation predictor NaN": X_val.isna().sum().sum(),
    "Validation predictor infinite": np.isinf(X_val.to_numpy()).sum(),
    "Validation target NaN": y_val_log.isna().sum(),
    "Validation original CHL NaN": y_val_original.isna().sum()
}

for name, value in checks.items():
    print(f"{name:<38}: {value}")

if any(value != 0 for value in checks.values()):
    raise ValueError("Data safety check failed.")

print("\n✓ Training and validation data are clean.")

# ======================================================================
# GPU VERIFICATION
# ======================================================================

print("\n" + "=" * 70)
print("GPU VERIFICATION")
print("=" * 70)

print("XGBoost version:", xgb.__version__)

build_info = xgb.build_info()

print("CUDA enabled:", build_info.get("USE_CUDA"))

try:
    gpu_name = !nvidia-smi --query-gpu=name --format=csv,noheader
    gpu_name = gpu_name[0] if len(gpu_name) > 0 else "Unknown"
except Exception:
    gpu_name = "Unknown"

print("GPU detected:", gpu_name)

if build_info.get("USE_CUDA") is not True:
    raise RuntimeError("CUDA-enabled XGBoost is required.")

print("✓ CUDA-enabled GPU confirmed.")

# ======================================================================
# CURRENT BEST CONFIGURATION
# ======================================================================

BASE_CONFIG = {
    "objective": "reg:squarederror",
    "n_estimators": 2000,
    "max_depth": 6,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 10,
    "reg_alpha": 0.0,
    "reg_lambda": 1.0,
    "random_state": 42,
    "tree_method": "hist",
    "device": "cuda",
    "early_stopping_rounds": 100
}

print("\n" + "=" * 70)
print("CURRENT BEST CONFIGURATION")
print("=" * 70)

for key, value in BASE_CONFIG.items():
    print(f"{key:<24}: {value}")

# ======================================================================
# L1 EXPERIMENT
# ======================================================================

L1_VALUES = [0.0, 0.1, 0.5]

print("\n" + "=" * 70)
print("L1 REGULARIZATION EXPERIMENT")
print("=" * 70)

print()
print("Only reg_alpha will change.")
print("All other parameters remain fixed.")

print()
print("Values to test:")

for value in L1_VALUES:
    print(f"    reg_alpha = {value}")

results = []

# ======================================================================
# RUN EXPERIMENTS
# ======================================================================

for alpha in L1_VALUES:

    experiment_name = f"L1_{str(alpha).replace('.', '_')}"

    print("\n" + "=" * 70)
    print(f"RUNNING: {experiment_name}")
    print("=" * 70)

    config = BASE_CONFIG.copy()
    config["reg_alpha"] = alpha

    print("\nParameters:")

    for key, value in config.items():
        print(f"{key:<24}: {value}")

    # --------------------------------------------------------------
    # CREATE TEMPORARY MODEL
    # --------------------------------------------------------------

    model = xgb.XGBRegressor(**config)

    print("\n✓ Temporary model created.")
    print("✓ Existing XGBoost model remains untouched.")

    # --------------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------------

    start_time = time.time()

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_val, y_val_log)],
        verbose=False
    )

    training_time = time.time() - start_time

    best_iteration = model.best_iteration
    best_trees = best_iteration + 1
    best_rmse = model.best_score

    print("\nTraining completed.")
    print(f"Training time: {training_time:.2f} seconds")
    print(f"Best iteration: {best_iteration}")
    print(f"Best number of trees: {best_trees}")
    print(f"Best validation RMSE: {best_rmse:.12f}")

    # --------------------------------------------------------------
    # VALIDATION PREDICTIONS
    # --------------------------------------------------------------

    pred_log = model.predict(X_val)

    pred_original = np.expm1(pred_log)

    # --------------------------------------------------------------
    # LOG SCALE METRICS
    # --------------------------------------------------------------

    log_mae = mean_absolute_error(y_val_log, pred_log)
    log_rmse = np.sqrt(
        mean_squared_error(y_val_log, pred_log)
    )
    log_r2 = r2_score(y_val_log, pred_log)
    log_bias = np.mean(pred_log - y_val_log)

    # --------------------------------------------------------------
    # ORIGINAL CHL SCALE METRICS
    # --------------------------------------------------------------

    original_mae = mean_absolute_error(
        y_val_original,
        pred_original
    )

    original_rmse = np.sqrt(
        mean_squared_error(
            y_val_original,
            pred_original
        )
    )

    original_r2 = r2_score(
        y_val_original,
        pred_original
    )

    original_bias = np.mean(
        pred_original - y_val_original
    )

    # --------------------------------------------------------------
    # DISPLAY RESULTS
    # --------------------------------------------------------------

    print("\nORIGINAL CHL SCALE")

    print(f"MAE : {original_mae:.6f}")
    print(f"RMSE: {original_rmse:.6f}")
    print(f"R²  : {original_r2:.6f}")
    print(f"Bias: {original_bias:.6f}")

    print("\nLOG1P SCALE")

    print(f"MAE : {log_mae:.6f}")
    print(f"RMSE: {log_rmse:.6f}")
    print(f"R²  : {log_r2:.6f}")
    print(f"Bias: {log_bias:.6f}")

    # --------------------------------------------------------------
    # STORE RESULTS
    # --------------------------------------------------------------

    results.append({
        "experiment": experiment_name,
        "reg_alpha": alpha,
        "reg_lambda": BASE_CONFIG["reg_lambda"],
        "max_depth": BASE_CONFIG["max_depth"],
        "min_child_weight": BASE_CONFIG["min_child_weight"],
        "learning_rate": BASE_CONFIG["learning_rate"],
        "subsample": BASE_CONFIG["subsample"],
        "colsample_bytree": BASE_CONFIG["colsample_bytree"],
        "MAE": original_mae,
        "RMSE": original_rmse,
        "R2": original_r2,
        "Log_MAE": log_mae,
        "Log_RMSE": log_rmse,
        "Log_R2": log_r2,
        "Bias": original_bias,
        "Log_Bias": log_bias,
        "best_iteration": best_iteration,
        "best_trees": best_trees,
        "training_time_seconds": training_time
    })

# ======================================================================
# RESULTS TABLE
# ======================================================================

results_df = pd.DataFrame(results)

print("\n" + "=" * 70)
print("L1 REGULARIZATION RESULTS")
print("=" * 70)

print(
    results_df[
        [
            "experiment",
            "reg_alpha",
            "MAE",
            "RMSE",
            "R2",
            "Log_MAE",
            "Log_RMSE",
            "Log_R2",
            "Bias",
            "Log_Bias",
            "best_iteration"
        ]
    ].to_string(index=False)
)

# ======================================================================
# RANKING
# ======================================================================

ranking_df = results_df.copy()

# Lower is better
ranking_df["MAE_rank"] = ranking_df["MAE"].rank(
    method="min",
    ascending=True
)

ranking_df["RMSE_rank"] = ranking_df["RMSE"].rank(
    method="min",
    ascending=True
)

ranking_df["Log_MAE_rank"] = ranking_df["Log_MAE"].rank(
    method="min",
    ascending=True
)

ranking_df["Log_RMSE_rank"] = ranking_df["Log_RMSE"].rank(
    method="min",
    ascending=True
)

# Higher is better
ranking_df["R2_rank"] = ranking_df["R2"].rank(
    method="min",
    ascending=False
)

ranking_df["Log_R2_rank"] = ranking_df["Log_R2"].rank(
    method="min",
    ascending=False
)

rank_columns = [
    "MAE_rank",
    "RMSE_rank",
    "R2_rank",
    "Log_MAE_rank",
    "Log_RMSE_rank",
    "Log_R2_rank"
]

ranking_df["average_rank"] = ranking_df[rank_columns].mean(axis=1)

ranking_df = ranking_df.sort_values(
    "average_rank"
).reset_index(drop=True)

ranking_df["overall_rank"] = np.arange(
    1,
    len(ranking_df) + 1
)

print("\n" + "=" * 70)
print("L1 REGULARIZATION OVERALL RANKING")
print("=" * 70)

print(
    ranking_df[
        [
            "experiment",
            "reg_alpha",
            "MAE",
            "RMSE",
            "R2",
            "Log_MAE",
            "Log_RMSE",
            "Log_R2",
            "average_rank",
            "overall_rank"
        ]
    ].to_string(index=False)
)

# ======================================================================
# BEST L1 CANDIDATE
# ======================================================================

best_row = ranking_df.iloc[0]

best_alpha = float(best_row["reg_alpha"])

print("\n" + "=" * 70)
print("CURRENT BEST L1 CANDIDATE")
print("=" * 70)

print("Winner:", best_row["experiment"])
print("reg_alpha:", best_alpha)
print(f"Average rank: {best_row['average_rank']:.4f}")

print()
print("IMPORTANT:")
print("This is a validation-based selection only.")
print("The test dataset was not loaded or used.")

# ======================================================================
# SAVE RESULTS
# ======================================================================

results_file = os.path.join(
    OPT_DIR,
    "xgboost_l1_regularization_results.csv"
)

ranking_file = os.path.join(
    OPT_DIR,
    "xgboost_l1_regularization_ranking.csv"
)

summary_file = os.path.join(
    OPT_DIR,
    "xgboost_l1_regularization_summary.json"
)

results_df.to_csv(
    results_file,
    index=False
)

ranking_df.to_csv(
    ranking_file,
    index=False
)

summary = {
    "experiment": "31.9P-16T-08A",
    "optimization_parameter": "reg_alpha",
    "tested_values": L1_VALUES,
    "best_reg_alpha": best_alpha,
    "best_experiment": best_row["experiment"],
    "selection_basis": "validation_only",
    "test_data_loaded": False,
    "test_data_used": False,
    "existing_model_modified": False,
    "base_configuration": BASE_CONFIG
}

with open(summary_file, "w") as f:
    json.dump(summary, f, indent=4)

print("\n" + "=" * 70)
print("RESULTS SAVED")
print("=" * 70)

print(results_file)
print(ranking_file)
print(summary_file)

# ======================================================================
# FINAL SAFETY CHECK
# ======================================================================

print("\n" + "=" * 70)
print("FINAL 31.9P-16T-08A SAFETY CHECK")
print("=" * 70)

print("Training data used       : TRUE")
print("Validation data used     : TRUE")
print("Test data loaded         : FALSE")
print("Test data used           : FALSE")
print("Test predictions         : FALSE")
print("Existing model modified  : FALSE")
print("Existing model overwritten: FALSE")

print()
print("✓ Only reg_alpha was varied.")
print("✓ max_depth = 6 remained fixed.")
print("✓ min_child_weight = 10 remained fixed.")
print("✓ learning_rate = 0.05 remained fixed.")
print("✓ subsample = 0.8 remained fixed.")
print("✓ colsample_bytree = 0.8 remained fixed.")
print("✓ reg_lambda = 1.0 remained fixed.")
print("✓ Training/validation data only.")
print("✓ Test data was not loaded.")
print("✓ Existing XGBoost model remains untouched.")

print("\n" + "=" * 70)
print("STEP 31.9P-16T-08A COMPLETE")
print("=" * 70)

print()
print("NEXT:")
print("Send me the COMPLETE OUTPUT before running 31.9P-16T-08B.")
print("DO NOT start L2 optimization yet.")

STEP 31.9P-16T-08A
XGBOOST L1 REGULARIZATION OPTIMIZATION

PROJECT PATHS
Optimization directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/02_XGBoost/hyperparameter_optimization
Feature list:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing/final_model_feature_columns.csv
Training data:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing/final_model_ready_train.csv
Validation data:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing/final_model_ready_validation.csv
Test data:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing/final_model_ready_test.csv

TEST DATA SAFETY
Test dataset exists on Drive: True

IMPORTANT:
Test data will NOT be loaded.
Test data will NOT be used.
Test predictions will NOT be generated.

REQUIRED FILE CHECK
Feature list: True
Training data: True
Validation data: True

✓ All required training/validation files found.

LOADING FINAL FEATURE LIST
Number of fea

In [7]:
# ======================================================================
# STEP 31.9P-16T-08B
# XGBOOST L2 REGULARIZATION OPTIMIZATION
# ======================================================================

import os
import json
import time
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("=" * 70)
print("STEP 31.9P-16T-08B")
print("XGBOOST L2 REGULARIZATION OPTIMIZATION")
print("=" * 70)

# ======================================================================
# PROJECT PATHS
# ======================================================================

PROJECT_DIR = "/content/drive/MyDrive/Indhi_Marine_Research_Data"

OPT_DIR = os.path.join(
    PROJECT_DIR,
    "06_models",
    "02_XGBoost",
    "hyperparameter_optimization"
)

FEATURE_FILE = os.path.join(
    PROJECT_DIR,
    "05_model_preprocessing",
    "final_model_feature_columns.csv"
)

TRAIN_FILE = os.path.join(
    PROJECT_DIR,
    "05_model_preprocessing",
    "final_model_ready_train.csv"
)

VAL_FILE = os.path.join(
    PROJECT_DIR,
    "05_model_preprocessing",
    "final_model_ready_validation.csv"
)

TEST_FILE = os.path.join(
    PROJECT_DIR,
    "05_model_preprocessing",
    "final_model_ready_test.csv"
)

os.makedirs(OPT_DIR, exist_ok=True)

print("\n" + "=" * 70)
print("PROJECT PATHS")
print("=" * 70)

print("Optimization directory:")
print(OPT_DIR)

print("Feature list:")
print(FEATURE_FILE)

print("Training data:")
print(TRAIN_FILE)

print("Validation data:")
print(VAL_FILE)

print("Test data:")
print(TEST_FILE)

# ======================================================================
# TEST DATA SAFETY
# ======================================================================

print("\n" + "=" * 70)
print("TEST DATA SAFETY")
print("=" * 70)

print("Test dataset exists on Drive:", os.path.exists(TEST_FILE))

print()
print("IMPORTANT:")
print("Test data will NOT be loaded.")
print("Test data will NOT be used.")
print("Test predictions will NOT be generated.")

# ======================================================================
# REQUIRED FILE CHECK
# ======================================================================

required_files = {
    "Feature list": FEATURE_FILE,
    "Training data": TRAIN_FILE,
    "Validation data": VAL_FILE
}

print("\n" + "=" * 70)
print("REQUIRED FILE CHECK")
print("=" * 70)

for name, path in required_files.items():

    exists = os.path.exists(path)

    print(f"{name}: {exists}")

    if not exists:
        raise FileNotFoundError(
            f"Missing required file: {path}"
        )

print("\n✓ All required training/validation files found.")

# ======================================================================
# LOAD FEATURE LIST
# ======================================================================

print("\n" + "=" * 70)
print("LOADING FINAL FEATURE LIST")
print("=" * 70)

feature_df = pd.read_csv(FEATURE_FILE)

features = feature_df["feature"].tolist()

print("Number of features:", len(features))

if len(features) != 69:
    raise ValueError(
        f"Expected 69 features, found {len(features)}"
    )

print("✓ Exactly 69 predictors confirmed.")

# ======================================================================
# LOAD TRAINING DATA
# ======================================================================

print("\n" + "=" * 70)
print("LOADING TRAINING DATA")
print("=" * 70)

train_df = pd.read_csv(TRAIN_FILE)

print("Training shape:", train_df.shape)

# ======================================================================
# LOAD VALIDATION DATA
# ======================================================================

print("\n" + "=" * 70)
print("LOADING VALIDATION DATA")
print("=" * 70)

val_df = pd.read_csv(VAL_FILE)

print("Validation shape:", val_df.shape)

# ======================================================================
# TARGET CONFIGURATION
# ======================================================================

TARGET = "chlorophyll_a_log1p"
ORIGINAL_TARGET = "chlorophyll_a"

print("\n" + "=" * 70)
print("TARGET CONFIGURATION")
print("=" * 70)

print("Model target:", TARGET)
print("Original target:", ORIGINAL_TARGET)

# ======================================================================
# COLUMN CHECK
# ======================================================================

missing_train = [
    f for f in features
    if f not in train_df.columns
]

missing_val = [
    f for f in features
    if f not in val_df.columns
]

print("\nTraining missing columns:", len(missing_train))
print("Validation missing columns:", len(missing_val))

if missing_train:
    raise ValueError(
        f"Missing training features: {missing_train}"
    )

if missing_val:
    raise ValueError(
        f"Missing validation features: {missing_val}"
    )

print("✓ All required columns available.")

# ======================================================================
# PREPARE MODEL INPUTS
# ======================================================================

print("\n" + "=" * 70)
print("PREPARING MODEL INPUTS")
print("=" * 70)

X_train = train_df[features]
y_train = train_df[TARGET]

X_val = val_df[features]
y_val_log = val_df[TARGET]
y_val_original = val_df[ORIGINAL_TARGET]

print("Training X:", X_train.shape)
print("Training y:", y_train.shape)

print("Validation X:", X_val.shape)
print("Validation y:", y_val_log.shape)

# ======================================================================
# DATA SAFETY CHECK
# ======================================================================

print("\n" + "=" * 70)
print("DATA SAFETY CHECK")
print("=" * 70)

checks = {
    "Training predictor NaN":
        X_train.isna().sum().sum(),

    "Training predictor infinite":
        np.isinf(X_train.to_numpy()).sum(),

    "Training target NaN":
        y_train.isna().sum(),

    "Validation predictor NaN":
        X_val.isna().sum().sum(),

    "Validation predictor infinite":
        np.isinf(X_val.to_numpy()).sum(),

    "Validation target NaN":
        y_val_log.isna().sum(),

    "Validation original CHL NaN":
        y_val_original.isna().sum()
}

for name, value in checks.items():

    print(f"{name:<40}: {value}")

if any(value != 0 for value in checks.values()):

    raise ValueError(
        "Training/validation data safety check failed."
    )

print("\n✓ Training and validation data are clean.")

# ======================================================================
# GPU VERIFICATION
# ======================================================================

print("\n" + "=" * 70)
print("GPU VERIFICATION")
print("=" * 70)

print("XGBoost version:", xgb.__version__)

build_info = xgb.build_info()

print(
    "CUDA enabled:",
    build_info.get("USE_CUDA")
)

try:

    gpu_name = !nvidia-smi --query-gpu=name --format=csv,noheader

    gpu_name = (
        gpu_name[0]
        if len(gpu_name) > 0
        else "Unknown"
    )

except Exception:

    gpu_name = "Unknown"

print("GPU detected:", gpu_name)

if build_info.get("USE_CUDA") is not True:

    raise RuntimeError(
        "CUDA-enabled XGBoost is required."
    )

print("✓ CUDA-enabled GPU confirmed.")

# ======================================================================
# CURRENT BEST CONFIGURATION
# ======================================================================

BASE_CONFIG = {

    "objective": "reg:squarederror",

    "n_estimators": 2000,

    "max_depth": 6,

    "learning_rate": 0.05,

    "subsample": 0.8,

    "colsample_bytree": 0.8,

    "min_child_weight": 10,

    "reg_alpha": 0.0,

    "reg_lambda": 1.0,

    "random_state": 42,

    "tree_method": "hist",

    "device": "cuda",

    "early_stopping_rounds": 100
}

print("\n" + "=" * 70)
print("CURRENT BEST CONFIGURATION")
print("=" * 70)

for key, value in BASE_CONFIG.items():

    print(f"{key:<24}: {value}")

# ======================================================================
# L2 EXPERIMENT
# ======================================================================

L2_VALUES = [
    1.0,
    2.0,
    5.0
]

print("\n" + "=" * 70)
print("L2 REGULARIZATION EXPERIMENT")
print("=" * 70)

print()
print("Only reg_lambda will change.")

print("All other parameters remain fixed.")

print()
print("Values to test:")

for value in L2_VALUES:

    print(f"    reg_lambda = {value}")

# ======================================================================
# RESULTS STORAGE
# ======================================================================

results = []

# ======================================================================
# RUN EXPERIMENTS
# ======================================================================

for lambda_value in L2_VALUES:

    experiment_name = (
        f"L2_{str(lambda_value).replace('.', '_')}"
    )

    print("\n" + "=" * 70)
    print(f"RUNNING: {experiment_name}")
    print("=" * 70)

    config = BASE_CONFIG.copy()

    config["reg_lambda"] = lambda_value

    print("\nParameters:")

    for key, value in config.items():

        print(f"{key:<24}: {value}")

    # --------------------------------------------------------------
    # CREATE TEMPORARY MODEL
    # --------------------------------------------------------------

    model = xgb.XGBRegressor(**config)

    print("\n✓ Temporary model created.")

    print(
        "✓ Existing XGBoost model remains untouched."
    )

    # --------------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------------

    start_time = time.time()

    model.fit(
        X_train,
        y_train,
        eval_set=[
            (X_val, y_val_log)
        ],
        verbose=False
    )

    training_time = (
        time.time() - start_time
    )

    best_iteration = model.best_iteration

    best_trees = (
        best_iteration + 1
    )

    best_rmse = model.best_score

    print("\nTraining completed.")

    print(
        f"Training time: "
        f"{training_time:.2f} seconds"
    )

    print(
        f"Best iteration: "
        f"{best_iteration}"
    )

    print(
        f"Best number of trees: "
        f"{best_trees}"
    )

    print(
        f"Best validation RMSE: "
        f"{best_rmse:.12f}"
    )

    # --------------------------------------------------------------
    # VALIDATION PREDICTIONS
    # --------------------------------------------------------------

    pred_log = model.predict(X_val)

    pred_original = np.expm1(
        pred_log
    )

    # --------------------------------------------------------------
    # LOG SCALE METRICS
    # --------------------------------------------------------------

    log_mae = mean_absolute_error(
        y_val_log,
        pred_log
    )

    log_rmse = np.sqrt(
        mean_squared_error(
            y_val_log,
            pred_log
        )
    )

    log_r2 = r2_score(
        y_val_log,
        pred_log
    )

    log_bias = np.mean(
        pred_log - y_val_log
    )

    # --------------------------------------------------------------
    # ORIGINAL CHL SCALE METRICS
    # --------------------------------------------------------------

    original_mae = mean_absolute_error(
        y_val_original,
        pred_original
    )

    original_rmse = np.sqrt(
        mean_squared_error(
            y_val_original,
            pred_original
        )
    )

    original_r2 = r2_score(
        y_val_original,
        pred_original
    )

    original_bias = np.mean(
        pred_original - y_val_original
    )

    # --------------------------------------------------------------
    # DISPLAY RESULTS
    # --------------------------------------------------------------

    print("\nORIGINAL CHL SCALE")

    print(
        f"MAE : {original_mae:.6f}"
    )

    print(
        f"RMSE: {original_rmse:.6f}"
    )

    print(
        f"R²  : {original_r2:.6f}"
    )

    print(
        f"Bias: {original_bias:.6f}"
    )

    print("\nLOG1P SCALE")

    print(
        f"MAE : {log_mae:.6f}"
    )

    print(
        f"RMSE: {log_rmse:.6f}"
    )

    print(
        f"R²  : {log_r2:.6f}"
    )

    print(
        f"Bias: {log_bias:.6f}"
    )

    # --------------------------------------------------------------
    # STORE RESULTS
    # --------------------------------------------------------------

    results.append({

        "experiment":
            experiment_name,

        "reg_alpha":
            BASE_CONFIG["reg_alpha"],

        "reg_lambda":
            lambda_value,

        "max_depth":
            BASE_CONFIG["max_depth"],

        "min_child_weight":
            BASE_CONFIG["min_child_weight"],

        "learning_rate":
            BASE_CONFIG["learning_rate"],

        "subsample":
            BASE_CONFIG["subsample"],

        "colsample_bytree":
            BASE_CONFIG["colsample_bytree"],

        "MAE":
            original_mae,

        "RMSE":
            original_rmse,

        "R2":
            original_r2,

        "Log_MAE":
            log_mae,

        "Log_RMSE":
            log_rmse,

        "Log_R2":
            log_r2,

        "Bias":
            original_bias,

        "Log_Bias":
            log_bias,

        "best_iteration":
            best_iteration,

        "best_trees":
            best_trees,

        "training_time_seconds":
            training_time
    })

# ======================================================================
# RESULTS TABLE
# ======================================================================

results_df = pd.DataFrame(
    results
)

print("\n" + "=" * 70)
print("L2 REGULARIZATION RESULTS")
print("=" * 70)

print(
    results_df[
        [
            "experiment",
            "reg_lambda",
            "MAE",
            "RMSE",
            "R2",
            "Log_MAE",
            "Log_RMSE",
            "Log_R2",
            "Bias",
            "Log_Bias",
            "best_iteration"
        ]
    ].to_string(index=False)
)

# ======================================================================
# RANKING
# ======================================================================

ranking_df = results_df.copy()

# Lower is better
ranking_df["MAE_rank"] = (
    ranking_df["MAE"].rank(
        method="min",
        ascending=True
    )
)

ranking_df["RMSE_rank"] = (
    ranking_df["RMSE"].rank(
        method="min",
        ascending=True
    )
)

ranking_df["Log_MAE_rank"] = (
    ranking_df["Log_MAE"].rank(
        method="min",
        ascending=True
    )
)

ranking_df["Log_RMSE_rank"] = (
    ranking_df["Log_RMSE"].rank(
        method="min",
        ascending=True
    )
)

# Higher is better
ranking_df["R2_rank"] = (
    ranking_df["R2"].rank(
        method="min",
        ascending=False
    )
)

ranking_df["Log_R2_rank"] = (
    ranking_df["Log_R2"].rank(
        method="min",
        ascending=False
    )
)

rank_columns = [
    "MAE_rank",
    "RMSE_rank",
    "R2_rank",
    "Log_MAE_rank",
    "Log_RMSE_rank",
    "Log_R2_rank"
]

ranking_df["average_rank"] = (
    ranking_df[rank_columns].mean(
        axis=1
    )
)

ranking_df = (
    ranking_df
    .sort_values("average_rank")
    .reset_index(drop=True)
)

ranking_df["overall_rank"] = np.arange(
    1,
    len(ranking_df) + 1
)

print("\n" + "=" * 70)
print("L2 REGULARIZATION OVERALL RANKING")
print("=" * 70)

print(
    ranking_df[
        [
            "experiment",
            "reg_lambda",
            "MAE",
            "RMSE",
            "R2",
            "Log_MAE",
            "Log_RMSE",
            "Log_R2",
            "average_rank",
            "overall_rank"
        ]
    ].to_string(index=False)
)

# ======================================================================
# BEST L2 CANDIDATE
# ======================================================================

best_row = ranking_df.iloc[0]

best_lambda = float(
    best_row["reg_lambda"]
)

print("\n" + "=" * 70)
print("CURRENT BEST L2 CANDIDATE")
print("=" * 70)

print(
    "Winner:",
    best_row["experiment"]
)

print(
    "reg_lambda:",
    best_lambda
)

print(
    f"Average rank: "
    f"{best_row['average_rank']:.4f}"
)

print()
print(
    "IMPORTANT:"
)

print(
    "This is a validation-based selection only."
)

print(
    "The test dataset was not loaded or used."
)

# ======================================================================
# SAVE RESULTS
# ======================================================================

results_file = os.path.join(
    OPT_DIR,
    "xgboost_l2_regularization_results.csv"
)

ranking_file = os.path.join(
    OPT_DIR,
    "xgboost_l2_regularization_ranking.csv"
)

summary_file = os.path.join(
    OPT_DIR,
    "xgboost_l2_regularization_summary.json"
)

results_df.to_csv(
    results_file,
    index=False
)

ranking_df.to_csv(
    ranking_file,
    index=False
)

summary = {

    "experiment":
        "31.9P-16T-08B",

    "optimization_parameter":
        "reg_lambda",

    "tested_values":
        L2_VALUES,

    "best_reg_lambda":
        best_lambda,

    "best_experiment":
        best_row["experiment"],

    "selection_basis":
        "validation_only",

    "test_data_loaded":
        False,

    "test_data_used":
        False,

    "existing_model_modified":
        False,

    "base_configuration":
        BASE_CONFIG
}

with open(
    summary_file,
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=4
    )

print("\n" + "=" * 70)
print("RESULTS SAVED")
print("=" * 70)

print(results_file)
print(ranking_file)
print(summary_file)

# ======================================================================
# FINAL SAFETY CHECK
# ======================================================================

print("\n" + "=" * 70)
print("FINAL 31.9P-16T-08B SAFETY CHECK")
print("=" * 70)

print("Training data used        : TRUE")
print("Validation data used      : TRUE")
print("Test data loaded          : FALSE")
print("Test data used            : FALSE")
print("Test predictions          : FALSE")
print("Existing model modified   : FALSE")
print("Existing model overwritten: FALSE")

print()
print("✓ Only reg_lambda was varied.")
print("✓ max_depth = 6 remained fixed.")
print("✓ min_child_weight = 10 remained fixed.")
print("✓ learning_rate = 0.05 remained fixed.")
print("✓ subsample = 0.8 remained fixed.")
print("✓ colsample_bytree = 0.8 remained fixed.")
print("✓ reg_alpha = 0.0 remained fixed.")
print("✓ Training/validation data only.")
print("✓ Test data was not loaded.")
print("✓ Existing XGBoost model remains untouched.")

print("\n" + "=" * 70)
print("STEP 31.9P-16T-08B COMPLETE")
print("=" * 70)

print()
print("NEXT:")
print(
    "Send me the COMPLETE OUTPUT before "
    "we select the final tuned configuration."
)
print("DO NOT run the final test yet.")

STEP 31.9P-16T-08B
XGBOOST L2 REGULARIZATION OPTIMIZATION

PROJECT PATHS
Optimization directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/02_XGBoost/hyperparameter_optimization
Feature list:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing/final_model_feature_columns.csv
Training data:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing/final_model_ready_train.csv
Validation data:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing/final_model_ready_validation.csv
Test data:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing/final_model_ready_test.csv

TEST DATA SAFETY
Test dataset exists on Drive: True

IMPORTANT:
Test data will NOT be loaded.
Test data will NOT be used.
Test predictions will NOT be generated.

REQUIRED FILE CHECK
Feature list: True
Training data: True
Validation data: True

✓ All required training/validation files found.

LOADING FINAL FEATURE LIST
Number of fea

In [9]:
# ======================================================================
# STEP 31.9P-16T-09
# FINAL TUNED-CANDIDATE VALIDATION COMPARISON
# ======================================================================

import os
import json
import time
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("=" * 70)
print("STEP 31.9P-16T-09")
print("FINAL TUNED-CANDIDATE VALIDATION COMPARISON")
print("=" * 70)

# ======================================================================
# 1. PROJECT PATHS
# ======================================================================

PROJECT_DIR = "/content/drive/MyDrive/Indhi_Marine_Research_Data"

OPT_DIR = os.path.join(
    PROJECT_DIR,
    "06_models",
    "02_XGBoost",
    "hyperparameter_optimization"
)

FEATURE_FILE = os.path.join(
    PROJECT_DIR,
    "05_model_preprocessing",
    "final_model_feature_columns.csv"
)

TRAIN_FILE = os.path.join(
    PROJECT_DIR,
    "05_model_preprocessing",
    "final_model_ready_train.csv"
)

VALIDATION_FILE = os.path.join(
    PROJECT_DIR,
    "05_model_preprocessing",
    "final_model_ready_validation.csv"
)

TEST_FILE = os.path.join(
    PROJECT_DIR,
    "05_model_preprocessing",
    "final_model_ready_test.csv"
)

BASELINE_CONFIG_FILE = os.path.join(
    OPT_DIR,
    "xgboost_baseline_configuration.json"
)

OUTPUT_CSV = os.path.join(
    OPT_DIR,
    "xgboost_baseline_vs_tuned_validation_comparison.csv"
)

OUTPUT_JSON = os.path.join(
    OPT_DIR,
    "xgboost_final_tuned_candidate_summary.json"
)

TUNED_CONFIG_CSV = os.path.join(
    OPT_DIR,
    "xgboost_final_tuned_candidate_configuration.csv"
)

print("\n" + "=" * 70)
print("PROJECT PATHS")
print("=" * 70)

print("Project:")
print(PROJECT_DIR)

print("\nOptimization directory:")
print(OPT_DIR)

# Create directory if required
os.makedirs(OPT_DIR, exist_ok=True)

# ======================================================================
# 2. TEST DATA SAFETY
# ======================================================================

print("\n" + "=" * 70)
print("TEST DATA SAFETY")
print("=" * 70)

print("Test dataset exists on Drive:", os.path.exists(TEST_FILE))

print("\nIMPORTANT:")
print("Test data will NOT be loaded.")
print("Test data will NOT be used.")
print("Test predictions will NOT be generated.")

# ======================================================================
# 3. REQUIRED FILE CHECK
# ======================================================================

print("\n" + "=" * 70)
print("REQUIRED FILE CHECK")
print("=" * 70)

required_files = {
    "Feature list": FEATURE_FILE,
    "Training data": TRAIN_FILE,
    "Validation data": VALIDATION_FILE,
    "Baseline configuration": BASELINE_CONFIG_FILE
}

for name, path in required_files.items():
    print(f"{name}: {os.path.exists(path)}")

    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Required file missing: {path}"
        )

print("\n✓ Required files found.")

# ======================================================================
# 4. LOAD FINAL FEATURE LIST
# ======================================================================

print("\n" + "=" * 70)
print("LOADING FINAL FEATURE LIST")
print("=" * 70)

feature_df = pd.read_csv(FEATURE_FILE)

if "feature" not in feature_df.columns:
    raise ValueError(
        "Feature list must contain a 'feature' column."
    )

FEATURES = feature_df["feature"].tolist()

print("Number of features:", len(FEATURES))

if len(FEATURES) != 69:
    raise ValueError(
        f"Expected exactly 69 features, found {len(FEATURES)}."
    )

print("✓ Exactly 69 predictors confirmed.")

# ======================================================================
# 5. LOAD TRAINING DATA
# ======================================================================

print("\n" + "=" * 70)
print("LOADING TRAINING DATA")
print("=" * 70)

train_df = pd.read_csv(TRAIN_FILE)

print("Training shape:", train_df.shape)

# ======================================================================
# 6. LOAD VALIDATION DATA
# ======================================================================

print("\n" + "=" * 70)
print("LOADING VALIDATION DATA")
print("=" * 70)

validation_df = pd.read_csv(VALIDATION_FILE)

print("Validation shape:", validation_df.shape)

# ======================================================================
# 7. TARGET CONFIGURATION
# ======================================================================

TARGET_LOG = "chlorophyll_a_log1p"
TARGET_ORIGINAL = "chlorophyll_a"

print("\n" + "=" * 70)
print("TARGET CONFIGURATION")
print("=" * 70)

print("Model target:", TARGET_LOG)
print("Original target:", TARGET_ORIGINAL)

# ======================================================================
# 8. FEATURE / TARGET CHECK
# ======================================================================

print("\n" + "=" * 70)
print("COLUMN CHECK")
print("=" * 70)

missing_train = [
    f for f in FEATURES
    if f not in train_df.columns
]

missing_validation = [
    f for f in FEATURES
    if f not in validation_df.columns
]

print("Training missing features:", len(missing_train))
print("Validation missing features:", len(missing_validation))

if missing_train:
    print(missing_train)
    raise ValueError("Training feature mismatch.")

if missing_validation:
    print(missing_validation)
    raise ValueError("Validation feature mismatch.")

for target in [TARGET_LOG, TARGET_ORIGINAL]:

    if target not in train_df.columns:
        raise ValueError(
            f"Training target missing: {target}"
        )

    if target not in validation_df.columns:
        raise ValueError(
            f"Validation target missing: {target}"
        )

print("✓ All 69 features and targets available.")

# ======================================================================
# 9. PREPARE INPUTS
# ======================================================================

print("\n" + "=" * 70)
print("PREPARING MODEL INPUTS")
print("=" * 70)

X_train = train_df[FEATURES]
y_train = train_df[TARGET_LOG]

X_validation = validation_df[FEATURES]
y_validation_log = validation_df[TARGET_LOG]
y_validation_original = validation_df[TARGET_ORIGINAL]

print("Training X:", X_train.shape)
print("Training y:", y_train.shape)

print("Validation X:", X_validation.shape)
print("Validation y:", y_validation_log.shape)

# ======================================================================
# 10. DATA SAFETY CHECK
# ======================================================================

print("\n" + "=" * 70)
print("DATA SAFETY CHECK")
print("=" * 70)

checks = {
    "Training predictor NaN":
        X_train.isna().sum().sum(),

    "Training predictor infinite":
        np.isinf(X_train.to_numpy()).sum(),

    "Training target NaN":
        y_train.isna().sum(),

    "Validation predictor NaN":
        X_validation.isna().sum().sum(),

    "Validation predictor infinite":
        np.isinf(X_validation.to_numpy()).sum(),

    "Validation log target NaN":
        y_validation_log.isna().sum(),

    "Validation original CHL NaN":
        y_validation_original.isna().sum()
}

for name, value in checks.items():
    print(f"{name:35s}: {value}")

if any(value != 0 for value in checks.values()):
    raise ValueError(
        "Data safety check failed."
    )

print("\n✓ Training and validation data are clean.")

# ======================================================================
# 11. LOCKED ORIGINAL BASELINE
# ======================================================================

print("\n" + "=" * 70)
print("LOCKED ORIGINAL XGBOOST BASELINE")
print("=" * 70)

# These are the established validation results from the existing
# early-stopped XGBoost model.
#
# IMPORTANT:
# We are NOT using the test result.
# We are NOT retraining the original model.
# We are using the already established validation evidence.

baseline_metrics = {
    "MAE": 0.1894004434498657,
    "RMSE": 0.8060904960285136,
    "R2": 0.5371393487829845,
    "Log_MAE": 0.06996042122254419,
    "Log_RMSE": 0.15257568941934896,
    "Log_R2": 0.8382274310440981,
    "Bias": -0.07618218652497848,
    "Log_Bias": -0.010555
}

print("Original baseline validation:")
print(f"MAE      : {baseline_metrics['MAE']:.6f}")
print(f"RMSE     : {baseline_metrics['RMSE']:.6f}")
print(f"R²       : {baseline_metrics['R2']:.6f}")

print("\nLog1p baseline validation:")
print(f"Log MAE  : {baseline_metrics['Log_MAE']:.6f}")
print(f"Log RMSE : {baseline_metrics['Log_RMSE']:.6f}")
print(f"Log R²   : {baseline_metrics['Log_R2']:.6f}")

print("\n✓ Original baseline remains locked.")

# ======================================================================
# 12. CURRENT TUNED CONFIGURATION
# ======================================================================

print("\n" + "=" * 70)
print("CURRENT TUNED CANDIDATE CONFIGURATION")
print("=" * 70)

TUNED_PARAMS = {
    "objective": "reg:squarederror",
    "n_estimators": 2000,
    "max_depth": 6,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 10,
    "reg_alpha": 0.0,
    "reg_lambda": 5.0,
    "random_state": 42,
    "tree_method": "hist",
    "device": "cuda",
    "early_stopping_rounds": 100
}

for key, value in TUNED_PARAMS.items():
    print(f"{key:25s}: {value}")

print("\n✓ Current tuned candidate defined.")

# ======================================================================
# 13. SAVE TUNED CONFIGURATION
# ======================================================================

tuned_config_df = pd.DataFrame(
    [TUNED_PARAMS]
)

tuned_config_df.to_csv(
    TUNED_CONFIG_CSV,
    index=False
)

print("\nSaved tuned configuration:")
print(TUNED_CONFIG_CSV)

# ======================================================================
# 14. BUILD TEMPORARY TUNED MODEL
# ======================================================================

print("\n" + "=" * 70)
print("BUILDING TEMPORARY TUNED XGBOOST MODEL")
print("=" * 70)

tuned_model = xgb.XGBRegressor(
    objective=TUNED_PARAMS["objective"],
    n_estimators=TUNED_PARAMS["n_estimators"],
    max_depth=TUNED_PARAMS["max_depth"],
    learning_rate=TUNED_PARAMS["learning_rate"],
    subsample=TUNED_PARAMS["subsample"],
    colsample_bytree=TUNED_PARAMS["colsample_bytree"],
    min_child_weight=TUNED_PARAMS["min_child_weight"],
    reg_alpha=TUNED_PARAMS["reg_alpha"],
    reg_lambda=TUNED_PARAMS["reg_lambda"],
    random_state=TUNED_PARAMS["random_state"],
    tree_method=TUNED_PARAMS["tree_method"],
    device=TUNED_PARAMS["device"],
    early_stopping_rounds=TUNED_PARAMS["early_stopping_rounds"]
)

print("✓ Temporary tuned model created.")
print("✓ Existing original XGBoost model remains untouched.")

# ======================================================================
# 15. TRAIN TUNED CANDIDATE
# ======================================================================

print("\n" + "=" * 70)
print("TRAINING TUNED XGBOOST CANDIDATE")
print("=" * 70)

print("Training data      : Training split only")
print("Validation data    : Validation split only")
print("Test data          : NOT USED")
print("GPU                : CUDA")
print("Early stopping     : 100 rounds")
print("Maximum trees      : 2000")

start_time = time.time()

tuned_model.fit(
    X_train,
    y_train,
    eval_set=[
        (X_train, y_train),
        (X_validation, y_validation_log)
    ],
    verbose=100
)

training_time = time.time() - start_time

print("\n✓ Tuned candidate training completed.")
print(f"Training time: {training_time:.2f} seconds")

# ======================================================================
# 16. BEST ITERATION
# ======================================================================

best_iteration = int(tuned_model.best_iteration)
best_trees = best_iteration + 1
best_validation_rmse = float(
    tuned_model.best_score
)

print("\n" + "=" * 70)
print("TUNED CANDIDATE BEST ITERATION")
print("=" * 70)

print("Best iteration:", best_iteration)
print("Best number of trees:", best_trees)
print("Best validation RMSE (log1p):",
      f"{best_validation_rmse:.12f}")

# ======================================================================
# 17. GENERATE VALIDATION PREDICTIONS
# ======================================================================

print("\n" + "=" * 70)
print("GENERATING TUNED VALIDATION PREDICTIONS")
print("=" * 70)

pred_log = tuned_model.predict(
    X_validation
)

pred_original = np.expm1(pred_log)

print("Prediction shape:", pred_log.shape)

# ======================================================================
# 18. PREDICTION SAFETY
# ======================================================================

if np.isnan(pred_log).any():
    raise ValueError("NaN detected in log predictions.")

if np.isinf(pred_log).any():
    raise ValueError("Infinite value detected in log predictions.")

if np.isnan(pred_original).any():
    raise ValueError("NaN detected in original predictions.")

if np.isinf(pred_original).any():
    raise ValueError("Infinite value detected in original predictions.")

print("✓ Prediction safety checks passed.")

# ======================================================================
# 19. CALCULATE TUNED METRICS
# ======================================================================

# --------------------------
# Original CHL scale
# --------------------------

tuned_mae = mean_absolute_error(
    y_validation_original,
    pred_original
)

tuned_rmse = np.sqrt(
    mean_squared_error(
        y_validation_original,
        pred_original
    )
)

tuned_r2 = r2_score(
    y_validation_original,
    pred_original
)

tuned_bias = np.mean(
    pred_original - y_validation_original
)

# --------------------------
# Log1p scale
# --------------------------

tuned_log_mae = mean_absolute_error(
    y_validation_log,
    pred_log
)

tuned_log_rmse = np.sqrt(
    mean_squared_error(
        y_validation_log,
        pred_log
    )
)

tuned_log_r2 = r2_score(
    y_validation_log,
    pred_log
)

tuned_log_bias = np.mean(
    pred_log - y_validation_log
)

# ======================================================================
# 20. DISPLAY TUNED RESULTS
# ======================================================================

print("\n" + "=" * 70)
print("TUNED CANDIDATE VALIDATION RESULTS")
print("=" * 70)

print("\nORIGINAL CHL SCALE")
print(f"MAE       : {tuned_mae:.6f}")
print(f"RMSE      : {tuned_rmse:.6f}")
print(f"R²        : {tuned_r2:.6f}")
print(f"Bias      : {tuned_bias:.6f}")

print("\nLOG1P SCALE")
print(f"MAE       : {tuned_log_mae:.6f}")
print(f"RMSE      : {tuned_log_rmse:.6f}")
print(f"R²        : {tuned_log_r2:.6f}")
print(f"Bias      : {tuned_log_bias:.6f}")

# ======================================================================
# 21. BUILD COMPARISON TABLE
# ======================================================================

comparison = pd.DataFrame({
    "metric": [
        "MAE",
        "RMSE",
        "R2",
        "Log_MAE",
        "Log_RMSE",
        "Log_R2",
        "Bias",
        "Log_Bias"
    ],

    "Original_XGBoost": [
        baseline_metrics["MAE"],
        baseline_metrics["RMSE"],
        baseline_metrics["R2"],
        baseline_metrics["Log_MAE"],
        baseline_metrics["Log_RMSE"],
        baseline_metrics["Log_R2"],
        baseline_metrics["Bias"],
        baseline_metrics["Log_Bias"]
    ],

    "Tuned_XGBoost": [
        tuned_mae,
        tuned_rmse,
        tuned_r2,
        tuned_log_mae,
        tuned_log_rmse,
        tuned_log_r2,
        tuned_bias,
        tuned_log_bias
    ]
})

# ======================================================================
# 22. CALCULATE IMPROVEMENT
# ======================================================================

# For error metrics:
# positive improvement = error decreased

comparison["improvement_percent"] = np.nan

error_metrics = [
    "MAE",
    "RMSE",
    "Log_MAE",
    "Log_RMSE"
]

higher_better_metrics = [
    "R2",
    "Log_R2"
]

for idx, row in comparison.iterrows():

    metric = row["metric"]

    baseline = row["Original_XGBoost"]
    tuned = row["Tuned_XGBoost"]

    if metric in error_metrics:

        comparison.loc[
            idx,
            "improvement_percent"
        ] = (
            (baseline - tuned) / abs(baseline)
        ) * 100

    elif metric in higher_better_metrics:

        comparison.loc[
            idx,
            "improvement_percent"
        ] = (
            (tuned - baseline) / abs(baseline)
        ) * 100

# ======================================================================
# 23. DETERMINE METRIC WINNERS
# ======================================================================

comparison["winner"] = ""

for idx, row in comparison.iterrows():

    metric = row["metric"]

    baseline = row["Original_XGBoost"]
    tuned = row["Tuned_XGBoost"]

    if metric in error_metrics:

        if tuned < baseline:
            winner = "Tuned XGBoost"
        elif tuned > baseline:
            winner = "Original XGBoost"
        else:
            winner = "Tie"

    elif metric in higher_better_metrics:

        if tuned > baseline:
            winner = "Tuned XGBoost"
        elif tuned < baseline:
            winner = "Original XGBoost"
        else:
            winner = "Tie"

    else:
        # Bias is interpreted separately.
        winner = "Compare magnitude"

    comparison.loc[idx, "winner"] = winner

# ======================================================================
# 24. DISPLAY COMPARISON
# ======================================================================

print("\n" + "=" * 70)
print("ORIGINAL vs TUNED XGBOOST VALIDATION COMPARISON")
print("=" * 70)

print(
    comparison.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

# ======================================================================
# 25. PRIMARY METRIC COMPARISON
# ======================================================================

print("\n" + "=" * 70)
print("PRIMARY VALIDATION METRICS")
print("=" * 70)

primary = comparison[
    comparison["metric"].isin(
        ["MAE", "RMSE", "R2"]
    )
].copy()

print(
    primary.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

# ======================================================================
# 26. COUNT PRIMARY METRIC WINS
# ======================================================================

primary_tuned_wins = (
    primary["winner"] == "Tuned XGBoost"
).sum()

primary_original_wins = (
    primary["winner"] == "Original XGBoost"
).sum()

print("\nTuned XGBoost primary metric wins:",
      primary_tuned_wins)

print("Original XGBoost primary metric wins:",
      primary_original_wins)

# ======================================================================
# 27. LOG-SCALE COMPARISON
# ======================================================================

print("\n" + "=" * 70)
print("LOG1P VALIDATION METRICS")
print("=" * 70)

log_comparison = comparison[
    comparison["metric"].isin(
        ["Log_MAE", "Log_RMSE", "Log_R2"]
    )
].copy()

print(
    log_comparison.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

# ======================================================================
# 28. FINAL CANDIDATE DECISION
# ======================================================================

print("\n" + "=" * 70)
print("TUNED CANDIDATE DECISION")
print("=" * 70)

if primary_tuned_wins > primary_original_wins:

    final_decision = "TUNED_XGBOOST"

    print(
        "🏆 TUNED XGBOOST IS THE CURRENT VALIDATION CANDIDATE."
    )

    print(
        "\nThe tuned configuration outperformed the original "
        "configuration on more primary validation metrics."
    )

elif primary_original_wins > primary_tuned_wins:

    final_decision = "ORIGINAL_XGBOOST"

    print(
        "🏆 ORIGINAL XGBOOST REMAINS THE VALIDATION CANDIDATE."
    )

    print(
        "\nThe tuned configuration did not provide sufficient "
        "improvement over the original configuration."
    )

else:

    final_decision = "REVIEW_REQUIRED"

    print(
        "⚠ PRIMARY METRIC RESULTS ARE TIED."
    )

    print(
        "Additional scientific review is required before "
        "selecting the final candidate."
    )

# ======================================================================
# 29. SAVE RESULTS
# ======================================================================

print("\n" + "=" * 70)
print("SAVING FINAL TUNED-CANDIDATE COMPARISON")
print("=" * 70)

comparison.to_csv(
    OUTPUT_CSV,
    index=False
)

print("\nSaved comparison:")
print(OUTPUT_CSV)

summary = {
    "step": "31.9P-16T-09",

    "purpose":
        "Final validation comparison between original "
        "and tuned XGBoost candidates",

    "test_data_loaded": False,
    "test_data_used": False,
    "test_predictions_generated": False,

    "original_xgboost": {
        "MAE": baseline_metrics["MAE"],
        "RMSE": baseline_metrics["RMSE"],
        "R2": baseline_metrics["R2"],
        "Log_MAE": baseline_metrics["Log_MAE"],
        "Log_RMSE": baseline_metrics["Log_RMSE"],
        "Log_R2": baseline_metrics["Log_R2"],
        "Bias": baseline_metrics["Bias"],
        "Log_Bias": baseline_metrics["Log_Bias"]
    },

    "tuned_xgboost": {
        "MAE": float(tuned_mae),
        "RMSE": float(tuned_rmse),
        "R2": float(tuned_r2),
        "Log_MAE": float(tuned_log_mae),
        "Log_RMSE": float(tuned_log_rmse),
        "Log_R2": float(tuned_log_r2),
        "Bias": float(tuned_bias),
        "Log_Bias": float(tuned_log_bias),
        "best_iteration": best_iteration,
        "best_number_of_trees": best_trees,
        "training_time_seconds": training_time
    },

    "tuned_parameters": TUNED_PARAMS,

    "primary_metric_wins": {
        "tuned_xgboost": int(primary_tuned_wins),
        "original_xgboost": int(primary_original_wins)
    },

    "final_validation_decision": final_decision
}

with open(
    OUTPUT_JSON,
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=4
    )

print("Saved summary:")
print(OUTPUT_JSON)

# ======================================================================
# 30. FINAL SAFETY CHECK
# ======================================================================

print("\n" + "=" * 70)
print("FINAL 31.9P-16T-09 SAFETY CHECK")
print("=" * 70)

print("Training data used        : TRUE")
print("Validation data used      : TRUE")
print("Test data loaded          : FALSE")
print("Test data used            : FALSE")
print("Test predictions          : FALSE")
print("Original model modified   : FALSE")
print("Original model overwritten : FALSE")
print("Hyperparameter search     : COMPLETE")
print("Current candidate         : TUNED XGBOOST")

print("\n✓ Original XGBoost remains untouched.")
print("✓ Tuned candidate was trained separately.")
print("✓ Only training + validation data were used.")
print("✓ Test data was not loaded.")
print("✓ No test predictions were generated.")
print("✓ Validation comparison completed.")

print("\n" + "=" * 70)
print("STEP 31.9P-16T-09 COMPLETE")
print("=" * 70)

print("\nFinal validation decision:")
print(final_decision)

print("\nNEXT:")
print(
    "Send me the COMPLETE OUTPUT before we perform "
    "any final test evaluation."
)

STEP 31.9P-16T-09
FINAL TUNED-CANDIDATE VALIDATION COMPARISON

PROJECT PATHS
Project:
/content/drive/MyDrive/Indhi_Marine_Research_Data

Optimization directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/02_XGBoost/hyperparameter_optimization

TEST DATA SAFETY
Test dataset exists on Drive: True

IMPORTANT:
Test data will NOT be loaded.
Test data will NOT be used.
Test predictions will NOT be generated.

REQUIRED FILE CHECK
Feature list: True
Training data: True
Validation data: True
Baseline configuration: True

✓ Required files found.

LOADING FINAL FEATURE LIST
Number of features: 69
✓ Exactly 69 predictors confirmed.

LOADING TRAINING DATA
Training shape: (736950, 75)

LOADING VALIDATION DATA
Validation shape: (157896, 75)

TARGET CONFIGURATION
Model target: chlorophyll_a_log1p
Original target: chlorophyll_a

COLUMN CHECK
Training missing features: 0
Validation missing features: 0
✓ All 69 features and targets available.

PREPARING MODEL INPUTS
Training X: (736950, 

In [10]:
# ================================================================
# STEP 31.9P-16T-10
# TUNED XGBOOST CANDIDATE FINALIZATION
# ================================================================

import os
import json
import shutil
import pandas as pd
import numpy as np

print("=" * 70)
print("STEP 31.9P-16T-10")
print("TUNED XGBOOST CANDIDATE FINALIZATION")
print("=" * 70)

# ================================================================
# PROJECT PATHS
# ================================================================

PROJECT_DIR = "/content/drive/MyDrive/Indhi_Marine_Research_Data"

OPT_DIR = os.path.join(
    PROJECT_DIR,
    "06_models",
    "02_XGBoost",
    "hyperparameter_optimization"
)

FINAL_CANDIDATE_DIR = os.path.join(
    OPT_DIR,
    "final_tuned_candidate"
)

FEATURE_FILE = os.path.join(
    PROJECT_DIR,
    "05_model_preprocessing",
    "final_model_feature_columns.csv"
)

TEST_FILE = os.path.join(
    PROJECT_DIR,
    "05_model_preprocessing",
    "final_model_ready_test.csv"
)

os.makedirs(FINAL_CANDIDATE_DIR, exist_ok=True)

print("\n" + "=" * 70)
print("PROJECT PATHS")
print("=" * 70)

print(f"Project:")
print(PROJECT_DIR)

print(f"\nOptimization directory:")
print(OPT_DIR)

print(f"\nFinal tuned candidate directory:")
print(FINAL_CANDIDATE_DIR)

# ================================================================
# TEST DATA SAFETY
# ================================================================

print("\n" + "=" * 70)
print("TEST DATA SAFETY")
print("=" * 70)

print(f"Test dataset exists on Drive: {os.path.exists(TEST_FILE)}")
print()
print("IMPORTANT:")
print("Test dataset will NOT be loaded.")
print("Test dataset will NOT be used.")
print("Test predictions will NOT be generated.")

# ================================================================
# REQUIRED FILE CHECK
# ================================================================

print("\n" + "=" * 70)
print("REQUIRED FILE CHECK")
print("=" * 70)

print(f"Feature list: {os.path.exists(FEATURE_FILE)}")

if not os.path.exists(FEATURE_FILE):
    raise FileNotFoundError(
        "Final model feature list was not found."
    )

# ================================================================
# LOAD FEATURE LIST
# ================================================================

print("\n" + "=" * 70)
print("LOADING FINAL FEATURE LIST")
print("=" * 70)

feature_df = pd.read_csv(FEATURE_FILE)

FEATURES = feature_df["feature"].tolist()

print(f"Number of features: {len(FEATURES)}")

if len(FEATURES) != 69:
    raise ValueError(
        f"Expected exactly 69 features, found {len(FEATURES)}."
    )

print("✓ Exactly 69 final predictors confirmed.")

# ================================================================
# EXPECTED TUNED CONFIGURATION
# ================================================================

TUNED_CONFIG = {
    "model": "XGBoost",
    "objective": "reg:squarederror",
    "n_estimators": 2000,
    "max_depth": 6,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 10,
    "reg_alpha": 0.0,
    "reg_lambda": 5.0,
    "random_state": 42,
    "tree_method": "hist",
    "device": "cuda",
    "early_stopping_rounds": 100,
    "target": "chlorophyll_a_log1p",
    "original_target": "chlorophyll_a",
    "number_of_features": 69,
    "best_iteration": 70,
    "best_number_of_trees": 71
}

print("\n" + "=" * 70)
print("LOCKED TUNED CANDIDATE CONFIGURATION")
print("=" * 70)

for key, value in TUNED_CONFIG.items():
    print(f"{key:25s}: {value}")

# ================================================================
# VALIDATION METRICS FROM STEP 16T-09
# ================================================================

VALIDATION_METRICS = {
    "Original_CHL_MAE": 0.183277,
    "Original_CHL_RMSE": 0.781189,
    "Original_CHL_R2": 0.565295,
    "Original_CHL_Bias": -0.066845,
    "Log1p_MAE": 0.067634,
    "Log1p_RMSE": 0.148240,
    "Log1p_R2": 0.847291,
    "Log1p_Bias": -0.008380,
    "best_iteration": 70,
    "best_number_of_trees": 71
}

print("\n" + "=" * 70)
print("VALIDATION PERFORMANCE TO BE PRESERVED")
print("=" * 70)

print(f"Original CHL MAE  : {VALIDATION_METRICS['Original_CHL_MAE']:.6f}")
print(f"Original CHL RMSE : {VALIDATION_METRICS['Original_CHL_RMSE']:.6f}")
print(f"Original CHL R²   : {VALIDATION_METRICS['Original_CHL_R2']:.6f}")

print(f"\nLog1p MAE         : {VALIDATION_METRICS['Log1p_MAE']:.6f}")
print(f"Log1p RMSE        : {VALIDATION_METRICS['Log1p_RMSE']:.6f}")
print(f"Log1p R²          : {VALIDATION_METRICS['Log1p_R2']:.6f}")

# ================================================================
# CHECK WHETHER TUNED MODEL EXISTS IN MEMORY
# ================================================================

print("\n" + "=" * 70)
print("TUNED MODEL OBJECT CHECK")
print("=" * 70)

candidate_variable_names = [
    "tuned_model",
    "best_model",
    "final_tuned_model",
    "candidate_model"
]

found_model = None
found_name = None

for name in candidate_variable_names:
    if name in globals():
        obj = globals()[name]

        # Basic XGBoost model check
        if hasattr(obj, "save_model") and hasattr(obj, "predict"):
            found_model = obj
            found_name = name
            break

if found_model is None:

    print("No tuned XGBoost model object was found in memory.")
    print()
    print("IMPORTANT:")
    print("The candidate must NOT be retrained automatically.")
    print()
    print("Please check whether STEP 31.9P-16T-09 saved")
    print("the tuned model file separately.")
    print()
    print("Candidate finalization STOPPED safely.")

    raise RuntimeError(
        "Tuned model object not found in memory. "
        "No retraining performed. "
        "Run the model-saving/recovery step before finalization."
    )

print(f"✓ Tuned model object found: {found_name}")

# ================================================================
# VERIFY MODEL FEATURE COUNT
# ================================================================

print("\n" + "=" * 70)
print("MODEL FEATURE VERIFICATION")
print("=" * 70)

booster = found_model.get_booster()

model_features = booster.feature_names

if model_features is None:
    raise ValueError(
        "Tuned model does not contain stored feature names."
    )

print(f"Stored model features: {len(model_features)}")

if len(model_features) != 69:
    raise ValueError(
        f"Expected 69 model features, found {len(model_features)}."
    )

if model_features != FEATURES:
    raise ValueError(
        "Model feature names/order do not exactly match "
        "final_model_feature_columns.csv."
    )

print("✓ Model contains exactly 69 features.")
print("✓ Feature names match.")
print("✓ Feature order matches.")

# ================================================================
# VERIFY BEST ITERATION
# ================================================================

print("\n" + "=" * 70)
print("BEST ITERATION VERIFICATION")
print("=" * 70)

actual_best_iteration = getattr(
    found_model,
    "best_iteration",
    None
)

actual_best_score = getattr(
    found_model,
    "best_score",
    None
)

print(f"Model best iteration: {actual_best_iteration}")
print(f"Model best score    : {actual_best_score}")

if actual_best_iteration is not None:

    if int(actual_best_iteration) != 70:
        raise ValueError(
            "Best iteration does not match the Step 16T-09 "
            "tuned candidate (expected 70)."
        )

    print("✓ Best iteration = 70 confirmed.")

else:
    print(
        "WARNING: best_iteration is unavailable from the model object."
    )

# ================================================================
# VERIFY CONFIGURATION
# ================================================================

print("\n" + "=" * 70)
print("MODEL CONFIGURATION VERIFICATION")
print("=" * 70)

params = found_model.get_params()

checks = {
    "max_depth": 6,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 10,
    "reg_alpha": 0.0,
    "reg_lambda": 5.0,
    "random_state": 42,
    "n_estimators": 2000
}

configuration_ok = True

for parameter, expected in checks.items():

    actual = params.get(parameter)

    print(
        f"{parameter:25s}: "
        f"actual={actual} | expected={expected}"
    )

    if actual != expected:
        configuration_ok = False

if not configuration_ok:
    raise ValueError(
        "Tuned model configuration does not exactly match "
        "the locked 16T-09 candidate."
    )

print("\n✓ Tuned model configuration exactly verified.")

# ================================================================
# SAVE MODEL
# ================================================================

print("\n" + "=" * 70)
print("SAVING FINAL TUNED CANDIDATE")
print("=" * 70)

FINAL_MODEL_FILE = os.path.join(
    FINAL_CANDIDATE_DIR,
    "xgboost_final_tuned_candidate.json"
)

found_model.save_model(FINAL_MODEL_FILE)

print(f"Saved:")
print(FINAL_MODEL_FILE)

if not os.path.exists(FINAL_MODEL_FILE):
    raise RuntimeError(
        "Final tuned candidate model was not saved successfully."
    )

print("✓ Tuned candidate model saved.")

# ================================================================
# SAVE CONFIGURATION
# ================================================================

CONFIG_FILE = os.path.join(
    FINAL_CANDIDATE_DIR,
    "xgboost_final_tuned_candidate_configuration.json"
)

with open(CONFIG_FILE, "w") as f:
    json.dump(
        TUNED_CONFIG,
        f,
        indent=4
    )

print("\nSaved configuration:")
print(CONFIG_FILE)

# ================================================================
# SAVE VALIDATION METRICS
# ================================================================

METRICS_FILE = os.path.join(
    FINAL_CANDIDATE_DIR,
    "xgboost_final_tuned_candidate_validation_metrics.json"
)

with open(METRICS_FILE, "w") as f:
    json.dump(
        VALIDATION_METRICS,
        f,
        indent=4
    )

print("\nSaved validation metrics:")
print(METRICS_FILE)

# ================================================================
# SAVE FEATURE LIST
# ================================================================

FINAL_FEATURE_FILE = os.path.join(
    FINAL_CANDIDATE_DIR,
    "xgboost_final_tuned_candidate_features.csv"
)

feature_df.to_csv(
    FINAL_FEATURE_FILE,
    index=False
)

print("\nSaved feature list:")
print(FINAL_FEATURE_FILE)

# ================================================================
# CREATE FINAL CANDIDATE SUMMARY
# ================================================================

SUMMARY = {
    "step": "31.9P-16T-10",
    "status": "FINAL_TUNED_CANDIDATE",
    "model": "XGBoost",
    "model_file": FINAL_MODEL_FILE,
    "configuration_file": CONFIG_FILE,
    "validation_metrics_file": METRICS_FILE,
    "feature_file": FINAL_FEATURE_FILE,
    "test_data_loaded": False,
    "test_data_used": False,
    "test_predictions_generated": False,
    "retraining_performed": False,
    "hyperparameters_changed": False,
    "original_model_modified": False,
    "number_of_features": 69,
    "best_iteration": 70,
    "best_number_of_trees": 71,
    "validation_metrics": VALIDATION_METRICS
}

SUMMARY_FILE = os.path.join(
    FINAL_CANDIDATE_DIR,
    "xgboost_final_tuned_candidate_summary.json"
)

with open(SUMMARY_FILE, "w") as f:
    json.dump(
        SUMMARY,
        f,
        indent=4
    )

print("\nSaved final candidate summary:")
print(SUMMARY_FILE)

# ================================================================
# FINAL SAFETY CHECK
# ================================================================

print("\n" + "=" * 70)
print("FINAL 31.9P-16T-10 SAFETY CHECK")
print("=" * 70)

print("Training data used during this step : FALSE")
print("Validation data loaded              : FALSE")
print("Test data loaded                    : FALSE")
print("Test data used                      : FALSE")
print("Test predictions generated          : FALSE")
print("Model retraining                    : FALSE")
print("Hyperparameters changed             : FALSE")
print("Original model modified             : FALSE")
print("Tuned candidate saved               : TRUE")

print()
print("✓ Final tuned candidate preserved.")
print("✓ Exact 69-feature structure verified.")
print("✓ Exact tuned configuration verified.")
print("✓ Best iteration verified.")
print("✓ Validation evidence preserved.")
print("✓ Original XGBoost remains untouched.")
print("✓ Test dataset was NOT loaded.")
print("✓ Test predictions were NOT generated.")

print("\n" + "=" * 70)
print("STEP 31.9P-16T-10 COMPLETE")
print("=" * 70)

print()
print("FINAL TUNED CANDIDATE:")
print("XGBoost")

print()
print("Configuration:")
print("max_depth       = 6")
print("learning_rate   = 0.05")
print("min_child_weight= 10")
print("subsample        = 0.8")
print("colsample       = 0.8")
print("reg_alpha       = 0.0")
print("reg_lambda      = 5.0")

print()
print("Validation:")
print("MAE  = 0.183277 mg/m³")
print("RMSE = 0.781189 mg/m³")
print("R²   = 0.565295")

print()
print("NEXT STEP:")
print("31.9P-16T-11 — Final Tuned XGBoost Validation Verification")

STEP 31.9P-16T-10
TUNED XGBOOST CANDIDATE FINALIZATION

PROJECT PATHS
Project:
/content/drive/MyDrive/Indhi_Marine_Research_Data

Optimization directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/02_XGBoost/hyperparameter_optimization

Final tuned candidate directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/02_XGBoost/hyperparameter_optimization/final_tuned_candidate

TEST DATA SAFETY
Test dataset exists on Drive: True

IMPORTANT:
Test dataset will NOT be loaded.
Test dataset will NOT be used.
Test predictions will NOT be generated.

REQUIRED FILE CHECK
Feature list: True

LOADING FINAL FEATURE LIST
Number of features: 69
✓ Exactly 69 final predictors confirmed.

LOCKED TUNED CANDIDATE CONFIGURATION
model                    : XGBoost
objective                : reg:squarederror
n_estimators             : 2000
max_depth                : 6
learning_rate            : 0.05
subsample                : 0.8
colsample_bytree         : 0.8
min_child_weight   

In [14]:
# ================================================================
# STEP 31.9P-16T-11-FIX
# FINAL SAVED TUNED XGBOOST VALIDATION VERIFICATION
# ================================================================

import os
import json
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("=" * 70)
print("STEP 31.9P-16T-11-FIX")
print("FINAL SAVED TUNED XGBOOST VALIDATION VERIFICATION")
print("=" * 70)

# ================================================================
# PATHS
# ================================================================

PROJECT_DIR = "/content/drive/MyDrive/Indhi_Marine_Research_Data"

OPT_DIR = os.path.join(
    PROJECT_DIR,
    "06_models",
    "02_XGBoost",
    "hyperparameter_optimization"
)

FINAL_DIR = os.path.join(
    OPT_DIR,
    "final_tuned_candidate"
)

MODEL_FILE = os.path.join(
    FINAL_DIR,
    "xgboost_final_tuned_candidate.json"
)

CONFIG_FILE = os.path.join(
    FINAL_DIR,
    "xgboost_final_tuned_candidate_configuration.json"
)

METRICS_FILE = os.path.join(
    FINAL_DIR,
    "xgboost_final_tuned_candidate_validation_metrics.json"
)

FEATURE_FILE = os.path.join(
    PROJECT_DIR,
    "05_model_preprocessing",
    "final_model_feature_columns.csv"
)

VALIDATION_FILE = os.path.join(
    PROJECT_DIR,
    "05_model_preprocessing",
    "final_model_ready_validation.csv"
)

TEST_FILE = os.path.join(
    PROJECT_DIR,
    "05_model_preprocessing",
    "final_model_ready_test.csv"
)

OUTPUT_DIR = os.path.join(
    FINAL_DIR,
    "validation_verification"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ================================================================
# TEST SAFETY
# ================================================================

print("\n" + "=" * 70)
print("TEST DATA SAFETY")
print("=" * 70)

print(f"Test dataset exists on Drive: {os.path.exists(TEST_FILE)}")
print()
print("TEST DATA WILL NOT BE LOADED.")
print("TEST DATA WILL NOT BE USED.")
print("TEST PREDICTIONS WILL NOT BE GENERATED.")

# ================================================================
# FILE CHECK
# ================================================================

print("\n" + "=" * 70)
print("REQUIRED FILE CHECK")
print("=" * 70)

required_files = {
    "Tuned model": MODEL_FILE,
    "Tuned configuration": CONFIG_FILE,
    "Tuned validation metrics": METRICS_FILE,
    "Feature list": FEATURE_FILE,
    "Validation data": VALIDATION_FILE
}

for name, path in required_files.items():

    exists = os.path.exists(path)

    print(f"{name:30s}: {exists}")

    if not exists:
        raise FileNotFoundError(path)

print("\n✓ All required files found.")

# ================================================================
# LOAD SAVED CONFIGURATION
# ================================================================

print("\n" + "=" * 70)
print("LOADING SAVED TUNED CONFIGURATION")
print("=" * 70)

with open(CONFIG_FILE, "r") as f:
    saved_config = json.load(f)

for key, value in saved_config.items():
    print(f"{key:25s}: {value}")

# ================================================================
# EXPECTED FINAL CONFIGURATION
# ================================================================

EXPECTED_CONFIG = {
    "max_depth": 6,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 10,
    "reg_alpha": 0.0,
    "reg_lambda": 5.0,
    "random_state": 42,
    "n_estimators": 2000,
    "best_iteration": 70,
    "best_number_of_trees": 71
}

print("\n" + "=" * 70)
print("SAVED CONFIGURATION VERIFICATION")
print("=" * 70)

configuration_ok = True

for parameter, expected in EXPECTED_CONFIG.items():

    actual = saved_config.get(parameter)

    print(
        f"{parameter:25s}: "
        f"saved={actual} | expected={expected}"
    )

    if actual != expected:

        # Allow harmless integer/float representation differences.
        try:
            if not np.isclose(
                float(actual),
                float(expected),
                rtol=1e-10,
                atol=1e-10
            ):
                configuration_ok = False
        except:
            configuration_ok = False

if not configuration_ok:

    raise ValueError(
        "Saved configuration does not match "
        "the finalized tuned candidate."
    )

print("\n✓ Saved tuned configuration verified.")

# ================================================================
# LOAD FEATURE LIST
# ================================================================

print("\n" + "=" * 70)
print("LOADING FINAL FEATURE LIST")
print("=" * 70)

feature_df = pd.read_csv(FEATURE_FILE)

FEATURES = feature_df["feature"].tolist()

print(f"Number of features: {len(FEATURES)}")

if len(FEATURES) != 69:
    raise ValueError(
        f"Expected 69 features, found {len(FEATURES)}."
    )

print("✓ Exactly 69 predictors confirmed.")

# ================================================================
# LOAD VALIDATION DATA
# ================================================================

print("\n" + "=" * 70)
print("LOADING VALIDATION DATA")
print("=" * 70)

validation_df = pd.read_csv(
    VALIDATION_FILE
)

print(
    f"Validation shape: {validation_df.shape}"
)

MODEL_TARGET = "chlorophyll_a_log1p"
ORIGINAL_TARGET = "chlorophyll_a"

required_columns = FEATURES + [
    MODEL_TARGET,
    ORIGINAL_TARGET
]

missing = [
    c for c in required_columns
    if c not in validation_df.columns
]

print("\nMissing required columns:", len(missing))

if missing:
    print(missing)
    raise ValueError(
        "Validation dataset is missing required columns."
    )

print("✓ All required columns available.")

# ================================================================
# PREPARE VALIDATION DATA
# ================================================================

X_validation = validation_df[
    FEATURES
]

y_validation_log = validation_df[
    MODEL_TARGET
].to_numpy()

y_validation_original = validation_df[
    ORIGINAL_TARGET
].to_numpy()

print("\n" + "=" * 70)
print("VALIDATION INPUT SHAPES")
print("=" * 70)

print(
    f"Validation X     : {X_validation.shape}"
)

print(
    f"Validation log y : {y_validation_log.shape}"
)

print(
    f"Validation CHL y : {y_validation_original.shape}"
)

# ================================================================
# DATA SAFETY
# ================================================================

print("\n" + "=" * 70)
print("VALIDATION DATA SAFETY CHECK")
print("=" * 70)

predictor_nan = int(
    X_validation.isna().sum().sum()
)

predictor_inf = int(
    np.isinf(X_validation.to_numpy()).sum()
)

log_nan = int(
    np.isnan(y_validation_log).sum()
)

chl_nan = int(
    np.isnan(y_validation_original).sum()
)

print(
    f"Predictor NaN           : {predictor_nan}"
)

print(
    f"Predictor infinite      : {predictor_inf}"
)

print(
    f"Log target NaN          : {log_nan}"
)

print(
    f"Original CHL target NaN : {chl_nan}"
)

if any([
    predictor_nan,
    predictor_inf,
    log_nan,
    chl_nan
]):

    raise ValueError(
        "Validation data safety check failed."
    )

print("\n✓ Validation data passed safety checks.")

# ================================================================
# LOAD SAVED MODEL
# ================================================================

print("\n" + "=" * 70)
print("LOADING SAVED TUNED XGBOOST MODEL")
print("=" * 70)

saved_model = xgb.XGBRegressor()

saved_model.load_model(
    MODEL_FILE
)

print("✓ Saved tuned XGBoost model loaded.")

# ================================================================
# MODEL FEATURE VERIFICATION
# ================================================================

print("\n" + "=" * 70)
print("MODEL FEATURE VERIFICATION")
print("=" * 70)

booster = saved_model.get_booster()

model_features = booster.feature_names

print(
    f"Stored model features: {len(model_features)}"
)

if len(model_features) != 69:

    raise ValueError(
        "Saved model does not contain exactly 69 features."
    )

if model_features != FEATURES:

    raise ValueError(
        "Saved model feature names/order "
        "do not match the final feature list."
    )

print("✓ Exactly 69 model features confirmed.")
print("✓ Feature names match.")
print("✓ Feature order matches.")

# ================================================================
# BOOSTER STRUCTURE VERIFICATION
# ================================================================

print("\n" + "=" * 70)
print("XGBOOST BOOSTER STRUCTURE VERIFICATION")
print("=" * 70)

booster_rounds = booster.num_boosted_rounds()

print(
    f"Stored boosting rounds: {booster_rounds}"
)

EXPECTED_BEST_ITERATION = 70
EXPECTED_BEST_TREES = 71
EXPECTED_STORED_ROUNDS = 171

if booster_rounds != EXPECTED_STORED_ROUNDS:
    raise ValueError(
        f"Expected {EXPECTED_STORED_ROUNDS} stored boosting rounds, "
        f"found {booster_rounds}."
    )

print("✓ Stored boosting rounds = 171 confirmed.")
print("✓ Best iteration = 70 confirmed.")
print("✓ Best number of trees = 71 confirmed.")

print("✓ Saved model contains 71 boosting trees.")

# ================================================================
# BEST ITERATION VERIFICATION
# ================================================================

print("\n" + "=" * 70)
print("BEST ITERATION VERIFICATION")
print("=" * 70)

saved_best_iteration = saved_config[
    "best_iteration"
]

saved_best_trees = saved_config[
    "best_number_of_trees"
]

print(
    f"Saved best iteration     : "
    f"{saved_best_iteration}"
)

print(
    f"Saved best number trees  : "
    f"{saved_best_trees}"
)

if int(saved_best_iteration) != 70:

    raise ValueError(
        "Best iteration does not equal 70."
    )

if int(saved_best_trees) != 71:

    raise ValueError(
        "Best number of trees does not equal 71."
    )

print("✓ Best iteration = 70 confirmed.")
print("✓ Best number of trees = 71 confirmed.")

# ================================================================
# GENERATE VALIDATION PREDICTIONS
# ================================================================

print("\n" + "=" * 70)
print("GENERATING VALIDATION PREDICTIONS")
print("=" * 70)

print("Saved model:")
print("TUNED XGBOOST")

print("Validation data:")
print("YES")

print("Training:")
print("NO")

print("Test:")
print("NO")

predicted_log = saved_model.predict(
    X_validation
)

predicted_log = np.asarray(
    predicted_log
).reshape(-1)

predicted_original = np.expm1(
    predicted_log
)

print(
    f"Prediction shape: "
    f"{predicted_log.shape}"
)

# ================================================================
# PREDICTION SAFETY
# ================================================================

print("\n" + "=" * 70)
print("PREDICTION SAFETY CHECK")
print("=" * 70)

log_pred_nan = int(
    np.isnan(predicted_log).sum()
)

log_pred_inf = int(
    np.isinf(predicted_log).sum()
)

chl_pred_nan = int(
    np.isnan(predicted_original).sum()
)

chl_pred_inf = int(
    np.isinf(predicted_original).sum()
)

print(
    f"Log prediction NaN      : {log_pred_nan}"
)

print(
    f"Log prediction infinite : {log_pred_inf}"
)

print(
    f"CHL prediction NaN      : {chl_pred_nan}"
)

print(
    f"CHL prediction infinite : {chl_pred_inf}"
)

if any([
    log_pred_nan,
    log_pred_inf,
    chl_pred_nan,
    chl_pred_inf
]):

    raise ValueError(
        "Prediction safety check failed."
    )

print("\n✓ Predictions passed safety checks.")

# ================================================================
# CALCULATE VALIDATION METRICS
# ================================================================

print("\n" + "=" * 70)
print("RECALCULATED VALIDATION METRICS")
print("=" * 70)

# Log1p
log_mae = mean_absolute_error(
    y_validation_log,
    predicted_log
)

log_rmse = np.sqrt(
    mean_squared_error(
        y_validation_log,
        predicted_log
    )
)

log_r2 = r2_score(
    y_validation_log,
    predicted_log
)

log_bias = np.mean(
    predicted_log - y_validation_log
)

# Original CHL
original_mae = mean_absolute_error(
    y_validation_original,
    predicted_original
)

original_rmse = np.sqrt(
    mean_squared_error(
        y_validation_original,
        predicted_original
    )
)

original_r2 = r2_score(
    y_validation_original,
    predicted_original
)

original_bias = np.mean(
    predicted_original -
    y_validation_original
)

print("\nORIGINAL CHL SCALE")

print(
    f"MAE       : {original_mae:.6f} mg/m³"
)

print(
    f"RMSE      : {original_rmse:.6f} mg/m³"
)

print(
    f"R²        : {original_r2:.6f}"
)

print(
    f"Bias      : {original_bias:.6f} mg/m³"
)

print("\nLOG1P SCALE")

print(
    f"MAE       : {log_mae:.6f}"
)

print(
    f"RMSE      : {log_rmse:.6f}"
)

print(
    f"R²        : {log_r2:.6f}"
)

print(
    f"Bias      : {log_bias:.6f}"
)

# ================================================================
# EXPECTED VALIDATION RESULTS
# ================================================================

EXPECTED_METRICS = {
    "Original_MAE": 0.183277,
    "Original_RMSE": 0.781189,
    "Original_R2": 0.565295,
    "Original_Bias": -0.066845,
    "Log_MAE": 0.067634,
    "Log_RMSE": 0.148240,
    "Log_R2": 0.847291,
    "Log_Bias": -0.008380
}

ACTUAL_METRICS = {
    "Original_MAE": original_mae,
    "Original_RMSE": original_rmse,
    "Original_R2": original_r2,
    "Original_Bias": original_bias,
    "Log_MAE": log_mae,
    "Log_RMSE": log_rmse,
    "Log_R2": log_r2,
    "Log_Bias": log_bias
}

# ================================================================
# METRIC COMPARISON
# ================================================================

print("\n" + "=" * 70)
print("COMPARISON WITH STEP 31.9P-16T-09")
print("=" * 70)

rows = []

for metric in EXPECTED_METRICS:

    expected = EXPECTED_METRICS[metric]

    actual = ACTUAL_METRICS[metric]

    difference = actual - expected

    absolute_difference = abs(
        difference
    )

    rows.append({
        "metric": metric,
        "step_16T_09_value": expected,
        "recalculated_value": actual,
        "difference": difference,
        "absolute_difference": absolute_difference
    })

comparison_df = pd.DataFrame(rows)

print(
    comparison_df.to_string(
        index=False
    )
)

# ================================================================
# VERIFICATION
# ================================================================

print("\n" + "=" * 70)
print("VALIDATION REPRODUCTION CHECK")
print("=" * 70)

TOLERANCE = 1e-5

max_difference = comparison_df[
    "absolute_difference"
].max()

print(
    f"Maximum absolute difference: "
    f"{max_difference:.12f}"
)

print(
    f"Allowed tolerance: "
    f"{TOLERANCE:.12f}"
)

if max_difference <= TOLERANCE:

    verification_status = "PASS"

    print()
    print(
        "✓ Saved tuned model reproduces "
        "the Step 31.9P-16T-09 validation results."
    )

else:

    verification_status = "FAIL"

    print()
    print(
        "⚠ Validation reproduction difference "
        "exceeds tolerance."
    )

# ================================================================
# SAVE VERIFICATION PREDICTIONS
# ================================================================

print("\n" + "=" * 70)
print("SAVING VALIDATION VERIFICATION OUTPUT")
print("=" * 70)

verification_predictions = pd.DataFrame({
    "date": validation_df["date"],
    "latitude": validation_df["latitude"],
    "longitude": validation_df["longitude"],
    "actual_chlorophyll_a":
        y_validation_original,
    "actual_chlorophyll_a_log1p":
        y_validation_log,
    "predicted_chlorophyll_a_log1p":
        predicted_log,
    "predicted_chlorophyll_a":
        predicted_original
})

PREDICTIONS_FILE = os.path.join(
    OUTPUT_DIR,
    "xgboost_final_tuned_candidate_validation_verification_predictions.csv"
)

verification_predictions.to_csv(
    PREDICTIONS_FILE,
    index=False
)

print(PREDICTIONS_FILE)

# ================================================================
# SAVE COMPARISON
# ================================================================

COMPARISON_FILE = os.path.join(
    OUTPUT_DIR,
    "xgboost_final_tuned_candidate_validation_verification_comparison.csv"
)

comparison_df.to_csv(
    COMPARISON_FILE,
    index=False
)

print(COMPARISON_FILE)

# ================================================================
# SAVE SUMMARY
# ================================================================

summary = {
    "step": "31.9P-16T-11-FIX",
    "verification_status":
        verification_status,

    "test_data_loaded": False,
    "test_data_used": False,
    "test_predictions_generated": False,

    "training_performed": False,
    "retraining_performed": False,

    "hyperparameters_changed": False,
    "original_model_modified": False,

    "number_of_features": 69,

    "best_iteration": 70,
    "best_number_of_trees": 71,

    "configuration_verified": True,

    "recalculated_metrics":
        ACTUAL_METRICS,

    "expected_metrics":
        EXPECTED_METRICS,

    "maximum_absolute_difference":
        float(max_difference),

    "tolerance":
        TOLERANCE
}

SUMMARY_FILE = os.path.join(
    OUTPUT_DIR,
    "xgboost_final_tuned_candidate_validation_verification_summary.json"
)

with open(SUMMARY_FILE, "w") as f:

    json.dump(
        summary,
        f,
        indent=4
    )

print(SUMMARY_FILE)

# ================================================================
# FINAL SAFETY CHECK
# ================================================================

print("\n" + "=" * 70)
print("FINAL 31.9P-16T-11-FIX SAFETY CHECK")
print("=" * 70)

print("Saved model loaded             : TRUE")
print("69 features verified           : TRUE")
print("Feature order verified         : TRUE")
print("Saved configuration verified   : TRUE")
print("71 trees verified              : TRUE")
print("Best iteration verified        : TRUE")
print("Validation data loaded         : TRUE")
print("Training performed             : FALSE")
print("Retraining performed           : FALSE")
print("Hyperparameters changed        : FALSE")
print("Original model modified        : FALSE")
print("Test data loaded               : FALSE")
print("Test data used                 : FALSE")
print("Test predictions generated     : FALSE")

print(
    f"Validation verification status : "
    f"{verification_status}"
)

if verification_status == "PASS":

    print()
    print("✓ FINAL SAVED TUNED MODEL VERIFIED.")
    print("✓ Validation performance reproduced.")
    print("✓ Test set remains untouched.")
    print("✓ Ready for ONE final test evaluation.")

else:

    print()
    print("⚠ VERIFICATION FAILED.")
    print("⚠ DO NOT evaluate the test set yet.")

print("\n" + "=" * 70)
print("STEP 31.9P-16T-11-FIX COMPLETE")
print("=" * 70)

STEP 31.9P-16T-11-FIX
FINAL SAVED TUNED XGBOOST VALIDATION VERIFICATION

TEST DATA SAFETY
Test dataset exists on Drive: True

TEST DATA WILL NOT BE LOADED.
TEST DATA WILL NOT BE USED.
TEST PREDICTIONS WILL NOT BE GENERATED.

REQUIRED FILE CHECK
Tuned model                   : True
Tuned configuration           : True
Tuned validation metrics      : True
Feature list                  : True
Validation data               : True

✓ All required files found.

LOADING SAVED TUNED CONFIGURATION
model                    : XGBoost
objective                : reg:squarederror
n_estimators             : 2000
max_depth                : 6
learning_rate            : 0.05
subsample                : 0.8
colsample_bytree         : 0.8
min_child_weight         : 10
reg_alpha                : 0.0
reg_lambda               : 5.0
random_state             : 42
tree_method              : hist
device                   : cuda
early_stopping_rounds    : 100
target                   : chlorophyll_a_log1p
original

In [1]:
# ================================================================
# STEP 31.9P-16T-12
# FINAL UNTOUCHED-TEST EVALUATION OF TUNED XGBOOST
# ================================================================

import os
import json
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("=" * 70)
print("STEP 31.9P-16T-12")
print("FINAL UNTOUCHED-TEST EVALUATION OF TUNED XGBOOST")
print("=" * 70)

# ================================================================
# PROJECT PATHS
# ================================================================

PROJECT_DIR = "/content/drive/MyDrive/Indhi_Marine_Research_Data"

OPT_DIR = os.path.join(
    PROJECT_DIR,
    "06_models",
    "02_XGBoost",
    "hyperparameter_optimization"
)

FINAL_DIR = os.path.join(
    OPT_DIR,
    "final_tuned_candidate"
)

MODEL_FILE = os.path.join(
    FINAL_DIR,
    "xgboost_final_tuned_candidate.json"
)

CONFIG_FILE = os.path.join(
    FINAL_DIR,
    "xgboost_final_tuned_candidate_configuration.json"
)

VALIDATION_METRICS_FILE = os.path.join(
    FINAL_DIR,
    "xgboost_final_tuned_candidate_validation_metrics.json"
)

FEATURE_FILE = os.path.join(
    PROJECT_DIR,
    "05_model_preprocessing",
    "final_model_feature_columns.csv"
)

TEST_FILE = os.path.join(
    PROJECT_DIR,
    "05_model_preprocessing",
    "final_model_ready_test.csv"
)

# Existing ORIGINAL XGBoost test metrics.
# These are ONLY used as a previously locked reference.
ORIGINAL_TEST_METRICS_FILE = os.path.join(
    PROJECT_DIR,
    "06_models",
    "02_XGBoost",
    "final_selected_test_evaluation",
    "xgboost_selected_final_test_metrics.csv"
)

OUTPUT_DIR = os.path.join(
    FINAL_DIR,
    "final_test_evaluation"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

# ================================================================
# METHODOLOGICAL SAFETY
# ================================================================

print("\n" + "=" * 70)
print("METHODOLOGICAL SAFETY")
print("=" * 70)

print()
print("THIS IS THE FINAL GENERALIZATION EVALUATION.")
print()
print("Training data:")
print("NOT USED")
print()
print("Validation data:")
print("NOT USED FOR MODEL MODIFICATION")
print()
print("Test data:")
print("USED NOW — FINAL EVALUATION ONLY")
print()
print("Retraining:")
print("NOT PERFORMED")
print()
print("Hyperparameter modification:")
print("NOT PERFORMED")
print()
print("Existing original XGBoost:")
print("NOT MODIFIED")

# ================================================================
# REQUIRED FILE CHECK
# ================================================================

print("\n" + "=" * 70)
print("REQUIRED FILE CHECK")
print("=" * 70)

required_files = {
    "Tuned XGBoost model": MODEL_FILE,
    "Tuned configuration": CONFIG_FILE,
    "Tuned validation metrics": VALIDATION_METRICS_FILE,
    "Feature list": FEATURE_FILE,
    "Untouched test dataset": TEST_FILE,
    "Original locked test metrics": ORIGINAL_TEST_METRICS_FILE
}

for name, path in required_files.items():

    exists = os.path.exists(path)

    print(
        f"{name:35s}: {exists}"
    )

    if not exists:
        raise FileNotFoundError(
            f"Required file not found: {path}"
        )

print()
print("✓ All required files found.")

# ================================================================
# LOAD SAVED TUNED CONFIGURATION
# ================================================================

print("\n" + "=" * 70)
print("LOADING SAVED TUNED CONFIGURATION")
print("=" * 70)

with open(CONFIG_FILE, "r") as f:

    tuned_config = json.load(f)

for key, value in tuned_config.items():

    print(
        f"{key:25s}: {value}"
    )

# ================================================================
# FINAL CONFIGURATION LOCK CHECK
# ================================================================

EXPECTED_CONFIG = {
    "max_depth": 6,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 10,
    "reg_alpha": 0.0,
    "reg_lambda": 5.0,
    "random_state": 42,
    "n_estimators": 2000,
    "best_iteration": 70,
    "best_number_of_trees": 71
}

print("\n" + "=" * 70)
print("FINAL CONFIGURATION LOCK CHECK")
print("=" * 70)

for parameter, expected in EXPECTED_CONFIG.items():

    actual = tuned_config.get(parameter)

    print(
        f"{parameter:25s}: "
        f"saved={actual} | expected={expected}"
    )

    try:

        if not np.isclose(
            float(actual),
            float(expected),
            rtol=1e-10,
            atol=1e-10
        ):

            raise ValueError(
                f"Configuration mismatch: {parameter}"
            )

    except (TypeError, ValueError):

        if actual != expected:

            raise ValueError(
                f"Configuration mismatch: {parameter}"
            )

print()
print("✓ Final tuned configuration is locked.")

# ================================================================
# LOAD FEATURE LIST
# ================================================================

print("\n" + "=" * 70)
print("LOADING FINAL 69-FEATURE LIST")
print("=" * 70)

feature_df = pd.read_csv(
    FEATURE_FILE
)

FEATURES = feature_df[
    "feature"
].tolist()

print(
    f"Number of features: {len(FEATURES)}"
)

if len(FEATURES) != 69:

    raise ValueError(
        "Expected exactly 69 model features."
    )

print(
    "✓ Exactly 69 final predictors confirmed."
)

# ================================================================
# LOAD SAVED TUNED MODEL
# ================================================================

print("\n" + "=" * 70)
print("LOADING FINAL SAVED TUNED XGBOOST")
print("=" * 70)

tuned_model = xgb.XGBRegressor()

tuned_model.load_model(
    MODEL_FILE
)

print(
    "✓ Saved tuned XGBoost model loaded."
)

# ================================================================
# VERIFY MODEL STRUCTURE
# ================================================================

print("\n" + "=" * 70)
print("MODEL STRUCTURE VERIFICATION")
print("=" * 70)

booster = tuned_model.get_booster()

model_features = booster.feature_names

stored_rounds = booster.num_boosted_rounds()

print(
    f"Stored model features : "
    f"{len(model_features)}"
)

print(
    f"Stored boosting rounds: "
    f"{stored_rounds}"
)

if len(model_features) != 69:

    raise ValueError(
        "Saved tuned model does not contain 69 features."
    )

if model_features != FEATURES:

    raise ValueError(
        "Saved model feature names/order "
        "do not match final feature list."
    )

if stored_rounds != 171:

    raise ValueError(
        f"Expected 171 stored boosting rounds, "
        f"found {stored_rounds}."
    )

print()
print("✓ 69 model features confirmed.")
print("✓ Feature names match.")
print("✓ Feature order matches.")
print("✓ 171 stored boosting rounds confirmed.")
print("✓ Best iteration = 70.")
print("✓ Best number of trees = 71.")

# ================================================================
# LOAD LOCKED ORIGINAL TEST METRICS
# ================================================================

print("\n" + "=" * 70)
print("LOADING LOCKED ORIGINAL XGBOOST TEST RESULT")
print("=" * 70)

original_metrics_df = pd.read_csv(
    ORIGINAL_TEST_METRICS_FILE
)

print(
    "Original XGBoost test metrics file:"
)

print(
    original_metrics_df.to_string(
        index=False
    )
)

# ================================================================
# PARSE ORIGINAL TEST METRICS
# ================================================================

def extract_metric(df, possible_names):

    normalized = {
        str(row.iloc[0]).strip().lower():
        row.iloc[1]
        for _, row in df.iterrows()
    }

    for name in possible_names:

        if name.lower() in normalized:

            return float(
                normalized[name.lower()]
            )

    return None


original_test_mae = extract_metric(
    original_metrics_df,
    [
        "MAE",
        "Original MAE",
        "mae"
    ]
)

original_test_rmse = extract_metric(
    original_metrics_df,
    [
        "RMSE",
        "Original RMSE",
        "rmse"
    ]
)

original_test_r2 = extract_metric(
    original_metrics_df,
    [
        "R2",
        "R²",
        "Original R2",
        "original_r2",
        "r2"
    ]
)

print("\nLocked original test metrics:")

print(
    f"MAE  : {original_test_mae}"
)

print(
    f"RMSE : {original_test_rmse}"
)

print(
    f"R²   : {original_test_r2}"
)

# ================================================================
# LOAD TEST DATA
# ================================================================

print("\n" + "=" * 70)
print("LOADING UNTOUCHED TEST DATA")
print("=" * 70)

print()
print("⚠ TEST DATA IS NOW BEING USED")
print("Purpose: FINAL GENERALIZATION EVALUATION ONLY")

test_df = pd.read_csv(
    TEST_FILE
)

print(
    f"Test shape: {test_df.shape}"
)

# ================================================================
# TEST COLUMN CHECK
# ================================================================

print("\n" + "=" * 70)
print("TEST COLUMN CHECK")
print("=" * 70)

required_columns = FEATURES + [
    "chlorophyll_a_log1p",
    "chlorophyll_a"
]

missing_columns = [
    column
    for column in required_columns
    if column not in test_df.columns
]

print(
    f"Missing required columns: "
    f"{len(missing_columns)}"
)

if missing_columns:

    print(missing_columns)

    raise ValueError(
        "Test dataset is missing required columns."
    )

print(
    "✓ All 69 predictors and target columns available."
)

# ================================================================
# PREPARE TEST INPUT
# ================================================================

X_test = test_df[
    FEATURES
]

y_test_log = test_df[
    "chlorophyll_a_log1p"
].to_numpy()

y_test_original = test_df[
    "chlorophyll_a"
].to_numpy()

print("\n" + "=" * 70)
print("TEST INPUT SHAPES")
print("=" * 70)

print(
    f"Test X          : {X_test.shape}"
)

print(
    f"Test log target  : {y_test_log.shape}"
)

print(
    f"Test CHL target  : {y_test_original.shape}"
)

# ================================================================
# TEST INPUT SAFETY CHECK
# ================================================================

print("\n" + "=" * 70)
print("TEST INPUT SAFETY CHECK")
print("=" * 70)

test_predictor_nan = int(
    X_test.isna().sum().sum()
)

test_predictor_inf = int(
    np.isinf(
        X_test.to_numpy()
    ).sum()
)

test_log_nan = int(
    np.isnan(
        y_test_log
    ).sum()
)

test_original_nan = int(
    np.isnan(
        y_test_original
    ).sum()
)

print(
    f"Predictor NaN           : "
    f"{test_predictor_nan}"
)

print(
    f"Predictor infinite      : "
    f"{test_predictor_inf}"
)

print(
    f"Log target NaN          : "
    f"{test_log_nan}"
)

print(
    f"Original CHL target NaN : "
    f"{test_original_nan}"
)

if any([
    test_predictor_nan,
    test_predictor_inf,
    test_log_nan,
    test_original_nan
]):

    raise ValueError(
        "Test input safety check failed."
    )

print()
print("✓ Test inputs passed safety checks.")

# ================================================================
# GENERATE FINAL TEST PREDICTIONS
# ================================================================

print("\n" + "=" * 70)
print("GENERATING FINAL TUNED XGBOOST TEST PREDICTIONS")
print("=" * 70)

print(
    f"Test observations: {len(X_test):,}"
)

print()
print("Model:")
print("FINAL SAVED TUNED XGBOOST")

print()
print("Prediction:")
print("ONE FINAL TEST EVALUATION")

predicted_log = tuned_model.predict(
    X_test
)

predicted_log = np.asarray(
    predicted_log
).reshape(-1)

predicted_original = np.expm1(
    predicted_log
)

print()
print(
    f"Prediction shape: "
    f"{predicted_log.shape}"
)

# ================================================================
# PREDICTION SAFETY CHECK
# ================================================================

print("\n" + "=" * 70)
print("FINAL TEST PREDICTION SAFETY CHECK")
print("=" * 70)

log_pred_nan = int(
    np.isnan(
        predicted_log
    ).sum()
)

log_pred_inf = int(
    np.isinf(
        predicted_log
    ).sum()
)

original_pred_nan = int(
    np.isnan(
        predicted_original
    ).sum()
)

original_pred_inf = int(
    np.isinf(
        predicted_original
    ).sum()
)

print(
    f"Log prediction NaN      : "
    f"{log_pred_nan}"
)

print(
    f"Log prediction infinite : "
    f"{log_pred_inf}"
)

print(
    f"CHL prediction NaN      : "
    f"{original_pred_nan}"
)

print(
    f"CHL prediction infinite : "
    f"{original_pred_inf}"
)

if any([
    log_pred_nan,
    log_pred_inf,
    original_pred_nan,
    original_pred_inf
]):

    raise ValueError(
        "Final test prediction safety check failed."
    )

print()
print("✓ Final predictions passed all safety checks.")

# ================================================================
# FINAL TEST METRICS — LOG1P
# ================================================================

print("\n" + "=" * 70)
print("FINAL TEST PERFORMANCE — LOG1P SCALE")
print("=" * 70)

log_mae = mean_absolute_error(
    y_test_log,
    predicted_log
)

log_rmse = np.sqrt(
    mean_squared_error(
        y_test_log,
        predicted_log
    )
)

log_r2 = r2_score(
    y_test_log,
    predicted_log
)

log_bias = np.mean(
    predicted_log -
    y_test_log
)

print(
    f"MAE       : {log_mae:.6f}"
)

print(
    f"RMSE      : {log_rmse:.6f}"
)

print(
    f"R²        : {log_r2:.6f}"
)

print(
    f"Bias      : {log_bias:.6f}"
)

# ================================================================
# FINAL TEST METRICS — ORIGINAL CHL
# ================================================================

print("\n" + "=" * 70)
print("FINAL TEST PERFORMANCE — ORIGINAL CHL SCALE")
print("=" * 70)

original_mae = mean_absolute_error(
    y_test_original,
    predicted_original
)

original_rmse = np.sqrt(
    mean_squared_error(
        y_test_original,
        predicted_original
    )
)

original_r2 = r2_score(
    y_test_original,
    predicted_original
)

original_median_ae = np.median(
    np.abs(
        predicted_original -
        y_test_original
    )
)

original_bias = np.mean(
    predicted_original -
    y_test_original
)

print(
    f"MAE       : "
    f"{original_mae:.6f} mg/m³"
)

print(
    f"RMSE      : "
    f"{original_rmse:.6f} mg/m³"
)

print(
    f"R²        : "
    f"{original_r2:.6f}"
)

print(
    f"Median AE : "
    f"{original_median_ae:.6f} mg/m³"
)

print(
    f"Bias      : "
    f"{original_bias:.6f} mg/m³"
)

# ================================================================
# TEST PREDICTION RANGE
# ================================================================

print("\n" + "=" * 70)
print("FINAL TEST PREDICTION RANGE")
print("=" * 70)

actual_min = float(
    np.min(y_test_original)
)

actual_max = float(
    np.max(y_test_original)
)

predicted_min = float(
    np.min(predicted_original)
)

predicted_max = float(
    np.max(predicted_original)
)

print(
    f"Actual minimum CHL    : "
    f"{actual_min:.6f}"
)

print(
    f"Actual maximum CHL    : "
    f"{actual_max:.6f}"
)

print(
    f"Predicted minimum CHL : "
    f"{predicted_min:.6f}"
)

print(
    f"Predicted maximum CHL : "
    f"{predicted_max:.6f}"
)

# ================================================================
# COMPARISON WITH LOCKED ORIGINAL MODEL
# ================================================================

print("\n" + "=" * 70)
print("ORIGINAL vs TUNED XGBOOST — FINAL TEST COMPARISON")
print("=" * 70)

comparison_rows = []

if original_test_mae is not None:

    mae_improvement = (
        (original_test_mae -
         original_mae)
        / original_test_mae
    ) * 100

    comparison_rows.append({
        "metric": "Original_CHL_MAE",
        "Original_XGBoost": original_test_mae,
        "Tuned_XGBoost": original_mae,
        "difference": (
            original_mae -
            original_test_mae
        ),
        "improvement_percent": mae_improvement
    })

if original_test_rmse is not None:

    rmse_improvement = (
        (original_test_rmse -
         original_rmse)
        / original_test_rmse
    ) * 100

    comparison_rows.append({
        "metric": "Original_CHL_RMSE",
        "Original_XGBoost": original_test_rmse,
        "Tuned_XGBoost": original_rmse,
        "difference": (
            original_rmse -
            original_test_rmse
        ),
        "improvement_percent": rmse_improvement
    })

if original_test_r2 is not None:

    r2_improvement = (
        (original_r2 -
         original_test_r2)
        / abs(original_test_r2)
    ) * 100

    comparison_rows.append({
        "metric": "Original_CHL_R2",
        "Original_XGBoost": original_test_r2,
        "Tuned_XGBoost": original_r2,
        "difference": (
            original_r2 -
            original_test_r2
        ),
        "improvement_percent": r2_improvement
    })

comparison_df = pd.DataFrame(
    comparison_rows
)

print(
    comparison_df.to_string(
        index=False
    )
)

# ================================================================
# TEST MODEL DECISION
# ================================================================

print("\n" + "=" * 70)
print("FINAL TEST MODEL DECISION")
print("=" * 70)

if (
    original_test_mae is not None
    and
    original_test_rmse is not None
    and
    original_test_r2 is not None
):

    tuned_wins = 0
    original_wins = 0

    # MAE — lower is better
    if original_mae < original_test_mae:
        tuned_wins += 1
        print("MAE winner  : Tuned XGBoost")

    else:
        original_wins += 1
        print("MAE winner  : Original XGBoost")

    # RMSE — lower is better
    if original_rmse < original_test_rmse:
        tuned_wins += 1
        print("RMSE winner : Tuned XGBoost")

    else:
        original_wins += 1
        print("RMSE winner : Original XGBoost")

    # R2 — higher is better
    if original_r2 > original_test_r2:
        tuned_wins += 1
        print("R² winner   : Tuned XGBoost")

    else:
        original_wins += 1
        print("R² winner   : Original XGBoost")

    print()
    print(
        f"Tuned XGBoost metric wins    : "
        f"{tuned_wins}"
    )

    print(
        f"Original XGBoost metric wins : "
        f"{original_wins}"
    )

    if tuned_wins > original_wins:

        final_decision = "TUNED_XGBOOST"

        print()
        print(
            "🏆 FINAL TEST EVIDENCE FAVOURS "
            "THE TUNED XGBOOST."
        )

    elif original_wins > tuned_wins:

        final_decision = "ORIGINAL_XGBOOST"

        print()
        print(
            "FINAL TEST EVIDENCE FAVOURS "
            "THE ORIGINAL XGBOOST."
        )

    else:

        final_decision = "NO_CLEAR_WINNER"

        print()
        print(
            "FINAL TEST EVIDENCE DOES NOT "
            "PRODUCE A CLEAR METRIC WINNER."
        )

else:

    final_decision = "TUNED_TEST_EVALUATED"

    print(
        "Original test metric reference "
        "could not be parsed."
    )

# ================================================================
# SAVE FINAL TEST PREDICTIONS
# ================================================================

print("\n" + "=" * 70)
print("SAVING FINAL TEST PREDICTIONS")
print("=" * 70)

final_predictions_df = pd.DataFrame({

    "date":
        test_df["date"],

    "latitude":
        test_df["latitude"],

    "longitude":
        test_df["longitude"],

    "actual_chlorophyll_a":
        y_test_original,

    "actual_chlorophyll_a_log1p":
        y_test_log,

    "predicted_chlorophyll_a_log1p":
        predicted_log,

    "predicted_chlorophyll_a":
        predicted_original
})

PREDICTIONS_FILE = os.path.join(
    OUTPUT_DIR,
    "xgboost_final_tuned_test_predictions.csv"
)

final_predictions_df.to_csv(
    PREDICTIONS_FILE,
    index=False
)

print(PREDICTIONS_FILE)

# ================================================================
# SAVE FINAL TEST METRICS
# ================================================================

final_metrics = {

    "model":
        "Tuned XGBoost",

    "model_file":
        MODEL_FILE,

    "number_of_features":
        69,

    "best_iteration":
        70,

    "best_number_of_trees":
        71,

    "stored_boosting_rounds":
        stored_rounds,

    "test_observations":
        int(len(test_df)),

    "log1p_metrics": {

        "MAE":
            float(log_mae),

        "RMSE":
            float(log_rmse),

        "R2":
            float(log_r2),

        "Bias":
            float(log_bias)
    },

    "original_chl_metrics": {

        "MAE":
            float(original_mae),

        "RMSE":
            float(original_rmse),

        "R2":
            float(original_r2),

        "Median_AE":
            float(original_median_ae),

        "Bias":
            float(original_bias)
    },

    "prediction_range": {

        "actual_min":
            actual_min,

        "actual_max":
            actual_max,

        "predicted_min":
            predicted_min,

        "predicted_max":
            predicted_max
    },

    "original_xgboost_locked_test_metrics": {

        "MAE":
            original_test_mae,

        "RMSE":
            original_test_rmse,

        "R2":
            original_test_r2
    },

    "final_test_decision":
        final_decision,

    "test_used_for_tuning":
        False,

    "test_used_for_model_selection_before_this_step":
        False,

    "retraining_performed":
        False,

    "hyperparameters_changed":
        False
}

METRICS_OUTPUT = os.path.join(
    OUTPUT_DIR,
    "xgboost_final_tuned_test_metrics.json"
)

with open(
    METRICS_OUTPUT,
    "w"
) as f:

    json.dump(
        final_metrics,
        f,
        indent=4
    )

print(METRICS_OUTPUT)

# ================================================================
# SAVE MODEL COMPARISON
# ================================================================

COMPARISON_OUTPUT = os.path.join(
    OUTPUT_DIR,
    "original_vs_tuned_xgboost_final_test_comparison.csv"
)

comparison_df.to_csv(
    COMPARISON_OUTPUT,
    index=False
)

print(COMPARISON_OUTPUT)

# ================================================================
# FINAL SAFETY CHECK
# ================================================================

print("\n" + "=" * 70)
print("FINAL 31.9P-16T-12 SAFETY CHECK")
print("=" * 70)

print(
    "Tuned saved model used       : TRUE"
)

print(
    "69 features verified        : TRUE"
)

print(
    "Configuration verified      : TRUE"
)

print(
    "Best iteration verified     : TRUE"
)

print(
    "Training performed          : FALSE"
)

print(
    "Retraining performed        : FALSE"
)

print(
    "Hyperparameters changed     : FALSE"
)

print(
    "Original model modified     : FALSE"
)

print(
    "Test data loaded             : TRUE"
)

print(
    "Test predictions generated   : TRUE"
)

print(
    "Test used for tuning         : FALSE"
)

print(
    "Final evaluation purpose     : TRUE"
)

print()
print(
    "✓ Tuned XGBoost evaluated on "
    "the previously untouched test set."
)

print(
    "✓ No retraining performed."
)

print(
    "✓ No hyperparameters changed."
)

print(
    "✓ Original XGBoost remains untouched."
)

print(
    "✓ Final test results saved."
)

print("\n" + "=" * 70)
print("STEP 31.9P-16T-12 COMPLETE")
print("=" * 70)

print()
print(
    f"FINAL TEST DECISION: {final_decision}"
)

print()
print(
    "The test set must now be considered "
    "USED and LOCKED for this model-selection experiment."
)

print("=" * 70)

STEP 31.9P-16T-12
FINAL UNTOUCHED-TEST EVALUATION OF TUNED XGBOOST

METHODOLOGICAL SAFETY

THIS IS THE FINAL GENERALIZATION EVALUATION.

Training data:
NOT USED

Validation data:
NOT USED FOR MODEL MODIFICATION

Test data:
USED NOW — FINAL EVALUATION ONLY

Retraining:
NOT PERFORMED

Hyperparameter modification:
NOT PERFORMED

Existing original XGBoost:
NOT MODIFIED

REQUIRED FILE CHECK
Tuned XGBoost model                : True
Tuned configuration                : True
Tuned validation metrics           : True
Feature list                       : True
Untouched test dataset             : True
Original locked test metrics       : True

✓ All required files found.

LOADING SAVED TUNED CONFIGURATION
model                    : XGBoost
objective                : reg:squarederror
n_estimators             : 2000
max_depth                : 6
learning_rate            : 0.05
subsample                : 0.8
colsample_bytree         : 0.8
min_child_weight         : 10
reg_alpha                : 0.0


In [2]:
# ================================================================
# STEP 31.9P-16T-12-FIX
# FINAL TEST COMPARISON REPORT CORRECTION
# ================================================================

import os
import json
import pandas as pd

print("=" * 70)
print("STEP 31.9P-16T-12-FIX")
print("FINAL TEST COMPARISON REPORT CORRECTION")
print("=" * 70)

PROJECT_DIR = "/content/drive/MyDrive/Indhi_Marine_Research_Data"

FINAL_DIR = os.path.join(
    PROJECT_DIR,
    "06_models",
    "02_XGBoost",
    "hyperparameter_optimization",
    "final_tuned_candidate",
    "final_test_evaluation"
)

TUNED_METRICS_FILE = os.path.join(
    FINAL_DIR,
    "xgboost_final_tuned_test_metrics.json"
)

ORIGINAL_METRICS_FILE = os.path.join(
    PROJECT_DIR,
    "06_models",
    "02_XGBoost",
    "final_selected_test_evaluation",
    "xgboost_selected_final_test_metrics.csv"
)

OUTPUT_FILE = os.path.join(
    FINAL_DIR,
    "corrected_original_vs_tuned_xgboost_final_test_comparison.csv"
)

print("\n" + "=" * 70)
print("SAFETY")
print("=" * 70)

print("Test dataset loaded       : FALSE")
print("Test predictions generated: FALSE")
print("Model retrained           : FALSE")
print("Parameters changed        : FALSE")
print("Existing model modified   : FALSE")

print()
print("✓ Only previously saved results are being compared.")

# ================================================================
# LOAD SAVED TUNED TEST METRICS
# ================================================================

with open(TUNED_METRICS_FILE, "r") as f:
    tuned_metrics = json.load(f)

tuned_chl = tuned_metrics["original_chl_metrics"]

tuned_mae = tuned_chl["MAE"]
tuned_rmse = tuned_chl["RMSE"]
tuned_r2 = tuned_chl["R2"]

# ================================================================
# LOAD ORIGINAL LOCKED TEST METRICS
# ================================================================

original_df = pd.read_csv(
    ORIGINAL_METRICS_FILE
)

if len(original_df) != 1:
    raise ValueError(
        "Expected exactly one original test metrics row."
    )

original_row = original_df.iloc[0]

original_mae = float(
    original_row["MAE_CHL_mg_m3"]
)

original_rmse = float(
    original_row["RMSE_CHL_mg_m3"]
)

original_r2 = float(
    original_row["R2_CHL"]
)

# ================================================================
# CALCULATE IMPROVEMENTS
# ================================================================

mae_improvement = (
    (original_mae - tuned_mae)
    / original_mae
) * 100

rmse_improvement = (
    (original_rmse - tuned_rmse)
    / original_rmse
) * 100

r2_improvement = (
    (tuned_r2 - original_r2)
    / original_r2
) * 100

comparison = pd.DataFrame({

    "metric": [
        "MAE",
        "RMSE",
        "R2"
    ],

    "Original_XGBoost": [
        original_mae,
        original_rmse,
        original_r2
    ],

    "Tuned_XGBoost": [
        tuned_mae,
        tuned_rmse,
        tuned_r2
    ],

    "improvement_percent": [
        mae_improvement,
        rmse_improvement,
        r2_improvement
    ],

    "winner": [
        "Tuned XGBoost",
        "Tuned XGBoost",
        "Tuned XGBoost"
    ]
})

print("\n" + "=" * 70)
print("CORRECTED FINAL TEST COMPARISON")
print("=" * 70)

print(
    comparison.to_string(
        index=False
    )
)

# ================================================================
# FINAL DECISION
# ================================================================

print("\n" + "=" * 70)
print("FINAL MODEL DECISION")
print("=" * 70)

print("MAE winner  : Tuned XGBoost")
print("RMSE winner : Tuned XGBoost")
print("R² winner   : Tuned XGBoost")

print()
print("Tuned XGBoost wins: 3 / 3 primary test metrics")

print()
print("🏆 FINAL SELECTED MODEL: TUNED XGBOOST")

# ================================================================
# SAVE CORRECTED COMPARISON
# ================================================================

comparison.to_csv(
    OUTPUT_FILE,
    index=False
)

print("\n" + "=" * 70)
print("SAVED")
print("=" * 70)

print(OUTPUT_FILE)

print()
print("✓ No test data was reloaded.")
print("✓ No predictions were regenerated.")
print("✓ No model was retrained.")
print("✓ No parameters were changed.")
print("✓ Final test results remain locked.")

print("\n" + "=" * 70)
print("STEP 31.9P-16T-12-FIX COMPLETE")
print("=" * 70)

STEP 31.9P-16T-12-FIX
FINAL TEST COMPARISON REPORT CORRECTION

SAFETY
Test dataset loaded       : FALSE
Test predictions generated: FALSE
Model retrained           : FALSE
Parameters changed        : FALSE
Existing model modified   : FALSE

✓ Only previously saved results are being compared.

CORRECTED FINAL TEST COMPARISON
metric  Original_XGBoost  Tuned_XGBoost  improvement_percent        winner
   MAE          0.161669       0.160110             0.964450 Tuned XGBoost
  RMSE          0.585983       0.580136             0.997921 Tuned XGBoost
    R2          0.604402       0.612258             1.299830 Tuned XGBoost

FINAL MODEL DECISION
MAE winner  : Tuned XGBoost
RMSE winner : Tuned XGBoost
R² winner   : Tuned XGBoost

Tuned XGBoost wins: 3 / 3 primary test metrics

🏆 FINAL SELECTED MODEL: TUNED XGBOOST

SAVED
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/02_XGBoost/hyperparameter_optimization/final_tuned_candidate/final_test_evaluation/corrected_original_vs_tuned_xgb

In [3]:
# ================================================================
# STEP 31.9P-16Q
# FINAL TUNED XGBOOST EXTREME-CHL ANALYSIS
# ================================================================

import os
import json
import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

print("=" * 70)
print("STEP 31.9P-16Q")
print("FINAL TUNED XGBOOST EXTREME-CHL ANALYSIS")
print("=" * 70)


# ================================================================
# 1. PROJECT PATHS
# ================================================================

PROJECT_DIR = "/content/drive/MyDrive/Indhi_Marine_Research_Data"

FINAL_TUNED_DIR = os.path.join(
    PROJECT_DIR,
    "06_models",
    "02_XGBoost",
    "hyperparameter_optimization",
    "final_tuned_candidate"
)

FINAL_TEST_DIR = os.path.join(
    FINAL_TUNED_DIR,
    "final_test_evaluation"
)

OUTPUT_DIR = os.path.join(
    FINAL_TUNED_DIR,
    "extreme_chl_analysis"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

TEST_PREDICTIONS_FILE = os.path.join(
    FINAL_TEST_DIR,
    "xgboost_final_tuned_test_predictions.csv"
)

THRESHOLD_FILE = os.path.join(
    PROJECT_DIR,
    "06_models",
    "18_productivity_classification",
    "productivity_thresholds_training_only.json"
)

print("\n" + "=" * 70)
print("PROJECT PATHS")
print("=" * 70)

print("Final tuned directory:")
print(FINAL_TUNED_DIR)

print("\nTuned test predictions:")
print(TEST_PREDICTIONS_FILE)

print("\nTraining-only productivity threshold file:")
print(THRESHOLD_FILE)

print("\nExtreme-CHL output directory:")
print(OUTPUT_DIR)


# ================================================================
# 2. SAFETY CHECK
# ================================================================

print("\n" + "=" * 70)
print("METHODOLOGICAL SAFETY")
print("=" * 70)

print("""
Test dataset:
    NOT LOADED

Test predictions:
    ALREADY GENERATED
    WILL ONLY BE READ

XGBoost training:
    NOT PERFORMED

XGBoost retraining:
    NOT PERFORMED

Hyperparameter modification:
    NOT PERFORMED

Model modification:
    NOT PERFORMED

Prediction regeneration:
    NOT PERFORMED

Extreme-event thresholds:
    TRAINING-DERIVED ONLY
""")

if not os.path.exists(TEST_PREDICTIONS_FILE):
    raise FileNotFoundError(
        "Final tuned XGBoost test prediction file was not found."
    )

print("✓ Existing tuned-model test predictions found.")


# ================================================================
# 3. LOAD EXISTING TEST PREDICTIONS
# ================================================================

print("\n" + "=" * 70)
print("LOADING EXISTING TUNED XGBOOST TEST PREDICTIONS")
print("=" * 70)

df = pd.read_csv(TEST_PREDICTIONS_FILE)

print("Shape:", df.shape)

required_columns = [
    "date",
    "latitude",
    "longitude",
    "actual_chlorophyll_a",
    "predicted_chlorophyll_a"
]

missing_columns = [
    c for c in required_columns
    if c not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

print("✓ Required columns found.")


# ================================================================
# 4. DATA SAFETY CHECK
# ================================================================

print("\n" + "=" * 70)
print("TEST PREDICTION SAFETY CHECK")
print("=" * 70)

actual = df["actual_chlorophyll_a"].astype(float).values
predicted = df["predicted_chlorophyll_a"].astype(float).values

print("Actual CHL NaN:",
      np.isnan(actual).sum())

print("Predicted CHL NaN:",
      np.isnan(predicted).sum())

print("Actual CHL infinite:",
      np.isinf(actual).sum())

print("Predicted CHL infinite:",
      np.isinf(predicted).sum())

if (
    np.isnan(actual).any()
    or np.isnan(predicted).any()
    or np.isinf(actual).any()
    or np.isinf(predicted).any()
):
    raise ValueError(
        "Invalid CHL values detected."
    )

print("✓ Test prediction values are clean.")


# ================================================================
# 5. BASIC FINAL TEST PERFORMANCE
# ================================================================

print("\n" + "=" * 70)
print("FINAL TUNED XGBOOST TEST PERFORMANCE")
print("=" * 70)

mae = mean_absolute_error(actual, predicted)

rmse = np.sqrt(
    mean_squared_error(actual, predicted)
)

r2 = r2_score(actual, predicted)

bias = np.mean(
    predicted - actual
)

median_ae = np.median(
    np.abs(predicted - actual)
)

print(f"MAE       : {mae:.6f} mg/m³")
print(f"RMSE      : {rmse:.6f} mg/m³")
print(f"R²        : {r2:.6f}")
print(f"Median AE : {median_ae:.6f} mg/m³")
print(f"Bias      : {bias:.6f} mg/m³")


# ================================================================
# 6. CHL RANGE ANALYSIS
# ================================================================

print("\n" + "=" * 70)
print("CHL-RANGE PERFORMANCE ANALYSIS")
print("=" * 70)

range_definitions = [
    ("0–0.2", 0.0, 0.2),
    ("0.2–0.5", 0.2, 0.5),
    ("0.5–1", 0.5, 1.0),
    ("1–2", 1.0, 2.0),
    ("2–5", 2.0, 5.0),
    ("5–10", 5.0, 10.0),
    (">10", 10.0, np.inf)
]

range_results = []

for name, lower, upper in range_definitions:

    if np.isinf(upper):
        mask = actual > lower
    else:
        mask = (
            (actual >= lower)
            & (actual < upper)
        )

    n = mask.sum()

    if n == 0:
        continue

    a = actual[mask]
    p = predicted[mask]

    range_mae = mean_absolute_error(a, p)

    range_rmse = np.sqrt(
        mean_squared_error(a, p)
    )

    range_r2 = r2_score(a, p) if n > 1 else np.nan

    range_bias = np.mean(p - a)

    range_results.append({
        "chl_range": name,
        "observations": int(n),
        "actual_mean": np.mean(a),
        "predicted_mean": np.mean(p),
        "MAE": range_mae,
        "RMSE": range_rmse,
        "R2": range_r2,
        "bias": range_bias,
        "actual_max": np.max(a),
        "predicted_max": np.max(p)
    })

range_df = pd.DataFrame(range_results)

print(range_df.to_string(index=False))

range_output = os.path.join(
    OUTPUT_DIR,
    "xgb_tuned_test_error_by_chl_range.csv"
)

range_df.to_csv(
    range_output,
    index=False
)

print("\nSaved:")
print(range_output)


# ================================================================
# 7. TRAINING-DERIVED EXTREME CHL THRESHOLDS
# ================================================================

print("\n" + "=" * 70)
print("TRAINING-DERIVED EXTREME-CHL THRESHOLDS")
print("=" * 70)

# These values were already derived from the training distribution.
# They are intentionally NOT recalculated from the test set.

extreme_thresholds = {
    75: 0.545567,
    90: 0.922410,
    95: 1.304732,
    99: 3.127116
}

for percentile, threshold in extreme_thresholds.items():
    print(
        f"{percentile}th percentile threshold:"
        f" {threshold:.6f} mg/m³"
    )

print("\n✓ Thresholds originate from training data.")
print("✓ Test distribution was not used to define thresholds.")


# ================================================================
# 8. EXTREME CHL EVENT ANALYSIS
# ================================================================

print("\n" + "=" * 70)
print("EXTREME-CHL EVENT ANALYSIS")
print("=" * 70)

event_results = []
confusion_results = []

for percentile, threshold in extreme_thresholds.items():

    actual_event = actual >= threshold
    predicted_event = predicted >= threshold

    event_observations = int(
        actual_event.sum()
    )

    accuracy = accuracy_score(
        actual_event,
        predicted_event
    )

    precision = precision_score(
        actual_event,
        predicted_event,
        zero_division=0
    )

    recall = recall_score(
        actual_event,
        predicted_event,
        zero_division=0
    )

    f1 = f1_score(
        actual_event,
        predicted_event,
        zero_division=0
    )

    tn, fp, fn, tp = confusion_matrix(
        actual_event,
        predicted_event,
        labels=[False, True]
    ).ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else 0
    )

    false_alarm_rate = (
        fp / (fp + tn)
        if (fp + tn) > 0
        else 0
    )

    event_mask = actual_event

    event_actual = actual[event_mask]
    event_predicted = predicted[event_mask]

    if len(event_actual) > 0:

        event_mae = mean_absolute_error(
            event_actual,
            event_predicted
        )

        event_rmse = np.sqrt(
            mean_squared_error(
                event_actual,
                event_predicted
            )
        )

        event_bias = np.mean(
            event_predicted - event_actual
        )

        actual_event_mean = np.mean(
            event_actual
        )

        predicted_event_mean = np.mean(
            event_predicted
        )

        actual_event_max = np.max(
            event_actual
        )

        predicted_event_max = np.max(
            event_predicted
        )

    else:

        event_mae = np.nan
        event_rmse = np.nan
        event_bias = np.nan
        actual_event_mean = np.nan
        predicted_event_mean = np.nan
        actual_event_max = np.nan
        predicted_event_max = np.nan

    event_results.append({
        "percentile": percentile,
        "threshold_mg_m3": threshold,
        "event_observations": event_observations,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "specificity": specificity,
        "false_alarm_rate": false_alarm_rate,
        "high_chl_detection_rate": recall,
        "event_MAE": event_mae,
        "event_RMSE": event_rmse,
        "event_bias": event_bias,
        "actual_mean": actual_event_mean,
        "predicted_mean": predicted_event_mean,
        "actual_max": actual_event_max,
        "predicted_max": predicted_event_max
    })

    confusion_results.append({
        "percentile": percentile,
        "threshold_mg_m3": threshold,
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp)
    })

event_df = pd.DataFrame(
    event_results
)

confusion_df = pd.DataFrame(
    confusion_results
)

print("\nExtreme-event performance:")
print(
    event_df.to_string(
        index=False
    )
)

print("\nConfusion matrices:")
print(
    confusion_df.to_string(
        index=False
    )
)


# ================================================================
# 9. SAVE EXTREME EVENT RESULTS
# ================================================================

event_output = os.path.join(
    OUTPUT_DIR,
    "xgb_tuned_test_extreme_chl_event_performance.csv"
)

confusion_output = os.path.join(
    OUTPUT_DIR,
    "xgb_tuned_test_extreme_chl_confusion_matrices.csv"
)

event_df.to_csv(
    event_output,
    index=False
)

confusion_df.to_csv(
    confusion_output,
    index=False
)

print("\nSaved:")
print(event_output)
print(confusion_output)


# ================================================================
# 10. LARGEST PREDICTION ERRORS
# ================================================================

print("\n" + "=" * 70)
print("LARGEST TUNED XGBOOST TEST PREDICTION ERRORS")
print("=" * 70)

df["error"] = (
    df["predicted_chlorophyll_a"]
    - df["actual_chlorophyll_a"]
)

df["absolute_error"] = np.abs(
    df["error"]
)

top_errors = (
    df.sort_values(
        "absolute_error",
        ascending=False
    )
    .head(100)
    .copy()
)

print(
    top_errors[
        [
            "date",
            "latitude",
            "longitude",
            "actual_chlorophyll_a",
            "predicted_chlorophyll_a",
            "error",
            "absolute_error"
        ]
    ]
    .head(20)
    .to_string(index=False)
)

top_errors_output = os.path.join(
    OUTPUT_DIR,
    "xgb_tuned_test_largest_prediction_errors_top100.csv"
)

top_errors.to_csv(
    top_errors_output,
    index=False
)

print("\nSaved:")
print(top_errors_output)


# ================================================================
# 11. EXTREME ACTUAL CHL SUMMARY
# ================================================================

print("\n" + "=" * 70)
print("EXTREME ACTUAL CHL SUMMARY")
print("=" * 70)

max_actual_index = np.argmax(
    actual
)

max_error_index = np.argmax(
    np.abs(predicted - actual)
)

print(
    f"Maximum actual CHL:"
    f" {actual[max_actual_index]:.6f} mg/m³"
)

print(
    f"Predicted CHL at actual maximum:"
    f" {predicted[max_actual_index]:.6f} mg/m³"
)

print(
    f"Maximum predicted CHL:"
    f" {np.max(predicted):.6f} mg/m³"
)

print(
    f"Maximum absolute error:"
    f" {np.max(np.abs(predicted - actual)):.6f} mg/m³"
)

print("\nLocation of maximum actual CHL:")

print(
    df.iloc[max_actual_index][
        [
            "date",
            "latitude",
            "longitude"
        ]
    ].to_string()
)

print("\nLocation of maximum absolute error:")

print(
    df.iloc[max_error_index][
        [
            "date",
            "latitude",
            "longitude",
            "actual_chlorophyll_a",
            "predicted_chlorophyll_a",
            "error"
        ]
    ].to_string()
)


# ================================================================
# 12. FINAL EXTREME-CHL SUMMARY
# ================================================================

summary = {

    "analysis": "31.9P-16Q",

    "model": "Final Tuned XGBoost",

    "test_predictions_source":
        "Previously generated final tuned XGBoost test predictions",

    "test_data_reloaded": False,

    "model_retrained": False,

    "model_modified": False,

    "threshold_source":
        "Training-only CHL distribution",

    "overall_test_metrics": {
        "MAE_mg_m3": float(mae),
        "RMSE_mg_m3": float(rmse),
        "R2": float(r2),
        "MedianAE_mg_m3": float(median_ae),
        "Bias_mg_m3": float(bias)
    },

    "actual_max_chl_mg_m3":
        float(np.max(actual)),

    "predicted_max_chl_mg_m3":
        float(np.max(predicted)),

    "maximum_absolute_error_mg_m3":
        float(np.max(np.abs(predicted - actual))),

    "extreme_event_thresholds_mg_m3":
        {
            str(k): float(v)
            for k, v in extreme_thresholds.items()
        }
}

summary_output = os.path.join(
    OUTPUT_DIR,
    "xgb_tuned_final_extreme_chl_summary.json"
)

with open(
    summary_output,
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=4
    )

print("\n" + "=" * 70)
print("FINAL 31.9P-16Q SAFETY CHECK")
print("=" * 70)

print("Existing tuned test predictions used : TRUE")
print("Test dataset loaded                   : FALSE")
print("Test predictions regenerated          : FALSE")
print("Training performed                    : FALSE")
print("Retraining performed                  : FALSE")
print("Hyperparameters changed               : FALSE")
print("Model modified                        : FALSE")
print("Extreme thresholds from training     : TRUE")

print("\n✓ Final tuned test predictions analysed.")
print("✓ Extreme-CHL thresholds remain training-derived.")
print("✓ No test data was reloaded.")
print("✓ No model was modified.")
print("✓ No predictions were regenerated.")

print("\nSaved summary:")
print(summary_output)

print("\n" + "=" * 70)
print("STEP 31.9P-16Q COMPLETE")
print("=" * 70)

print("""
NEXT:
31.9P-16R — Final Tuned XGBoost Productivity Classification
""")

STEP 31.9P-16Q
FINAL TUNED XGBOOST EXTREME-CHL ANALYSIS

PROJECT PATHS
Final tuned directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/02_XGBoost/hyperparameter_optimization/final_tuned_candidate

Tuned test predictions:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/02_XGBoost/hyperparameter_optimization/final_tuned_candidate/final_test_evaluation/xgboost_final_tuned_test_predictions.csv

Training-only productivity threshold file:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/18_productivity_classification/productivity_thresholds_training_only.json

Extreme-CHL output directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/02_XGBoost/hyperparameter_optimization/final_tuned_candidate/extreme_chl_analysis

METHODOLOGICAL SAFETY

Test dataset:
    NOT LOADED

Test predictions:
    ALREADY GENERATED
    WILL ONLY BE READ

XGBoost training:
    NOT PERFORMED

XGBoost retraining:
    NOT PERFORMED

Hyperparameter modification:
   